# CS781 Phase 3: Rhetorical role prediction

**Best leaderboard result of this phase:** macro-F1 0.3922 (precision 0.5177, recall 0.3768), rank 127.

**What this notebook does.** Sentence classification (13 roles) with the Phase 1 decoder, separately for the CL and IT domains, with context sentences (3 before, 1 after). It compares full fine-tuning, LoRA, LoRMA+, IA3, QLoRA, prompt tuning and a distilled half-depth student, then tunes ensembling, label-transition smoothing and class biases and writes the final prediction file.

**Inputs.** Phase 1 decoder checkpoint, tokenizer and the Phase 3 data. Internet and a Hugging Face token (secret HF_TOKEN) are needed to download the data.

**Source.** This is the notebook, exactly as it ran on Kaggle, that produced our best Phase 3 submission (stored outputs removed to keep the file small). Run it top to bottom on a Kaggle T4 GPU with Internet on.

## Fixed rolling checkpoints + controlled durable publishing

This notebook uses a **fixed local checkpoint set** per run: `latest.pt` is atomically replaced after each completed epoch and `best.pt` is atomically replaced only when validation improves. It does not create epoch-numbered checkpoint files. The resume checkpoint no longer duplicates the best model weights inside `latest.pt`, reducing disk usage.

Durable Kaggle Dataset publishing is **not done every few epochs**. It happens at experiment completion and once at the final recovery step. Each publish stages only `latest.pt`, `best.pt`, `result.json`, the ledger, and prediction JSONs—never the large logits/embedding `.npy` artifacts. This avoids rapid version growth while keeping completed experiments recoverable.

Important: Kaggle retains Dataset versions according to its own versioning/storage policy; this notebook cannot delete old remote versions or guarantee a hard cap on remote history. For the most conservative storage use, leave experiment-end publishing enabled only if remote recovery is needed, and periodically remove obsolete Dataset versions in Kaggle if the UI/API offers that option.


# CS781 Phase 3 — Rhetorical Role Prediction (CL + IT): training pipeline

**What this notebook does:** loads the Phase 3 data, builds sentence-only and context-aware inputs, fine-tunes your Phase 1 language models with
six adaptation techniques (full FT, LoRA/rsLoRA, **LoRMA+/LoRMAπ**, IA³, QLoRA-style NF4, prompt tuning) plus **knowledge distillation**,
logs everything (macro-F1/P/R, trainable params, model size, GPU memory, time) to `ledger.csv`, and writes `prediction.json` + `submission.zip`.

**Run order:** 1 setup → 2 checkpoint inventory → 3 data → 4 tokenizer/examples → 5 stats → 6 metrics → 7 models → 8 trainer → **9 smoke tests (must pass)** → 10 plan → 11 RUN A → 12 RUN B → 13 KD → 14 summary → 15 post-hoc → 16 submission.

**Read before running**
* The dataset is gated, so cell 3 auto-detects the JSON layout and prints it. If it fails, paste the printed schema and set `CUSTOM_PARSER`.
* Only the GPT-2-style Phase 1 checkpoint has a loader. Cell 2 lists every checkpoint with a family guess; the other three architectures need a loader in cell 7 (send the cell 2 output).
* Check the output format of cell 16 against the official `prediction.json` schema (`COMBINE_MODE`).
* Dev has only a few documents per domain, so differences below ~0.02 macro-F1 are noise. Trust results that agree across CL and IT.


## Checkpointing and recovery (updated)
- Every completed training epoch writes an atomic `latest.pt` checkpoint containing model, optimizer, scheduler, AMP scaler, epoch, early-stopping state, and RNG states.
- `best.pt` stores the best validation macro-F1 model for inference.
- If an experiment is interrupted, rerun its cell: it resumes from `latest.pt` when available. Completed experiments with `result.json` are skipped unless `force=True`.
- The final cell creates `/kaggle/working/phase3_recovery.zip`. Download it or publish it as a dedicated Kaggle Dataset to make it durable outside the current session.
- This checkpointing update does not add missing Phase 1 architectures. The architecture-coverage requirement remains unresolved until those architectures are registered and tested.


In [ ]:
# ===== CELL 6 — REGISTER THE CORRECT PHASE 1 CHECKPOINT =====
# The architecture cell later in this notebook registers this exact checkpoint.
print("Target checkpoint: /kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt")


In [ ]:
# ================= CELL 0 — optional Kaggle secrets =================
# Do not require unrelated Kaggle API secrets just to run the notebook.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except Exception:
    pass

print("HF_TOKEN secret available:", bool(HF_TOKEN))


In [ ]:
# ================= CELL 0b — PERIODIC DURABLE BACKUP CONFIG =================
# Required Kaggle Secrets:
#   KAGGLE_USERNAME = your Kaggle username
#   KAGGLE_KEY      = your Kaggle API key
#   KAGGLE_DATASET_SLUG = username/cs781-phase3-recovery
#
# Create these under Kaggle Notebook > Add-ons > Secrets and enable notebook access.
# The dataset slug must use YOUR Kaggle username. First publish creates the dataset;
# later publishes create new dataset versions.
import os, subprocess, shutil, json, time
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    _ks = UserSecretsClient()
    for _secret_name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_DATASET_SLUG"):
        if not os.environ.get(_secret_name):
            try:
                os.environ[_secret_name] = _ks.get_secret(_secret_name)
            except Exception:
                pass
except Exception:
    pass

# Confirmed dataset created successfully in this account.
KAGGLE_DATASET_SLUG = "armankrsingh/cs781-phase3-recovery"
os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
PUBLISH_EVERY_N_EPOCHS = 0  # FIXED POLICY: no per-epoch remote versions
# Local latest.pt/best.pt are still saved after every epoch.
# Remote versions are published only at experiment completion and final recovery.
PUBLISH_AT_EACH_EXPERIMENT_END = True
DURABLE_PUBLISH_ENABLED = (
    bool(os.environ.get("KAGGLE_USERNAME"))
    and bool(os.environ.get("KAGGLE_KEY"))
    and KAGGLE_DATASET_SLUG == "armankrsingh/cs781-phase3-recovery"
)

if DURABLE_PUBLISH_ENABLED:
    print("Periodic Kaggle Dataset publishing: ENABLED")
    print("Target dataset:", KAGGLE_DATASET_SLUG)
    print("Per-epoch remote publishing: DISABLED (rolling local checkpoints only)")
    print("Publish at experiment end:", PUBLISH_AT_EACH_EXPERIMENT_END)
else:
    print("Periodic Kaggle Dataset publishing: NOT CONFIGURED.")
    print("Add Kaggle Secrets KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_DATASET_SLUG")
    print("and replace the example slug with your own username/dataset-slug.")
    print("Training can still run; checkpoints remain in /kaggle/working until uploaded.")

In [ ]:
# ================= CELL 1 — environment, paths, helpers =================
import os, sys, json, math, time, random, re, copy, gc, glob, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import snapshot_download, HfApi

SEED = 42

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device,
      torch.cuda.get_device_name(0) if device.type == "cuda"
      else "(no GPU: training will be slow)")

WORK = Path("/kaggle/working/phase3_rr")
(WORK / "runs").mkdir(parents=True, exist_ok=True)
LEDGER = WORK / "ledger.csv"



# ---------------- PERIODIC KAGGLE DATASET PUBLISHER ----------------
PUBLISH_DIR = Path("/kaggle/working/phase3_publish")
_last_publish_epoch = {}

def _prepare_recovery_upload(reason="periodic"):
    """
    Disk-safe recovery staging:
    - completed runs (result.json exists): publish result metadata only
    - unfinished runs: include latest.pt/best.pt so they can resume
    - never duplicate completed-run checkpoints into the staging directory
    - use hardlinks for unfinished checkpoints when possible; copy only as fallback
    """
    import os, shutil, json, time
    from pathlib import Path

    if PUBLISH_DIR.exists():
        shutil.rmtree(PUBLISH_DIR)
    PUBLISH_DIR.mkdir(parents=True, exist_ok=True)
    runs_src = WORK / "runs"
    runs_dst = PUBLISH_DIR / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)

    staged_results = 0
    staged_checkpoints = 0
    staged_small = 0
    checkpoint_bytes = 0

    def _stage_file(src, dst, prefer_link=False):
        dst.parent.mkdir(parents=True, exist_ok=True)
        if prefer_link:
            try:
                os.link(src, dst)
                return "hardlink"
            except OSError:
                pass
        # Only use a real copy if there is reasonable free space.
        size = src.stat().st_size
        try:
            free = shutil.disk_usage(str(PUBLISH_DIR)).free
        except Exception:
            free = 0
        reserve = 512 * 1024 * 1024
        if free and free < size + reserve:
            raise OSError(
                f"Not enough free disk to stage {src.name} ({size/2**30:.2f} GiB). "
                "The checkpoint is kept in its original location. Free space or "
                "use a remote/object-storage upload method before retrying."
            )
        shutil.copy2(src, dst)
        return "copy"

    if runs_src.exists():
        for run_dir in sorted(runs_src.iterdir()):
            if not run_dir.is_dir():
                continue
            dst = runs_dst / run_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            result_file = run_dir / "result.json"
            is_complete = result_file.is_file()

            # The small result record is enough to skip a completed run on restart.
            if is_complete:
                _stage_file(result_file, dst / "result.json")
                staged_results += 1
                # Teacher training logits are a required downstream KD dependency.
                # Preserve only these specific arrays; do not stage every run's logits/embeddings.
                if run_dir.name in {
                    "E0b_full_ctx_CL", "E0b_full_ctx_IT",
                    "E2_lorma_plus_CL", "E2_lorma_plus_IT",
                }:
                    logits_file = run_dir / "train_logits.npy"
                    if logits_file.is_file():
                        _stage_file(logits_file, dst / "train_logits.npy", prefer_link=True)
                # Completed runs do not need latest.pt/best.pt for training resume.
                continue

            # Only unfinished runs need checkpoints for resume.
            for filename in ("latest.pt", "best.pt"):
                file = run_dir / filename
                if file.is_file():
                    how = _stage_file(file, dst / filename, prefer_link=True)
                    staged_checkpoints += 1
                    checkpoint_bytes += file.stat().st_size

    for filename in ("ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"):
        file = WORK / filename
        if file.is_file():
            _stage_file(file, PUBLISH_DIR / filename)
            staged_small += 1

    metadata = {
        "title": "CS781 Phase 3 Compact Recovery",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": True,
    }
    (PUBLISH_DIR / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    (PUBLISH_DIR / "publish_manifest.json").write_text(json.dumps({
        "project": "CS781 Phase 3 Rhetorical Role Prediction",
        "published_unix": time.time(),
        "reason": reason,
        "dataset_slug": KAGGLE_DATASET_SLUG,
        "completed_run_results_staged": staged_results,
        "unfinished_run_checkpoints_staged": staged_checkpoints,
        "small_metadata_files_staged": staged_small,
        "policy": "Completed runs publish result.json only; unfinished runs publish latest.pt/best.pt if present."
    }, indent=2), encoding="utf-8")

    total_bytes = sum(p.stat().st_size for p in PUBLISH_DIR.rglob("*") if p.is_file())
    print(
        f"[durable backup] Staged {staged_results} completed-run result records, "
        f"{staged_checkpoints} checkpoint files for unfinished runs, "
        f"{staged_small} small metadata/prediction files; apparent size "
        f"{total_bytes/2**30:.2f} GiB."
    )
    if staged_checkpoints == 0:
        print("[durable backup] Metadata-only snapshot: no checkpoints needed for completed runs.")

def publish_recovery_dataset(reason="periodic"):
    """Publish a new version of the already-created recovery Dataset."""
    if not DURABLE_PUBLISH_ENABLED:
        print("[durable backup] Skipped: Kaggle credentials not configured.")
        return False

    cli = shutil.which("kaggle")
    if cli is None:
        print("[durable backup] Installing Kaggle CLI...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        cli = shutil.which("kaggle")
    if cli is None:
        raise RuntimeError("Kaggle CLI installation did not provide the 'kaggle' command.")

    env = os.environ.copy()
    env["KAGGLE_USERNAME"] = os.environ["KAGGLE_USERNAME"]
    env["KAGGLE_KEY"] = os.environ["KAGGLE_KEY"]

    _prepare_recovery_upload(reason)

    # Dataset is already created. Do NOT use `datasets status` here:
    # status returned 403 in this environment despite list/create working.
    cmd = [
        cli, "datasets", "version",
        "-p", str(PUBLISH_DIR),
        "-m", f"CS781 recovery update: {reason}",
        "--dir-mode", "zip",
    ]
    print("[durable backup] Publishing to", KAGGLE_DATASET_SLUG, "| reason:", reason)
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-2500:])

    combined_output = (result.stdout or "") + "\n" + (result.stderr or "")
    if result.returncode != 0 or any(
        marker in combined_output.lower()
        for marker in ("dataset creation error", "skipping folder:", "403 client error", "forbidden")
    ):
        raise RuntimeError(
            "Kaggle did not confirm a clean dataset version upload. "
            "Inspect the CLI output above and check the dataset page."
        )
    if "dataset version is being created" not in combined_output.lower() and \
       "upload successful" not in combined_output.lower():
        raise RuntimeError(
            "Kaggle CLI exited without a recognizable upload confirmation. "
            "Verify the dataset page before assuming the backup succeeded."
        )

    print("[durable backup] Upload submitted. Confirm the new version and checkpoint files on Kaggle.")
    return True


def find_files(names, roots=("/kaggle/input", "/kaggle/working", "/content", ".")):
    found = {}
    for root in roots:
        if not os.path.exists(root):
            continue
        for n in names:
            if n in found:
                continue
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                found[n] = hits[0]
    return found

NEED = [
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
    "label_vocab.json"
]

paths = find_files(NEED)
missing = [n for n in NEED if n not in paths]

if missing:
    print("Missing locally:", missing)
    print("Trying Hugging Face dataset download...")

    token = os.environ.get("HF_TOKEN") or HF_TOKEN

    # Kaggle Secrets fallback
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None

    if not token:
        raise RuntimeError(
            "HF_TOKEN was not found. In Kaggle, open Add-ons > Secrets, "
            "add a secret named HF_TOKEN with your Hugging Face READ token, "
            "and enable notebook access. Also accept the dataset conditions "
            "on the Hugging Face dataset page. Alternatively, attach the "
            "Phase_3_RR dataset to this Kaggle notebook."
        )

    try:
        local = snapshot_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            token=token,
            allow_patterns=["Phase_3_RR/**"],
            local_dir=str(WORK / "hf_data")
        )
    except Exception as e:
        raise RuntimeError(
            "Hugging Face download failed. Check that you have accepted "
            "the dataset conditions and that HF_TOKEN is a valid token "
            "with read access. Original error: " + repr(e)
        ) from e

    paths = find_files(NEED, roots=(local, str(WORK / "hf_data")))
    missing = [n for n in NEED if n not in paths]

    if missing:
        # Fallback search if the files were placed in nested folders
        paths.update(find_files(missing, roots=(str(WORK / "hf_data"), local)))
        missing = [n for n in NEED if n not in paths]

    if missing:
        raise FileNotFoundError(
            f"Download completed, but these files were not found: {missing}. "
            "Check the dataset's folder structure."
        )

for k, v in paths.items():
    print(f"{k:18s} {v}")

print("Cell 1 setup complete.")

# Load the discovered files once. The parser cell below will convert them into DOCS.
RAW = {name: json.loads(Path(path).read_text(encoding="utf-8")) for name, path in paths.items()}
print("Loaded raw JSON files:", {k: type(v).__name__ for k, v in RAW.items()})


In [ ]:
# ================= CELL 2 — inventory of every Phase 1 checkpoint (use this to fill the architecture registry) =================
def family_guess(keys):
    s = " ".join(keys[:400])
    if "attn.c_attn" in s: return "gpt2-style (HF GPT2: fused c_attn Conv1D)"
    if "self_attn.q_proj" in s and "gate_proj" in s: return "llama-style (q_proj/k_proj/v_proj + gated MLP)"
    if "A_log" in s or "mixer" in s or "in_proj" in s: return "state-space / mamba-style"
    if "rwkv" in s.lower() or "time_mix" in s: return "rwkv-style"
    if "encoder.layer" in s or "attention.self.query" in s: return "bert-style (bidirectional encoder)"
    return "unknown - paste the key names to the assistant"

CKPT_EXT = (".pt", ".pth", ".bin", ".ckpt")
cands = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.endswith(CKPT_EXT) and os.path.getsize(p) > 5e6)
print(f"{len(cands)} candidate checkpoint files under /kaggle/input\n")
for p in cands:
    try:
        try: obj = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: obj = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print(p, "-> load failed:", str(e)[:120], "\n"); continue
    sd = obj.get("model_state_dict", obj) if isinstance(obj, dict) else None
    print(p, f"({os.path.getsize(p)/2**20:.0f} MB)")
    if isinstance(obj, dict):
        print("   top-level keys:", [k for k in obj.keys()][:12])
        if "model_config" in obj: print("   model_config  :", obj["model_config"])
        for k in ("step", "epoch", "context_length"):
            if k in obj: print(f"   {k}: {obj[k]}")
    if isinstance(sd, dict) and sd and hasattr(next(iter(sd.values())), "numel"):
        keys = list(sd.keys()); n = sum(v.numel() for v in sd.values() if hasattr(v, "numel"))
        print(f"   tensors: {len(keys)} | params: {n/1e6:.1f}M | family guess: {family_guess(keys)}")
        print("   first keys:", keys[:3])
    print()
    del obj, sd; gc.collect()
print("Map each of the FOUR Phase 1 models to an entry in ARCHS (cell 7). If a family says 'unknown', send this output to get a loader.")


In [ ]:

# ============================================================
# RECOVERY CELL — RESTORE PHASE 3 CHECKPOINTS FROM KAGGLE BACKUP
# Run before Cells 16–17 (experiment execution).
# Does not start training or overwrite existing files.
# ============================================================

from pathlib import Path
import shutil
import json
import torch

BACKUP = Path(
    "/kaggle/input/datasets/armankrsingh/"
    "cs781-phase3-recovery/runs"
)
RUNS = Path("/kaggle/working/phase3_rr/runs")

if not BACKUP.exists():
    raise FileNotFoundError(
        f"Recovery dataset not found: {BACKUP}\n"
        "Attach armankrsingh/cs781-phase3-recovery "
        "in Kaggle Notebook > Add Input."
    )

RUNS.mkdir(parents=True, exist_ok=True)

allowed_names = {"latest.pt", "best.pt", "result.json"}
restored, unchanged, conflicts = [], [], []

# Copy checkpoint/result files while preserving their run-folder paths.
for src in sorted(BACKUP.rglob("*")):
    if not src.is_file() or src.name not in allowed_names:
        continue

    rel = src.relative_to(BACKUP)
    dst = RUNS / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        # Never overwrite an existing local file automatically.
        if (
            dst.stat().st_size == src.stat().st_size
            and dst.name == src.name
        ):
            unchanged.append(str(rel))
        else:
            conflicts.append((str(rel), str(src), str(dst)))
        continue

    shutil.copy2(src, dst)
    restored.append(str(rel))

print(f"Restored: {len(restored)}")
for x in restored:
    print("  +", x)

print(f"\nAlready present and same size: {len(unchanged)}")
print(f"Conflicts (left untouched): {len(conflicts)}")
for rel, src, dst in conflicts:
    print(f"  ! {rel}\n      backup: {src}\n      local:  {dst}")

# Verify checkpoint readability and show saved progress.
print("\n" + "=" * 72)
print("LOCAL EXPERIMENT STATUS")
print("=" * 72)

for run_dir in sorted(p for p in RUNS.iterdir() if p.is_dir()):
    print(f"\n[{run_dir.name}]")

    result_path = run_dir / "result.json"
    if result_path.exists():
        try:
            result = json.loads(result_path.read_text())
            print("result.json:", json.dumps(result, indent=2)[:1600])
        except Exception as e:
            print("result.json unreadable:", repr(e))
    else:
        print("result.json: MISSING")

    for name in ("latest.pt", "best.pt"):
        cp = run_dir / name
        if not cp.exists():
            print(f"{name}: MISSING")
            continue

        print(f"{name}: {cp.stat().st_size / 1024**3:.3f} GiB")

        try:
            ck = torch.load(
                cp, map_location="cpu",
                weights_only=False, mmap=True
            )
            if isinstance(ck, dict):
                for key in (
                    "epoch", "step", "global_step",
                    "best_epoch", "best_f1"
                ):
                    if key in ck:
                        print(f"  {key}: {ck[key]}")
            del ck
        except Exception as e:
            print(f"  READ ERROR: {type(e).__name__}: {e}")

print("\nRecovery cell finished. No training was started.")


In [ ]:
# ================= CELL 3 — PARSE PHASE 3 DATA (TRAIN / DEV / TEST) =================
# Supports the observed test schema {doc_id: [sentence, ...]} and common labeled schemas.

LABELS_FALLBACK = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute",
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon",
    "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]

def _norm_key(k):
    return str(k).strip().lower().replace("-", "_").replace(" ", "_")

def _find_sentence_list(payload):
    if isinstance(payload, list):
        if all(isinstance(x, str) for x in payload):
            return payload
        if payload and all(isinstance(x, dict) for x in payload):
            out = []
            for item in payload:
                sent = None
                for k, v in item.items():
                    if _norm_key(k) in {"text", "sentence", "sent", "content"} and isinstance(v, str):
                        sent = v
                        break
                if sent is None:
                    out = []
                    break
                out.append(sent)
            if out:
                return out
        for item in payload:
            got = _find_sentence_list(item)
            if got is not None:
                return got
        return None

    if isinstance(payload, dict):
        # Prefer conventional keys first.
        for k, v in payload.items():
            if _norm_key(k) in {"text", "texts", "sentences", "sentence", "sents", "content"}:
                if isinstance(v, list) and all(isinstance(x, str) for x in v):
                    return v
                if isinstance(v, str):
                    return [v]
        # Search nested values, but don't treat a labels list as sentence text.
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "y", "tags"}:
                continue
            if isinstance(v, (dict, list)):
                got = _find_sentence_list(v)
                if got is not None:
                    return got
    return None

def _find_label_list(payload):
    if isinstance(payload, dict):
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "tags", "roles", "y"} and isinstance(v, list):
                return v
        for k, v in payload.items():
            if isinstance(v, (dict, list)):
                got = _find_label_list(v)
                if got is not None:
                    return got
    elif isinstance(payload, list):
        if payload and all(isinstance(x, dict) for x in payload):
            vals = []
            for item in payload:
                lab = None
                for k, v in item.items():
                    if _norm_key(k) in {"label", "labels", "tag", "role"} and isinstance(v, str):
                        lab = v
                        break
                if lab is None:
                    vals = []
                    break
                vals.append(lab)
            if vals:
                return vals
        for item in payload:
            if isinstance(item, (dict, list)):
                got = _find_label_list(item)
                if got is not None:
                    return got
    return None

def parse_split(obj, style=None):
    if isinstance(obj, dict):
        items = list(obj.items())
    elif isinstance(obj, list):
        items = [(str(i), x) for i, x in enumerate(obj)]
    else:
        raise ValueError(f"Unsupported split JSON type: {type(obj).__name__}")

    docs = []
    for did, payload in items:
        # The test files were confirmed to map document IDs directly to sentence strings.
        if isinstance(payload, list) and all(isinstance(x, str) for x in payload):
            docs.append({"id": str(did), "sents": payload, "labels": None, "setter": None})
            continue

        sents = _find_sentence_list(payload)
        labels = _find_label_list(payload)
        if sents is None:
            raise ValueError(
                f"{did}: cannot find a sentence list. Payload type={type(payload).__name__}; "
                f"keys={list(payload.keys())[:30] if isinstance(payload, dict) else 'n/a'}"
            )
        if labels is not None:
            labels = [str(x) for x in labels]
            if len(labels) != len(sents):
                raise ValueError(f"{did}: {len(sents)} sentences but {len(labels)} labels")
        docs.append({"id": str(did), "sents": [str(x) for x in sents],
                     "labels": labels, "setter": None})
    return docs, style

# Load label vocabulary if it is a recognizable format; otherwise use the official label names.
_vocab_obj = RAW.get("label_vocab.json")
LABELS = None
if isinstance(_vocab_obj, list) and all(isinstance(x, str) for x in _vocab_obj):
    LABELS = _vocab_obj
elif isinstance(_vocab_obj, dict):
    for key in ("labels", "label_vocab", "id2label", "label2id"):
        value = _vocab_obj.get(key)
        if isinstance(value, list) and all(isinstance(x, str) for x in value):
            LABELS = value
            break
        if isinstance(value, dict) and value:
            try:
                LABELS = [k for k, v in sorted(value.items(), key=lambda kv: int(kv[1]))]
            except Exception:
                LABELS = list(value.keys())
            break
if not LABELS:
    LABELS = LABELS_FALLBACK
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STYLE = {"CL": None, "IT": None}

DOCS = {"CL": {}, "IT": {}}
TEST_OBJ = {}
for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        DOCS[dom][split], _ = parse_split(RAW[f"{dom}_{split}.json"])
        if not DOCS[dom][split]:
            raise ValueError(f"{dom} {split} split is empty")
        for d in DOCS[dom][split]:
            if d["labels"] is None:
                raise ValueError(f"{dom} {split} document {d['id']} has no labels")
            bad = sorted(set(d["labels"]) - set(LABELS))
            if bad:
                raise ValueError(f"{dom} {split} document {d['id']} has labels absent from vocabulary: {bad}")
    TEST_OBJ[dom] = RAW[f"{dom}_test.json"]
    DOCS[dom]["test"], _ = parse_split(TEST_OBJ[dom])
    print(dom, {sp: len(DOCS[dom][sp]) for sp in ("train", "dev", "test")})
    print("  sample sentence counts:", [len(d["sents"]) for d in DOCS[dom]["test"][:2]])

print("Labels:", LABELS)
print("DOCS and LABEL2ID are ready.")


# Validate schema and labels before training.
assert set(("CL", "IT")).issubset(DOCS)
for _dom in ("CL", "IT"):
    for _split in ("train", "dev"):
        assert DOCS[_dom][_split], f"Empty {_dom}/{_split}"
        for _d in DOCS[_dom][_split]:
            assert _d["labels"] is not None, f"Missing labels in {_dom}/{_split}/{_d['id']}"
            assert len(_d["sents"]) == len(_d["labels"]), f"Sentence/label mismatch in {_dom}/{_split}/{_d['id']}"
    for _d in DOCS[_dom]["test"]:
        assert _d["labels"] is None, f"Expected unlabeled test doc: {_d['id']}"
print("Train/dev/test schema validation passed.")


In [ ]:
# ================= CELL 5 — tokenizer, example builder (sentence-only vs context-aware), batching =================
from tokenizers import Tokenizer
_tok_files = find_files(["legal_bpe_32k.json"])
if "legal_bpe_32k.json" not in _tok_files:
    raise FileNotFoundError(
        "legal_bpe_32k.json was not found. Attach the Phase 1 tokenizer dataset "
        "to Kaggle or update tok_path to its actual path."
    )
tok_path = _tok_files["legal_bpe_32k.json"]
tokenizer = Tokenizer.from_file(tok_path)
print("tokenizer:", tok_path, "| vocab:", tokenizer.get_vocab_size())

_ENC = {}
def enc(text):
    r = _ENC.get(text)
    if r is None: r = _ENC[text] = tokenizer.encode(text, add_special_tokens=False).ids
    return r

MK = {"prev": enc(" <<previous>> "), "next": enc(" <<next>> "), "tgt": enc(" <<target>> ")}
SEP = enc(" | ")

def build_examples(docs, ctx, k_left=3, k_right=1, t_budget=96, l_budget=160, r_budget=64, max_sent=96):
    """One example per sentence. Context-aware input (for a causal LM the target comes LAST so it can attend to everything):
       <<previous>> left sentences <<next>> right sentences <<target>> target sentence."""
    exs = []
    for d in docs:
        toks = [enc(s)[:max_sent] for s in d["sents"]]
        n = len(toks)
        for i in range(n):
            tgt = toks[i][:t_budget]
            if ctx:
                left, right = [], []
                for j in range(max(0, i - k_left), i): left += toks[j] + SEP
                for j in range(i + 1, min(n, i + 1 + k_right)): right += toks[j] + SEP
                ids = MK["prev"] + left[-l_budget:] + MK["next"] + right[:r_budget] + MK["tgt"] + tgt
            else:
                ids = list(tgt)
            if not ids: ids = [0]
            rel = i / max(n - 1, 1)
            y = LABEL2ID[d["labels"][i]] if d["labels"] is not None and d["labels"][i] is not None else -1
            exs.append(dict(ids=ids, pos=[rel, rel * rel], y=y, doc=d["id"], i=i))
    return exs

def make_batches(exs, bs, shuffle, rng):
    idx = list(range(len(exs)))
    if shuffle: rng.shuffle(idx)
    batches = []
    for c in range(0, len(idx), bs * 50):
        ch = sorted(idx[c:c + bs * 50], key=lambda j: len(exs[j]["ids"]))
        batches += [ch[k:k + bs] for k in range(0, len(ch), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

def collate(exs, bidx):
    import torch
    L = max(len(exs[j]["ids"]) for j in bidx)
    ids = torch.zeros((len(bidx), L), dtype=torch.long); mask = torch.zeros_like(ids)
    for r, j in enumerate(bidx):
        n = len(exs[j]["ids"]); ids[r, :n] = torch.tensor(exs[j]["ids"]); mask[r, :n] = 1
    pos = torch.tensor([exs[j]["pos"] for j in bidx], dtype=torch.float32)
    y = torch.tensor([exs[j]["y"] for j in bidx], dtype=torch.long)
    return ids, mask, pos, y


In [ ]:
# ================= CELL 4 — parser status =================
# Do not redefine parse_split here. Cell 3 is the single source of truth.
print("DOCS keys:", list(DOCS.keys()))
for _dom in ("CL", "IT"):
    print(_dom, {k: len(v) for k, v in DOCS[_dom].items()})


In [ ]:
# ================= CELL 7a — metrics + inference-time post-processing (numpy only) =================
def prf_report(y_true, y_pred, K):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    cm = np.zeros((K, K), dtype=np.int64); np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(K), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros(K), where=(tp + fn) > 0)
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0          # labels in gold or prediction = sklearn's default macro average
    return dict(macro_f1=float(f1[present].mean()), macro_p=float(prec[present].mean()), macro_r=float(rec[present].mean()),
                macro_f1_all=float(f1.mean()), acc=float(tp.sum() / cm.sum()), f1=f1, prec=prec, rec=rec,
                support=cm.sum(1), cm=cm, n_present=int(present.sum()))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def transition_logprobs(docs, label2id, K, alpha=1.0):
    T = np.full((K, K), alpha); S = np.full(K, alpha)
    for d in docs:
        ys = [label2id[l] for l in d["labels"]]
        S[ys[0]] += 1
        for a, b in zip(ys[:-1], ys[1:]): T[a, b] += 1
    return np.log(T / T.sum(1, keepdims=True)), np.log(S / S.sum())

def viterbi_smooth(logp, logT, logS, lam):
    """argmax_y  sum_i logp[i,y_i] + lam * (logS[y_0] + sum_i logT[y_{i-1}, y_i])"""
    n, K = logp.shape
    if lam == 0 or n == 1: return logp.argmax(1)
    dp = np.zeros((n, K)); bp = np.zeros((n, K), dtype=int)
    dp[0] = logp[0] + lam * logS
    for i in range(1, n):
        cand = dp[i - 1][:, None] + lam * logT
        bp[i] = cand.argmax(0); dp[i] = cand.max(0) + logp[i]
    path = [int(dp[-1].argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i, path[-1]]))
    return np.array(path[::-1])

def group_by_doc(exs):
    g = {}
    for j, e in enumerate(exs): g.setdefault(e["doc"], []).append(j)
    return {d: sorted(ix, key=lambda j: exs[j]["i"]) for d, ix in g.items()}

def smooth_split(probs, exs, logT, logS, lam):
    pred = np.zeros(len(exs), dtype=int)
    for d, ix in group_by_doc(exs).items():
        pred[ix] = viterbi_smooth(np.log(np.clip(probs[ix], 1e-9, 1)), logT, logS, lam)
    return pred

def knn_interpolate(q_emb, tr_emb, tr_y, q_probs, K, k=10, tau=0.1, lam=0.5):
    """p = lam * p_model + (1-lam) * p_kNN  (neighbourhood-at-inference idea; neighbours come from TRAIN sentences only)"""
    qe = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    te = tr_emb / (np.linalg.norm(tr_emb, axis=1, keepdims=True) + 1e-9)
    sims = qe @ te.T
    top = np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
    out = np.zeros_like(q_probs)
    for r in range(len(qe)):
        s = sims[r, top[r]] / tau; w = softmax_np(s)
        np.add.at(out[r], tr_y[top[r]], w)
    return lam * q_probs + (1 - lam) * out

def validate_submission(obj, docs, label_set, style=None):
    """every document and sentence present, order preserved, exactly one valid label per sentence"""
    d2 = {d["id"]: d for d in docs}
    problems = []
    parsed, _ = parse_split(obj, style)
    if [d["id"] for d in parsed] != [d["id"] for d in docs]: problems.append("document ids / order differ from the test file")
    for p in parsed:
        ref = d2.get(p["id"])
        if ref is None: continue
        if p["sents"] != ref["sents"]: problems.append(f"{p['id']}: sentence text/order changed")
        if p["labels"] is None or len(p["labels"]) != len(ref["sents"]): problems.append(f"{p['id']}: label count mismatch"); continue
        bad = [l for l in p["labels"] if l not in label_set]
        if bad: problems.append(f"{p['id']}: invalid labels {set(bad)}")
    return problems


In [ ]:
# ================= CELL 6 — backbones, classifier, adapters (LoRA / rsLoRA / LoRMA+ / LoRMAπ / IA3 / QLoRA / prompt) =================
from transformers import GPT2Config, GPT2Model
from transformers.pytorch_utils import Conv1D

# ---- architecture registry: exact known 3,000-step Phase 1 checkpoint ----
PHASE1_CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
if not os.path.isfile(PHASE1_CHECKPOINT):
    raise FileNotFoundError(
        f"Required 3,000-step checkpoint not found: {PHASE1_CHECKPOINT}. "
        "Attach the Kaggle dataset 'CS781 Phase 1 Checkpoints'."
    )
ARCHS = {"gpt2_a": {"path": PHASE1_CHECKPOINT, "family": "gpt2"}}
print("Registered architecture:", ARCHS)

def build_backbone(arch, bb_dropout=0.1, n_layers=None):
    """-> (backbone, hidden, causal, n_positions, family). n_layers: keep only the first n layers (student initialisation)."""
    spec = ARCHS[arch]
    if spec["family"] != "gpt2":
        raise NotImplementedError(f"No loader for family '{spec['family']}'. Run the inventory cell, paste its output, and add a loader here.")
    ck = torch.load(spec["path"], map_location="cpu", weights_only=False)
    cfg = ck.get("model_config")
    if not isinstance(cfg, dict):
        raise ValueError(f"Checkpoint has no model_config dictionary: {spec['path']}")
    vocab_size = cfg.get("vocab_size")
    context_length = cfg.get("context_length", cfg.get("n_positions"))
    hidden_size = cfg.get("hidden_size", cfg.get("n_embd"))
    num_layers = cfg.get("num_layers", cfg.get("n_layer"))
    num_heads = cfg.get("num_heads", cfg.get("n_head"))
    if not all(v is not None for v in (vocab_size, context_length, hidden_size, num_layers, num_heads)):
        raise ValueError(f"Incomplete model configuration: {cfg}")
    gcfg = GPT2Config(
        vocab_size=vocab_size, n_positions=context_length, n_ctx=context_length,
        n_embd=hidden_size, n_layer=num_layers, n_head=num_heads,
        resid_pdrop=bb_dropout, embd_pdrop=bb_dropout, attn_pdrop=bb_dropout
    )
    bb = GPT2Model(gcfg)
    sd = ck.get("model_state_dict", ck)
    if not isinstance(sd, dict):
        raise ValueError("model_state_dict is not a dictionary")
    prefixed = {k[len("transformer."):]: v for k, v in sd.items() if k.startswith("transformer.")}
    if prefixed:
        sd = prefixed
    sd = {k: v for k, v in sd.items() if not k.startswith("lm_head.")}
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint/backbone mismatch. Missing keys (first 8): {missing[:8]}; "
            f"unexpected keys (first 8): {unexpected[:8]}"
        )
    del ck
    if n_layers is not None and n_layers < len(bb.h):
        bb.h = nn.ModuleList(list(bb.h)[:n_layers]); bb.config.n_layer = n_layers
    return bb, cfg["hidden_size"], True, cfg["context_length"], "gpt2"

class SeqClassifier(nn.Module):
    def __init__(self, backbone, hidden, n_labels, causal=True, pos_dim=2, dropout=0.1, family="gpt2"):
        super().__init__()
        self.backbone, self.causal, self.pos_dim, self.family = backbone, causal, pos_dim, family
        self.drop = nn.Dropout(dropout); self.head = nn.Linear(hidden + pos_dim, n_labels); self.prompt = None
    def forward(self, ids, mask, pos=None, return_emb=False):
        B = ids.size(0); lengths = mask.sum(1)
        if self.prompt is not None:                       # prompt tuning: learnable vectors in front of the input
            emb = self.backbone.get_input_embeddings()(ids)
            P = self.prompt.shape[0]
            emb = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(emb.dtype), emb], 1)
            mask = torch.cat([torch.ones(B, P, device=ids.device, dtype=mask.dtype), mask], 1); lengths = lengths + P
            h = self.backbone(inputs_embeds=emb, attention_mask=mask, use_cache=False).last_hidden_state
        else:
            h = self.backbone(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        if self.causal: pooled = h[torch.arange(B, device=h.device), lengths - 1]       # last real token sees the whole input
        else:
            m = mask.unsqueeze(-1).to(h.dtype); pooled = (h * m).sum(1) / m.sum(1)
        feats = pooled if (self.pos_dim == 0 or pos is None) else torch.cat([pooled, pos.to(pooled.dtype)], -1)
        logits = self.head(self.drop(feats))
        return (logits, pooled) if return_emb else logits

def count_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tot, tr
def model_size_mb(m):
    return (sum(p.numel() * p.element_size() for p in m.parameters()) + sum(b.numel() * b.element_size() for b in m.buffers())) / 2**20

# ---------------- adapter building blocks ----------------
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale

class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B=0 at init
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale      # = W0x + (alpha/r) B A (W0 x)

class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)   # row i rotated right by i
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)

class QVAdapt(nn.Module):                  # wraps GPT-2's fused c_attn (q|k|v); adapts q and v (k and v for IA3)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__()
        nx, nf = conv_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora":         self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi":   self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3":        self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora":            q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else:                              k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)

class IA3Input(nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(conv_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)

class LinAdapt(nn.Module):                 # same adapters for plain nn.Linear projections (non-GPT-2 backbones)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)

def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"

# ---------------- NF4 quantisation (QLoRA-style frozen base weights, 4-bit packed) ----------------
NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])
def nf4_quantize(w, block=64):
    flat = w.detach().float().reshape(-1); n = flat.numel(); pad = (-n) % block
    if pad: flat = F.pad(flat, (0, pad))
    blocks = flat.view(-1, block); absmax = blocks.abs().amax(dim=1).clamp_min(1e-8)
    normed = (blocks / absmax[:, None]).reshape(-1)
    code = NF4_CODE.to(flat.device); mids = (code[:-1] + code[1:]) / 2
    idx = torch.bucketize(normed, mids).to(torch.uint8)
    return ((idx[0::2] << 4) | idx[1::2]), absmax.half(), n
def nf4_dequant(packed, absmax, n, shape, block=64):
    code = NF4_CODE.to(packed.device)
    idx = torch.stack([(packed >> 4).long(), (packed & 0x0F).long()], dim=1).reshape(-1)
    return (code[idx].view(-1, block) * absmax.float()[:, None]).reshape(-1)[:n].view(shape)

class NF4Conv1D(nn.Module):
    def __init__(self, conv, block=64):
        super().__init__(); nx, nf = conv.weight.shape; self.nx, self.nf, self.block = nx, nf, block
        packed, absmax, n = nf4_quantize(conv.weight.data, block); self.n = n
        self.register_buffer("packed", packed); self.register_buffer("absmax", absmax); self.register_buffer("bias", conv.bias.data.clone())
    def forward(self, x):
        W = nf4_dequant(self.packed, self.absmax, self.n, (self.nx, self.nf), self.block).to(x.dtype)
        return x @ W + self.bias.to(x.dtype)

def conv_dims(m):
    if isinstance(m, NF4Conv1D): return m.nx, m.nf
    return tuple(m.weight.shape)                     # HF Conv1D weight is [in, out]

METHODS = ("full", "lora", "rslora", "qlora", "lorma_plus", "lorma_pi", "ia3", "prompt")
def apply_method(model, method, hp):
    assert method in METHODS, method
    if method == "full":
        for p in model.parameters(): p.requires_grad = True
        return model
    for p in model.backbone.parameters(): p.requires_grad = False
    r, alpha, drop = hp.get("r", 8), hp.get("alpha", 16), hp.get("adapter_dropout", 0.05)
    if method == "prompt":
        emb = model.backbone.get_input_embeddings().weight; P = hp.get("n_prompt", 20)
        init = emb[torch.randint(0, emb.shape[0], (P,))].detach().clone()
        model.prompt = nn.Parameter(init); return model
    kind = {"lora": "lora", "rslora": "lora", "qlora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]
    kw = dict(r=r, alpha=alpha, drop=drop, rs=(method == "rslora"))
    if model.family == "gpt2":
        for blk in model.backbone.h:
            if method == "qlora":
                for owner, name in ((blk.attn, "c_attn"), (blk.attn, "c_proj"), (blk.mlp, "c_fc"), (blk.mlp, "c_proj")):
                    setattr(owner, name, NF4Conv1D(getattr(owner, name)))
            assert isinstance(blk.attn.c_attn, (Conv1D, NF4Conv1D)), "unexpected c_attn type in this transformers version"
            blk.attn.c_attn = QVAdapt(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    else:
        if method == "qlora": raise NotImplementedError("QLoRA is implemented for the GPT-2 family only")
        names = ("k_proj", "v_proj") if kind == "ia3" else ("q_proj", "v_proj")
        wrap_linear_targets(model.backbone, kind, names, **({} if kind == "ia3" else kw))
    return model


In [ ]:
# ================= CELL 8 — training / evaluation harness + RESUMABLE CHECKPOINTS =================
# Recovery strategy:
#   latest.pt: full model + optimizer/scheduler/scaler + epoch + RNG state (resume after interruption)
#   best.pt: best validation model weights (use for inference)
#   result.json/logits/embeddings: completed experiment artifacts
K = len(LABELS)
HP_DEFAULT = {
    "full": dict(lr=3e-5), "lora": dict(lr=5e-4, r=8, alpha=16), "rslora": dict(lr=5e-4, r=8, alpha=16), "qlora": dict(lr=5e-4, r=8, alpha=16),
    "lorma_plus": dict(lr=5e-4, r=8, alpha=16), "lorma_pi": dict(lr=5e-4, r=8), "ia3": dict(lr=3e-3), "prompt": dict(lr=1e-2, n_prompt=20),
}
CFG_DEFAULT = dict(arch=None, method="full", ctx=True, epochs=8, patience=3, bs=16, warmup=0.1, wd=0.01, head_lr=1e-3, label_smooth=0.05,
                   class_weights=True, bb_dropout=0.1, k_left=3, k_right=1, seed=SEED, n_layers=None, kd=None, merge_dev=False,
                   fixed_epochs=None, hp={}, force=False, resume=True)
use_amp = device.type == "cuda"

def atomic_torch_save(obj, path):
    """Write checkpoint to a temporary file then atomically replace destination."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def checkpoint_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])

@torch.no_grad()
def predict(model, exs, bs=64, emb=False):
    model.eval(); lg = np.zeros((len(exs), K), np.float32); em = None
    order = sorted(range(len(exs)), key=lambda j: len(exs[j]["ids"]))
    for s in range(0, len(order), bs):
        bidx = order[s:s + bs]; ids, mask, pos, _ = collate(exs, bidx)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = model(ids.to(device), mask.to(device), pos.to(device), return_emb=emb)
        if emb:
            out, pooled = out
            if em is None: em = np.zeros((len(exs), pooled.shape[-1]), np.float16)
            em[bidx] = pooled.float().cpu().numpy().astype(np.float16)
        lg[bidx] = out.float().cpu().numpy()
    return (lg, em) if emb else lg

def snapshot(model):
    return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}

def restore(model, snap):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap: p.copy_(snap[n].to(p.device))

def run_one(name, dom, cfg):
    c = {**CFG_DEFAULT, **cfg}
    if not ARCHS:
        raise RuntimeError("ARCHS is empty; configure the Phase 1 checkpoint in Cell 6.")
    c["arch"] = c["arch"] or next(iter(ARCHS))
    hp = {**HP_DEFAULT[c["method"]], **c["hp"]}
    rdir = WORK / "runs" / f"{name}_{dom}"
    rdir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = rdir / "latest.pt", rdir / "best.pt"
    if (rdir / "result.json").exists() and not c["force"]:
        print(f"[{name}/{dom}] already done - skipping (force=True to redo)")
        return json.load(open(rdir / "result.json", encoding="utf-8"))

    seed_all(c["seed"]); t_all = time.time()
    tr_docs = DOCS[dom]["train"] + (DOCS[dom]["dev"] if c["merge_dev"] else [])
    kw = dict(ctx=c["ctx"], k_left=c["k_left"], k_right=c["k_right"])
    ex_tr = build_examples(tr_docs, **kw)
    ex_dv = build_examples(DOCS[dom]["dev"], **kw)
    ex_te = build_examples(DOCS[dom]["test"], **kw)
    y_tr = np.array([e["y"] for e in ex_tr]); y_dv = np.array([e["y"] for e in ex_dv])

    bb, hidden, causal, npos, fam = build_backbone(c["arch"], c["bb_dropout"], c["n_layers"])
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam)
    apply_method(model, c["method"], hp); model.to(device)
    longest = max(len(e["ids"]) for e in ex_tr + ex_dv + ex_te) + (hp.get("n_prompt", 0) if c["method"] == "prompt" else 0)
    assert longest <= npos, f"input length {longest} exceeds the model context {npos}"

    tot_p, tr_p = count_params(model); size_mb = model_size_mb(model)
    cnt = np.bincount(y_tr, minlength=K).astype(float); cw = None
    if c["class_weights"]:
        w = 1 / np.sqrt(np.maximum(cnt, 1)); w = w * cnt.sum() / (w * cnt).sum()
        cw = torch.tensor(w, dtype=torch.float32, device=device)
    T_logits = None
    if c["kd"]:
        assert not c["merge_dev"], "KD + merge_dev not supported"
        T_logits = np.load(WORK / "runs" / f"teacher_{dom}.npy")
        assert len(T_logits) == len(ex_tr), "teacher logits do not match the training examples"

    other = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    opt = torch.optim.AdamW(
        [{"params": other, "lr": hp["lr"]},
         {"params": list(model.head.parameters()), "lr": c["head_lr"]}],
        weight_decay=c["wd"])
    epochs = c["fixed_epochs"] or c["epochs"]
    spe = math.ceil(len(ex_tr) / c["bs"]); total = spe * epochs
    warm = int(c["warmup"] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda step: min((step + 1) / max(1, warm),
                              max(0.0, (total - step) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = random.Random(c["seed"])
    trainable = [p for p in model.parameters() if p.requires_grad]

    start_epoch, best, best_ep, bad, hist = 1, -1.0, 0, 0, []
    best_state = None
    if c["resume"] and latest_path.exists() and not c["force"]:
        try:
            ck = torch.load(latest_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["model_state"])
            opt.load_state_dict(ck["optimizer_state"])
            sched.load_state_dict(ck["scheduler_state"])
            scaler.load_state_dict(ck["scaler_state"])
            start_epoch = int(ck["epoch"]) + 1
            best, best_ep, bad = float(ck["best_f1"]), int(ck["best_epoch"]), int(ck["bad_epochs"])
            hist = ck.get("history", [])
            # Backward-compatible: older checkpoints embedded best weights.
            # New checkpoints keep best weights only in best.pt to avoid duplication.
            best_state = ck.get("best_model_state")
            if best_state is None and best_path.exists():
                try:
                    best_ck = torch.load(best_path, map_location="cpu", weights_only=False)
                    best_state = best_ck.get("model_state")
                    del best_ck
                except Exception as best_e:
                    print("   Could not load separate best.pt; continuing with latest weights:", repr(best_e))
            restore_rng_state(ck.get("rng_state"))
            print(f"[{name}/{dom}] RESUMING from epoch {start_epoch} (latest checkpoint epoch {ck['epoch']})")
        except Exception as e:
            print(f"[{name}/{dom}] WARNING: could not resume checkpoint ({e}); starting fresh.")
            start_epoch, best, best_ep, bad, hist, best_state = 1, -1.0, 0, 0, [], None
    if c["force"]:
        print(f"[{name}/{dom}] force=True: starting a fresh run; old checkpoints will be replaced.")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{name}/{dom}] arch={c['arch']} method={c['method']} ctx={c['ctx']} | train {len(ex_tr)} sents | "
          f"params total {tot_p/1e6:.1f}M trainable {tr_p/1e6:.3f}M ({100*tr_p/tot_p:.2f}%) | size {size_mb:.0f} MB")
    t0 = time.time()
    for ep in range(start_epoch, epochs + 1):
        model.train(); tl, nb = 0.0, 0
        for bidx in make_batches(ex_tr, c["bs"], True, rng):
            ids, mask, pos, y = collate(ex_tr, bidx)
            ids, mask, pos, y = ids.to(device), mask.to(device), pos.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, mask, pos)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=c["label_smooth"])
            if T_logits is not None:
                temp, alpha = c["kd"].get("T", 2.0), c["kd"].get("alpha", 0.5)
                tl_ = torch.tensor(T_logits[bidx], device=device)
                kl = F.kl_div(F.log_softmax(logits.float() / temp, -1),
                              F.softmax(tl_ / temp, -1), reduction="batchmean") * temp * temp
                loss = alpha * loss + (1 - alpha) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1

        if c["merge_dev"]:
            epoch_f1 = None
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} (train+dev refit: no validation)")
            best_state = cpu_state_dict(model); best_ep = ep
        else:
            rep_ep = prf_report(y_dv, predict(model, ex_dv).argmax(1), K)
            epoch_f1 = float(rep_ep["macro_f1"])
            hist.append(dict(epoch=ep, loss=tl / max(nb, 1), dev_macro_f1=epoch_f1))
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} | dev macro-F1 {epoch_f1:.4f} "
                  f"(P {rep_ep['macro_p']:.3f} R {rep_ep['macro_r']:.3f}) acc {rep_ep['acc']:.3f}")
            if epoch_f1 > best + 1e-4:
                best, best_ep, bad = epoch_f1, ep, 0
                best_state = cpu_state_dict(model)
                atomic_torch_save({
                    "model_state": best_state, "epoch": ep, "best_f1": best,
                    "best_epoch": best_ep, "arch": c["arch"], "method": c["method"],
                    "domain": dom, "experiment": name, "config": c,
                }, best_path)
            else:
                bad += 1

        # Save after EVERY completed epoch. A runtime interruption loses at most the in-progress epoch.
        atomic_torch_save({
            "format_version": 1, "experiment": name, "domain": dom, "epoch": ep,
            "model_state": cpu_state_dict(model), "optimizer_state": opt.state_dict(),
            "scheduler_state": sched.state_dict(), "scaler_state": scaler.state_dict(),
            "best_f1": best, "best_epoch": best_ep, "bad_epochs": bad,
            # Do not embed best_model_state here: best.pt is the single rolling copy.
            "history": hist, "rng_state": checkpoint_rng_state(),
            "config": c, "hp": hp, "saved_unix": time.time(),
        }, latest_path)
        print(f"   checkpoint saved: {latest_path.name} (epoch {ep})")

        # Remote per-epoch publishing is intentionally disabled by default.
        # This prevents creating a new Kaggle Dataset version every few epochs.
        # Local rolling latest.pt/best.pt remain available for same-session resume.


        if not c["merge_dev"] and bad >= c["patience"] and not c["fixed_epochs"]:
            print("   early stop"); break

    # If all epochs were already completed in a valid checkpoint, use its best state.
    if best_state is not None:
        model.load_state_dict(best_state)
        atomic_torch_save({
            "model_state": cpu_state_dict(model), "epoch": best_ep or epochs,
            "best_f1": best, "best_epoch": best_ep, "arch": c["arch"],
            "method": c["method"], "domain": dom, "experiment": name,
            "config": c, "saved_unix": time.time(),
        }, best_path)

    train_sec = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    t1 = time.time(); lg_dv, em_dv = predict(model, ex_dv, emb=True)
    sps = len(ex_dv) / max(time.time() - t1, 1e-6)
    lg_te, em_te = predict(model, ex_te, emb=True)
    lg_tr, em_tr = predict(model, ex_tr, emb=True)
    for nm, arr in dict(dev_logits=lg_dv, test_logits=lg_te, train_logits=lg_tr,
                        dev_emb=em_dv, test_emb=em_te, train_emb=em_tr).items():
        np.save(rdir / f"{nm}.npy", arr)
    np.save(rdir / "train_y.npy", y_tr)
    rep = prf_report(y_dv, lg_dv.argmax(1), K)
    res = dict(name=name, domain=dom, arch=c["arch"], method=c["method"], ctx=c["ctx"],
               kd=bool(c["kd"]), merge_dev=c["merge_dev"], n_layers=c["n_layers"],
               best_epoch=best_ep or epochs, epochs_run=len(hist) or epochs,
               dev_macro_f1=rep["macro_f1"], dev_macro_p=rep["macro_p"], dev_macro_r=rep["macro_r"],
               dev_macro_f1_all13=rep["macro_f1_all"], dev_acc=rep["acc"],
               params_total=tot_p, params_trainable=tr_p, trainable_pct=100 * tr_p / tot_p,
               model_size_mb=size_mb, peak_gpu_mb=peak_mb, train_sec=train_sec,
               sents_per_sec=sps, cfg=json.dumps({**c, "hp": hp}, default=str),
               **{f"f1_{l}": float(rep["f1"][i]) for i, l in enumerate(LABELS)})
    tmp_result = rdir / "result.json.tmp"
    with open(tmp_result, "w", encoding="utf-8") as f:
        json.dump({**res, "confusion": rep["cm"].tolist(), "history": hist}, f, indent=2)
    os.replace(tmp_result, rdir / "result.json")
    newrow = pd.DataFrame([res])
    if LEDGER.exists():
        old = pd.read_csv(LEDGER)
        old = old[~((old["name"] == name) & (old["domain"] == dom))]
        newrow = pd.concat([old, newrow], ignore_index=True)
    newrow.to_csv(LEDGER, index=False)
    if PUBLISH_AT_EACH_EXPERIMENT_END and DURABLE_PUBLISH_ENABLED:
        publish_recovery_dataset(reason=f"{name}_{dom}_experiment_complete")
    print(f"[{name}/{dom}] DONE dev macro-F1 {rep['macro_f1']:.4f} @epoch {res['best_epoch']} | "
          f"{train_sec/60:.1f} min | peak {peak_mb:.0f} MB | {sps:.0f} sent/s")
    del model, opt, scaler, best_state
    gc.collect(); torch.cuda.empty_cache()
    return res

def run_experiment(name, domains=("CL", "IT"), **cfg):
    return {d: run_one(name, d, cfg) for d in domains}

def make_teacher(run_names, dom):
    """Geometric-mean ensemble of finished runs' train logits -> teacher for KD."""
    lp = [np.log(softmax_np(np.load(WORK / "runs" / f"{n}_{dom}" / "train_logits.npy")) + 1e-9)
          for n in run_names]
    np.save(WORK / "runs" / f"teacher_{dom}.npy", np.mean(lp, 0).astype(np.float32))
    print(f"teacher_{dom}: ensemble of {run_names}")

def show_ledger(cols=None):
    if not LEDGER.exists():
        print("No completed experiments in ledger yet.")
        return pd.DataFrame()
    df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
    cols = cols or ["name", "domain", "arch", "method", "ctx", "kd", "dev_macro_f1",
                    "dev_macro_p", "dev_macro_r", "best_epoch", "params_trainable",
                    "model_size_mb", "peak_gpu_mb", "train_sec"]
    print(df[cols].round(4).to_string(index=False))
    return df


In [ ]:
# ================= CELL 9 — smoke tests (run once BEFORE any long training; ~2-3 min) =================
def smoke_tests(arch=None):
    if not ARCHS:
        raise RuntimeError(
            "ARCHS is empty: no compatible Phase 1 checkpoint was detected. "
            "Run CELL 2 to inspect checkpoints, then edit ARCHS in CELL 6 "
            "with the correct checkpoint path and family='gpt2' for the GPT-2 checkpoint."
        )
    arch = arch or next(iter(ARCHS))
    if arch not in ARCHS:
        raise KeyError(f"Unknown architecture {arch!r}. Registered: {list(ARCHS)}")
    print("testing architecture:", arch)
    bb, hidden, causal, npos, fam = build_backbone(arch, bb_dropout=0.0)
    base = SeqClassifier(bb, hidden, K, causal, 2, 0.0, fam).eval()
    exs = build_examples(DOCS["CL"]["train"][:1], True)[:4]; ids, mask, pos, y = collate(exs, list(range(len(exs))))
    with torch.no_grad(): ref = base(ids, mask, pos)
    rows = []
    for m in ("lora", "rslora", "lorma_plus", "lorma_pi", "ia3", "qlora", "prompt"):
        mdl = copy.deepcopy(base); apply_method(mdl, m, {"n_prompt": 8}); mdl.eval()
        with torch.no_grad(): out = mdl(ids, mask, pos)
        diff = (out - ref).abs().max().item(); tot, tr = count_params(mdl)
        if m in ("qlora", "prompt"): ok = bool(torch.isfinite(out).all()); note = "changes outputs by design (quantisation / extra tokens)"
        else: ok = diff < 1e-3; note = "must equal the base model at initialisation"
        rows.append(dict(method=m, max_logit_diff=round(diff, 5), trainable_M=round(tr / 1e6, 3), size_MB=round(model_size_mb(mdl)), ok=ok, check=note))
        del mdl; gc.collect()
    pi = apply_method(copy.deepcopy(base), "lorma_pi", {}).backbone.h[0].attn.c_attn.q.matrix()
    rows.append(dict(method="lorma_pi: matrix==I", max_logit_diff=round((pi - torch.eye(pi.shape[0])).abs().max().item(), 6), ok=bool(torch.allclose(pi, torch.eye(pi.shape[0]), atol=1e-6)), check="permutation inflation starts as identity"))
    df = pd.DataFrame(rows); print(df.to_string(index=False)); assert df["ok"].all(), "an adapter test failed - do not run that method"
    if device.type == "cuda":
        for m in ("lorma_plus", "full"):
            mdl = SeqClassifier(*build_backbone(arch, 0.0)[:2], K, causal, 2, 0.0, fam); apply_method(mdl, m, {}); mdl.to(device).train()
            ids_, mask_, pos_, y_ = [t.to(device) for t in collate(exs, list(range(len(exs))))]
            opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad], lr=1e-3 if m != "full" else 1e-4); L = []
            for _ in range(25):
                loss = F.cross_entropy(mdl(ids_, mask_, pos_), y_); opt.zero_grad(); loss.backward(); opt.step(); L.append(loss.item())
            print(f"overfit test [{m}]: loss {L[0]:.3f} -> {L[-1]:.3f}"); assert L[-1] < 0.7 * L[0], f"{m} did not learn"
            del mdl, opt; gc.collect(); torch.cuda.empty_cache()
    print("\nALL SMOKE TESTS PASSED")
smoke_tests()


### Smoke tests passed? Then run the experiments in this order (compulsory ones first). Each run skips itself if its result already exists, so it is safe to re-run after a crash.

In [ ]:
# ================= CELL 10 — EXPERIMENT PLAN =================
# Only the GPT-2-style 3,000-step checkpoint is currently registered.
# This does NOT satisfy four-architecture coverage until the other Phase 1 models
# are connected and independently smoke-tested.
if not ARCHS:
    raise RuntimeError("ARCHS is empty. Fix checkpoint registration before experiments.")
ARCH_DEFAULT = "gpt2_a"
PLAN = {
    "E0b_full_ctx":  dict(method="full",       ctx=True,  arch=ARCH_DEFAULT),
    "E2_lorma_plus": dict(method="lorma_plus", ctx=True,  arch=ARCH_DEFAULT),
    "E1_lora":       dict(method="lora",       ctx=True,  arch=ARCH_DEFAULT),
    "E3_ia3":        dict(method="ia3",        ctx=True,  arch=ARCH_DEFAULT),
    "E4_qlora":      dict(method="qlora",      ctx=True,  arch=ARCH_DEFAULT),
    "E5_prompt":     dict(method="prompt",     ctx=True,  arch=ARCH_DEFAULT),
    "E0a_full_sent": dict(method="full",       ctx=False, arch=ARCH_DEFAULT),
}
print(pd.DataFrame(PLAN).T.to_string())


In [ ]:
# ================= CELL 11 — RUN A: baseline + the compulsory LoRMA experiment (both domains) =================
for nm in ("E0b_full_ctx", "E2_lorma_plus"):
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:
# ================= CELL 12 — RUN B: the other adaptation techniques + context ablation =================
for nm in ("E1_lora", "E3_ia3", "E4_qlora", "E0a_full_sent"):     # add "E5_prompt" if you want a sixth technique
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:

# HOTFIX — recover missing KD teacher logits without retraining
import shutil
import gc
from pathlib import Path

TEACHERS = ["E0b_full_ctx", "E2_lorma_plus"]
DOMAINS = ("CL", "IT")
INPUT_ROOT = Path("/kaggle/input")

def _find_recovery_file(run_name, dom, filename):
    """Search attached Kaggle Datasets for a missing artifact."""
    run_folder = f"{run_name}_{dom}"
    for root in INPUT_ROOT.iterdir():
        if not root.is_dir():
            continue
        # Search for the expected run folder/file under each attached Dataset.
        for p in root.rglob(filename):
            if p.parent.name == run_folder:
                return p
    return None

def _ensure_train_logits(run_name, dom):
    rdir = WORK / "runs" / f"{run_name}_{dom}"
    rdir.mkdir(parents=True, exist_ok=True)
    logits_path = rdir / "train_logits.npy"

    if logits_path.is_file():
        return np.load(logits_path, mmap_mode="r")

    # First choice: restore the exact saved logits from an attached Dataset.
    source = _find_recovery_file(run_name, dom, "train_logits.npy")
    if source is not None:
        shutil.copy2(source, logits_path)
        print(f"[KD recovery] Restored logits: {source}")
        return np.load(logits_path, mmap_mode="r")

    # Second choice: find best.pt and regenerate logits from the trained model.
    ckpt = rdir / "best.pt"
    source_ckpt = _find_recovery_file(run_name, dom, "best.pt")
    if not ckpt.is_file() and source_ckpt is not None:
        shutil.copy2(source_ckpt, ckpt)

    if not ckpt.is_file():
        raise FileNotFoundError(
            f"No train_logits.npy or best.pt found for {run_name}/{dom}. "
            "Attach the recovery Dataset containing this run's logits or best.pt. "
            "A result.json alone cannot reconstruct the teacher predictions."
        )

    checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False)
    cfg = checkpoint.get("config")

    if not isinstance(cfg, dict) or not cfg.get("arch") or not cfg.get("method"):
        raise RuntimeError(
            f"{ckpt} lacks the architecture/method configuration required "
            "to regenerate logits safely."
        )

    tr_docs = DOCS[dom]["train"] + (
        DOCS[dom]["dev"] if cfg.get("merge_dev", False) else []
    )
    kw = dict(
        ctx=cfg.get("ctx", True),
        k_left=cfg.get("k_left", 3),
        k_right=cfg.get("k_right", 1),
    )
    examples = build_examples(tr_docs, **kw)
    method = cfg["method"]
    hp = {**HP_DEFAULT[method], **(cfg.get("hp") or {})}

    bb, hidden, causal, npos, fam = build_backbone(
        cfg["arch"], cfg.get("bb_dropout", 0.1), cfg.get("n_layers")
    )
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam)
    apply_method(model, method, hp)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()

    with torch.no_grad():
        logits = predict(model, examples, bs=64, emb=False).astype(np.float32)

    np.save(logits_path, logits)
    del model, checkpoint, logits, examples
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[KD recovery] Regenerated {logits_path.name} without retraining.")
    return np.load(logits_path, mmap_mode="r")


def make_teacher(run_names, dom):
    lp = []
    for name in run_names:
        logits = _ensure_train_logits(name, dom)
        lp.append(np.log(softmax_np(np.asarray(logits)) + 1e-9))

    np.save(
        WORK / "runs" / f"teacher_{dom}.npy",
        np.mean(lp, axis=0).astype(np.float32),
    )
    print(f"[KD recovery] teacher_{dom} ready from {run_names}")
    del lp
    gc.collect()


# Check all required artifacts BEFORE starting KD.
for dom in DOMAINS:
    print(f"\nPreparing teacher for {dom}...")
    make_teacher(TEACHERS, dom)

print("\nTeacher logits are ready. You can now rerun Cell 13.")


In [ ]:
from pathlib import Path
import json
import shutil

ROOT = Path("/kaggle/working/phase3_rr/runs")

free = shutil.disk_usage("/kaggle/working").free
print(f"Free disk: {free / 2**30:.2f} GiB")

for name in [
    "E0a_full_sent_CL", "E0a_full_sent_IT",
    "E1_lora_CL", "E1_lora_IT",
    "E3_ia3_CL", "E3_ia3_IT",
    "E4_qlora_CL", "E4_qlora_IT",
]:
    d = ROOT / name
    print(f"\n{name}:")
    if not d.exists():
        print("  Run directory missing")
        continue

    for fname in ["best.pt", "latest.pt", "result.json"]:
        p = d / fname
        if p.exists():
            print(f"  {fname}: {p.stat().st_size / 2**30:.2f} GiB")

    rp = d / "result.json"
    if rp.exists():
        try:
            data = json.loads(rp.read_text())
            print("  result.json:", data)
        except Exception as e:
            print("  Could not read result.json:", e)

In [ ]:

# ================= CELL 13 — RUN C: knowledge distillation =================
from pathlib import Path
import numpy as np

TEACHERS = ["E0b_full_ctx", "E2_lorma_plus"]

def make_teacher(run_names, dom):
    """Load teacher logits from local or attached recovery Dataset files."""
    import gc

    log_probs = []
    for name in run_names:
        filename = "train_logits.npy"
        run_folder = f"{name}_{dom}"

        # 1. Look in the expected local run directory.
        candidates = [
            WORK / "runs" / run_folder / filename,
        ]

        # 2. Search attached Kaggle Datasets without copying files.
        input_root = Path("/kaggle/input")
        if input_root.exists():
            candidates.extend(
                p for root in input_root.iterdir() if root.is_dir()
                for p in root.rglob(filename)
                if p.parent.name == run_folder
            )

        source = next((p for p in candidates if p.is_file()), None)
        if source is None:
            raise FileNotFoundError(
                f"Missing {filename} for {run_folder}. "
                "Attach the recovery Dataset containing this run's logits. "
                "If logits were deleted, regenerate them from best.pt first."
            )

        print(f"[KD] Loading {name}/{dom} from {source}")
        logits = np.load(source, mmap_mode="r")
        log_probs.append(np.log(softmax_np(np.asarray(logits)) + 1e-9))
        del logits
        gc.collect()

    out = WORK / "runs" / f"teacher_{dom}.npy"
    np.save(out, np.mean(log_probs, axis=0).astype(np.float32))
    print(f"[KD] Saved {out}")
    del log_probs
    gc.collect()


# Prepare both teacher ensembles before starting student training.
for dom in ("CL", "IT"):
    make_teacher(TEACHERS, dom)

run_experiment(
    "E6_kd_student", method="full", ctx=True,
    arch=ARCH_DEFAULT, n_layers=6,
    kd=dict(T=2.0, alpha=0.5)
)

run_experiment(
    "E6_student_nokd", method="full", ctx=True,
    arch=ARCH_DEFAULT, n_layers=6
)

show_ledger()


In [ ]:
# ================= CELL 14 — comparison tables, per-class F1, confusion matrices, architecture coverage =================
import matplotlib; import matplotlib.pyplot as plt
df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
df = df[~df["merge_dev"].astype(bool)]
main = df.pivot_table(index="name", columns="domain", values="dev_macro_f1").round(4); main["mean"] = main.mean(1)
print("dev macro-F1 by experiment and domain\n", main.sort_values("mean", ascending=False).to_string())
eff = df.groupby("name")[["params_trainable", "model_size_mb", "peak_gpu_mb", "train_sec", "sents_per_sec"]].mean().round(1); print("\nefficiency (mean over domains)\n", eff.to_string())
pc = df.set_index(["name", "domain"])[[f"f1_{l}" for l in LABELS]].round(3); pc.columns = LABELS; print("\nper-class F1\n", pc.to_string())
print("\narchitecture coverage (all four Phase 1 architectures must appear):\n", df.groupby("arch")["name"].nunique().to_string())
def plot_cm(name, dom):
    r = json.load(open(WORK / "runs" / f"{name}_{dom}" / "result.json")); cm = np.array(r["confusion"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(8, 7)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_xticklabels(LABELS, rotation=90, fontsize=7); ax.set_yticks(range(K)); ax.set_yticklabels(LABELS, fontsize=7)
    ax.set_title(f"{name} / {dom}  (row-normalised)"); plt.colorbar(im); plt.tight_layout(); plt.savefig(WORK / f"cm_{name}_{dom}.png", dpi=120); plt.show()
BEST = main["mean"].idxmax(); print("\nbest by mean dev macro-F1:", BEST)
for dom in ("CL", "IT"): plot_cm(BEST, dom)


In [ ]:

# ================= CELL 15 — POST-HOC TUNING (RECOVERY-SAFE) =================
from pathlib import Path
import numpy as np
import torch
import gc

def _ensure_posthoc_logits(name, dom):
    """Recover dev/test logits from attached files or best.pt; no retraining."""
    rdir = WORK / "runs" / f"{name}_{dom}"
    rdir.mkdir(parents=True, exist_ok=True)

    missing = [
        split for split in ("dev", "test")
        if not (rdir / f"{split}_logits.npy").is_file()
    ]
    if not missing:
        return

    # Search attached recovery Datasets for the missing arrays first.
    for split in list(missing):
        found = None
        for root in Path("/kaggle/input").glob("*"):
            if not root.is_dir():
                continue
            for p in root.rglob(f"{split}_logits.npy"):
                if p.parent.name == f"{name}_{dom}":
                    found = p
                    break
            if found:
                break
        if found:
            import shutil
            shutil.copy2(found, rdir / found.name)
            missing.remove(split)

    if not missing:
        return

    # Recompute missing predictions from the trained best checkpoint.
    ckpt_path = rdir / "best.pt"
    if not ckpt_path.is_file():
        found = None
        for root in Path("/kaggle/input").glob("*"):
            if root.is_dir():
                for p in root.rglob("best.pt"):
                    if p.parent.name == f"{name}_{dom}":
                        found = p
                        break
            if found:
                break

        if found is None:
            raise FileNotFoundError(
                f"Missing {name}_{dom}/best.pt. Cannot regenerate {missing} "
                "without the trained checkpoint. Attach the recovery Dataset "
                "containing best.pt."
            )
        ckpt_path = found

    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ck.get("config")
    if not isinstance(cfg, dict):
        raise RuntimeError(f"{ckpt_path} has no saved training config.")

    arch = cfg.get("arch", ck.get("arch"))
    method = cfg.get("method", ck.get("method"))
    if not arch or not method:
        raise RuntimeError(f"Missing architecture/method metadata in {ckpt_path}")

    hp = ck.get("hp") or {**HP_DEFAULT[method], **cfg.get("hp", {})}
    bb, hidden, causal, npos, fam = build_backbone(
        arch, cfg.get("bb_dropout", 0.1), cfg.get("n_layers")
    )
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam)
    apply_method(model, method, hp)
    model.load_state_dict(ck["model_state"])
    model.to(device).eval()

    kw = dict(
        ctx=cfg.get("ctx", True),
        k_left=cfg.get("k_left", 3),
        k_right=cfg.get("k_right", 1),
    )

    with torch.no_grad():
        for split in missing:
            examples = build_examples(DOCS[dom][split], **kw)
            logits = predict(model, examples, emb=False)
            np.save(rdir / f"{split}_logits.npy", logits.astype(np.float32))
            print(f"[Recovery] Regenerated {name}/{dom}/{split}_logits.npy")
            del examples, logits
            gc.collect()

    del model, ck
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Ensure logits exist for the selected best run in both domains.
for dom in ("CL", "IT"):
    _ensure_posthoc_logits(BEST, dom)


# Safe post-hoc tuning: transition smoothing only.
# kNN is omitted because the compact recovery may not contain embeddings.
def tune_posthoc(names, dom):
    ex_dv = build_examples(DOCS[dom]["dev"], True)
    y = np.array([e["y"] for e in ex_dv])
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K)

    p = ens_probs(names, dom, "dev")
    rows = []
    for sm in (0.0, 0.25, 0.5, 1.0):
        pred = predict_labels(p, ex_dv, logT, logS, sm)
        score = prf_report(y, pred, K)["macro_f1"]
        rows.append({"domain": dom, "smooth": sm, "dev_macro_f1": score})

    t = pd.DataFrame(rows)
    base = t.loc[t["smooth"] == 0.0, "dev_macro_f1"].iloc[0]
    print(t.round(4).to_string(index=False))
    print(f"Baseline {base:.4f}; best {t.dev_macro_f1.max():.4f}")
    return t, base


CANDIDATES = {dom: [BEST] for dom in ("CL", "IT")}
for dom in ("CL", "IT"):
    _ = tune_posthoc(CANDIDATES[dom], dom)

print("\nPost-hoc tuning complete. Continue to Cell 16 for final predictions.")


In [ ]:

# ================= CELL 16 — FINAL PREDICTIONS + VALIDATION + ZIP =================

from pathlib import Path
from collections import defaultdict, Counter
import json
import zipfile
import time

# Ensure output directory exists.
WORK = Path(WORK)
WORK.mkdir(parents=True, exist_ok=True)

# Choose the validated best configuration.
FINAL = {
    "CL": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
    "IT": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
}

PREDICTIONS = {}

# ---------- Generate and validate per-domain predictions ----------
for dom in ("CL", "IT"):
    f = FINAL[dom]

    ex_te = build_examples(
        DOCS[dom]["test"],
        ctx=PLAN[f["names"][0]]["ctx"],
    )

    probs = posthoc_probs(
        f["names"],
        dom,
        "test",
        knn_lam=f["knn_lam"],
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"] + DOCS[dom]["dev"],
        LABEL2ID,
        K,
    )

    yhat = predict_labels(probs, ex_te, logT, logS, f["smooth"])

    assert len(yhat) == len(ex_te), (
        f"{dom}: got {len(yhat)} predictions for {len(ex_te)} examples"
    )

    by_doc = defaultdict(list)
    ordered = sorted(
        zip(ex_te, yhat),
        key=lambda pair: (pair[0]["doc"], pair[0]["i"]),
    )

    for e, y_ in ordered:
        by_doc[e["doc"]].append(LABELS[int(y_)])

    pred_obj = {}

    for d in DOCS[dom]["test"]:
        doc_id = d["id"]
        labs = by_doc[doc_id]

        assert len(labs) == len(d["sents"]), (
            f"{dom}/{doc_id}: predicted {len(labs)} labels "
            f"for {len(d['sents'])} sentences"
        )
        assert all(lab in LABEL2ID for lab in labs), (
            f"{dom}/{doc_id}: invalid label found"
        )

        pred_obj[doc_id] = labs

    # Validate document IDs and order against the source test object.
    assert list(pred_obj.keys()) == list(TEST_OBJ[dom].keys()), (
        f"{dom}: document IDs/order mismatch"
    )

    for doc_id, sentences in TEST_OBJ[dom].items():
        assert len(pred_obj[doc_id]) == len(sentences), (
            f"{dom}/{doc_id}: sentence count mismatch"
        )

    PREDICTIONS[dom] = pred_obj

    out_file = WORK / f"prediction_{dom}.json"
    with open(out_file, "w", encoding="utf-8") as f_out:
        json.dump(pred_obj, f_out, ensure_ascii=False, indent=2)

    print(f"{dom}: {sum(map(len, pred_obj.values()))} sentence labels")
    print(
        "  label distribution:",
        dict(Counter(l for labs in pred_obj.values() for l in labs)),
    )
    print("  saved:", out_file)


# ---------- Merge into required flat document-ID mapping ----------
all_ids = list(PREDICTIONS["CL"].keys()) + list(PREDICTIONS["IT"].keys())
assert len(all_ids) == len(set(all_ids)), (
    "Document IDs overlap between CL and IT; refusing to overwrite predictions."
)

combined = {
    **PREDICTIONS["CL"],
    **PREDICTIONS["IT"],
}

combined_file = WORK / "prediction.json"
with open(combined_file, "w", encoding="utf-8") as f_out:
    json.dump(combined, f_out, ensure_ascii=False, indent=2)

# ---------- Validate the flat JSON ----------
assert list(combined.keys()) == (
    list(TEST_OBJ["CL"].keys()) + list(TEST_OBJ["IT"].keys())
), "Combined document IDs/order do not match the expected CL + IT order."

for dom in ("CL", "IT"):
    for doc_id, sentences in TEST_OBJ[dom].items():
        assert doc_id in combined, f"Missing document: {doc_id}"
        assert len(combined[doc_id]) == len(sentences), (
            f"{doc_id}: label count differs from sentence count"
        )
        assert all(label in LABEL2ID for label in combined[doc_id]), (
            f"{doc_id}: invalid rhetorical-role label"
        )

print("\nCombined prediction file:", combined_file)
print("Total documents:", len(combined))
print("Total sentence labels:", sum(len(v) for v in combined.values()))


# ---------- Create submission ZIP ----------
# Keep prediction.json directly at the ZIP root.
# Per-domain debug JSON files are deliberately excluded from submission.zip.
zip_path = Path("/kaggle/working/submission.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(combined_file, "prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"], (
        f"Unexpected submission ZIP contents: {zf.namelist()}"
    )
    check = json.load(zf.open("prediction.json"))

    assert list(check.keys()) == list(combined.keys())
    assert check == combined

print("Submission ZIP validated:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))


# ================= DURABLE RECOVERY BUNDLE =================
RECOVERY_DIR = Path("/kaggle/working/phase3_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_ZIP = Path("/kaggle/working/phase3_recovery.zip")

manifest = {
    "created_unix": time.time(),
    "project": "CS781 Phase 3 Rhetorical Role Prediction",
    "source_dir": str(WORK),
    "checkpoint_policy": (
        "Rolling latest.pt stores resumable state; best.pt stores best validation weights. "
        "No epoch-numbered checkpoints are retained; logits/embeddings are excluded from recovery ZIP."
    ),
    "contains": [
         "Rolling latest.pt and best.pt checkpoints",
        "Small result.json summaries and ledger",
        "Prediction JSON files, when present (no logits/embeddings)",
        "Recovery manifest",
    ],
    "durability_note": (
        "This ZIP remains session-local until downloaded or published "
        "as a version of a Kaggle Dataset."
    ),
}

manifest_file = RECOVERY_DIR / "recovery_manifest.json"
with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if RECOVERY_ZIP.exists():
    RECOVERY_ZIP.unlink()

with zipfile.ZipFile(
    RECOVERY_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    # Include only rolling checkpoints and small recovery metadata.
    # Large logits/embedding arrays (*.npy) are intentionally excluded.
    allowed_root_files = {
        "ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"
    }
    allowed_run_files = {"latest.pt", "best.pt", "result.json"}
    for src in WORK.rglob("*"):
        if not src.is_file() or src.name.endswith(".tmp"):
            continue
        if src.resolve() == RECOVERY_ZIP.resolve():
            continue
        if RECOVERY_DIR in src.parents:
            continue

        rel = src.relative_to(WORK)
        if rel.parts and rel.parts[0] == "runs":
            if src.name not in allowed_run_files:
                continue
        elif rel.parts and len(rel.parts) == 1:
            if src.name not in allowed_root_files:
                continue
        else:
            continue

        zf.write(src, Path("phase3_rr") / rel)

    zf.write(manifest_file, "recovery_manifest.json")

with zipfile.ZipFile(RECOVERY_ZIP, "r") as zf:
    bad = zf.testzip()
    assert bad is None, f"Recovery ZIP failed at: {bad}"
    print("\nRecovery bundle verified.")
    print("Files included:", len(zf.namelist()))

print("Recovery ZIP:", RECOVERY_ZIP)
print("Recovery ZIP size (MB):", round(RECOVERY_ZIP.stat().st_size / (1024**2), 2))


# ================= FINAL DURABLE DATASET PUBLISH =================
if DURABLE_PUBLISH_ENABLED:
    publish_recovery_dataset(reason="final_notebook_recovery")
else:
    print("\nFINAL DURABLE PUBLISH SKIPPED.")
    print("Configure Kaggle Secrets in Cell 0b to publish the backup.")
    print("Local recovery ZIP:", RECOVERY_ZIP)

print("\nAll prediction structure and ZIP validations passed.")


In [ ]:

# EMERGENCY DISK CLEANUP + REGENERATE TUNED SUBMISSION
from pathlib import Path
import shutil, zipfile, json, gc
from collections import defaultdict, Counter
import numpy as np

WORK = Path(WORK)
free_before = shutil.disk_usage("/kaggle/working").free
print(f"Free disk before cleanup: {free_before / 2**30:.2f} GiB")

# Remove only regenerable staging/partial backup artifacts.
# Original runs, checkpoints, logits and embeddings are preserved.
for p in [
    Path("/kaggle/working/phase3_recovery.zip"),
    Path("/kaggle/working/phase3_publish"),
    Path("/kaggle/working/phase3_recovery"),
]:
    try:
        if p.is_dir():
            shutil.rmtree(p)
            print("Removed temporary directory:", p)
        elif p.exists():
            p.unlink()
            print("Removed temporary file:", p)
    except Exception as e:
        print(f"Could not remove {p}: {e}")

gc.collect()
free_now = shutil.disk_usage("/kaggle/working").free
print(f"Free disk after cleanup: {free_now / 2**30:.2f} GiB")

# Ensure the chosen model has test logits. This uses saved artifacts or
# the previously defined recovery helper; it does not retrain the model.
for dom in ("CL", "IT"):
    if not (WORK / "runs" / f"{BEST}_{dom}" / "test_logits.npy").is_file():
        if "_ensure_posthoc_logits" not in globals():
            raise RuntimeError(
                f"Missing test logits for {BEST}/{dom}. "
                "Run the existing post-hoc artifact recovery helper first."
            )
        _ensure_posthoc_logits(BEST, dom)

FINAL = {
    "CL": dict(names=[BEST], knn_lam=1.0, smooth=1.0),
    "IT": dict(names=[BEST], knn_lam=1.0, smooth=1.0),
}

PREDICTIONS = {}

for dom in ("CL", "IT"):
    f = FINAL[dom]
    ex_te = build_examples(
        DOCS[dom]["test"],
        ctx=PLAN[f["names"][0]]["ctx"],
    )
    probs = posthoc_probs(
        f["names"], dom, "test", knn_lam=f["knn_lam"]
    )

    # Match the existing final-submission transition setup.
    logT, logS = transition_logprobs(
        DOCS[dom]["train"] + DOCS[dom]["dev"], LABEL2ID, K
    )
    yhat = predict_labels(probs, ex_te, logT, logS, f["smooth"])

    assert len(yhat) == len(ex_te)
    by_doc = defaultdict(list)
    for e, y in sorted(
        zip(ex_te, yhat), key=lambda pair: (pair[0]["doc"], pair[0]["i"])
    ):
        by_doc[e["doc"]].append(LABELS[int(y)])

    pred_obj = {}
    for d in DOCS[dom]["test"]:
        labs = by_doc[d["id"]]
        assert len(labs) == len(d["sents"]), (
            f"{dom}/{d['id']}: sentence count mismatch"
        )
        assert all(x in LABEL2ID for x in labs)
        pred_obj[d["id"]] = labs

    assert list(pred_obj) == list(TEST_OBJ[dom])
    PREDICTIONS[dom] = pred_obj
    print(
        f"{dom}: {sum(map(len, pred_obj.values()))} labels; "
        f"smoothing={f['smooth']}"
    )
    del probs, ex_te, yhat
    gc.collect()

combined = {**PREDICTIONS["CL"], **PREDICTIONS["IT"]}
expected_ids = list(TEST_OBJ["CL"]) + list(TEST_OBJ["IT"])
assert list(combined) == expected_ids

for dom in ("CL", "IT"):
    for doc_id, sents in TEST_OBJ[dom].items():
        assert len(combined[doc_id]) == len(sents)

# Write JSON and the small submission ZIP only.
combined_file = WORK / "prediction.json"
with combined_file.open("w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

zip_path = Path("/kaggle/working/submission.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(combined_file, "prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    check = json.load(zf.open("prediction.json"))
    assert check == combined

print("\nSUCCESS: tuned submission generated and validated.")
print("Submission:", zip_path)
print("Documents:", len(combined))
print("Sentence labels:", sum(len(v) for v in combined.values()))
print("ZIP size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")
print("Free disk now:", round(
    shutil.disk_usage("/kaggle/working").free / 2**30, 2
), "GiB")
print("\nDownload submission.zip from Kaggle Output before further cleanup.")


In [ ]:

# CS781 — COMPACT PERSISTENT BACKUP, NO LARGE RECOVERY ZIP
import os, json, shutil, subprocess, sys, time, gc
from pathlib import Path

WORK = Path(WORK)
PUBLISH_DIR = Path("/kaggle/working/phase3_publish")
RUNS = WORK / "runs"

# 1. Remove only the incomplete recovery archive from the failed cell.
partial_zip = Path("/kaggle/working/phase3_recovery.zip")
if partial_zip.exists():
    partial_zip.unlink()
    print("Removed incomplete recovery ZIP.")

def _prepare_recovery_upload(reason="compact_persistent_backup"):
    # Remove stale staging. Hardlinks do not delete the original files.
    if PUBLISH_DIR.exists():
        shutil.rmtree(PUBLISH_DIR)
    PUBLISH_DIR.mkdir(parents=True)

    staged = []
    skipped = []

    def stage(src, relative, required=True):
        src = Path(src)
        if not src.is_file():
            return False
        dst = PUBLISH_DIR / relative
        dst.parent.mkdir(parents=True, exist_ok=True)

        # Prefer a hardlink: same-filesystem links consume virtually no
        # additional data blocks and preserve the source checkpoint.
        try:
            os.link(src, dst)
            staged.append(str(relative))
            return True
        except OSError:
            pass

        # Copy only when there is sufficient headroom.
        size = src.stat().st_size
        free = shutil.disk_usage("/kaggle/working").free
        reserve = 512 * 1024**2
        if free < size + reserve:
            skipped.append((str(relative), size))
            if required:
                raise OSError(
                    f"Cannot safely stage required file {src}: "
                    f"needs {size/2**20:.0f} MiB; "
                    f"free space is {free/2**20:.0f} MiB. "
                    "Original file remains untouched."
                )
            return False

        shutil.copy2(src, dst)
        staged.append(str(relative))
        return True

    # Preserve completion records and small outputs for all runs.
    teacher_runs = {
        "E0b_full_ctx_CL", "E0b_full_ctx_IT",
        "E2_lorma_plus_CL", "E2_lorma_plus_IT",
    }
    best_name = str(globals().get("BEST", ""))

    for run_dir in sorted(RUNS.iterdir()):
        if not run_dir.is_dir():
            continue

        name = run_dir.name
        complete = (run_dir / "result.json").is_file()

        if complete:
            stage(run_dir / "result.json",
                  Path("runs") / name / "result.json")

            # Preserve validation/test logits for later ensemble comparison.
            for split in ("dev", "test"):
                f = run_dir / f"{split}_logits.npy"
                if f.is_file():
                    stage(f, Path("runs") / name / f.name)

            # Preserve teacher inputs needed for future KD recovery.
            if name in teacher_runs:
                f = run_dir / "train_logits.npy"
                if f.is_file():
                    stage(f, Path("runs") / name / f.name)

            # Keep the selected best model checkpoint for future regeneration.
            if best_name and name in {
                f"{best_name}_CL", f"{best_name}_IT"
            }:
                f = run_dir / "best.pt"
                if f.is_file():
                    stage(f, Path("runs") / name / f.name)
        else:
            # Only unfinished runs need resumable checkpoints.
            for filename in ("latest.pt", "best.pt"):
                f = run_dir / filename
                if f.is_file():
                    stage(f, Path("runs") / name / filename)

    # Completion ledger and current prediction artifacts.
    for filename in (
        "ledger.csv", "prediction.json",
        "prediction_CL.json", "prediction_IT.json"
    ):
        f = WORK / filename
        if f.is_file():
            stage(f, Path(filename))

    # Preserve the already-generated submission ZIP, if it exists.
    submission = Path("/kaggle/working/submission.zip")
    if submission.is_file():
        stage(submission, Path("submission.zip"))

    manifest = {
        "project": "CS781 Phase 3",
        "reason": reason,
        "created_unix": time.time(),
        "completed_runs_policy": (
            "result.json and available dev/test logits; teacher train logits; "
            "selected best checkpoints"
        ),
        "unfinished_runs_policy": "latest.pt and best.pt when present",
        "staged_file_count": len(staged),
        "staged_files": staged,
        "skipped_optional_files": skipped,
    }
    (PUBLISH_DIR / "publish_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    metadata = {
        "title": "CS781 Phase 3 Compact Recovery",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": True,
    }
    (PUBLISH_DIR / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )

    print(f"Prepared {len(staged)} files; no recovery ZIP created.")
    print(f"Free disk: {shutil.disk_usage('/kaggle/working').free/2**30:.2f} GiB")
    if skipped:
        print("Optional files skipped due to low disk:", skipped)

# 2. Publish the compact recovery snapshot using the existing Kaggle publisher.
if not globals().get("DURABLE_PUBLISH_ENABLED", False):
    raise RuntimeError(
        "Durable publishing is not enabled. Check Kaggle credentials/secrets "
        "before proceeding; the original files have not been deleted."
    )

_prepare_recovery_upload(reason="pre_ensemble_persistent_backup")

cli = shutil.which("kaggle")
if not cli:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kaggle"],
        check=True
    )
    cli = shutil.which("kaggle")

env = os.environ.copy()
cmd = [
    cli, "datasets", "version",
    "-p", str(PUBLISH_DIR),
    "-m", "CS781 compact persistent recovery before post-hoc improvements",
    "--dir-mode", "zip",
]
result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout[-4000:])
if result.stderr:
    print(result.stderr[-2000:])

combined_output = (result.stdout + "\n" + result.stderr).lower()
confirmed = (
    result.returncode == 0
    and (
        "dataset version is being created" in combined_output
        or "upload successful" in combined_output
    )
)
if not confirmed:
    raise RuntimeError(
        "Kaggle has not confirmed the recovery upload. "
        "Original checkpoints remain untouched. Check the output above "
        "and the Dataset page before retrying."
    )

print("\nUpload submitted. Verify the new Dataset version on Kaggle.")
print("Keep the local checkpoints until the remote version is confirmed.")


In [ ]:

# CS781 — Validation-only ensemble + smoothing sweep
# No training. No test labels or test scores used for model selection.

from pathlib import Path
import json
import numpy as np
import pandas as pd

RUNS_DIR = Path(WORK) / "runs"
SMOOTH_VALUES = [0.0, 0.25, 0.5, 1.0]

def _softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.maximum(ex.sum(axis=1, keepdims=True), 1e-12)

def _validation_candidates(dom):
    candidates = {}
    for rd in sorted(RUNS_DIR.glob(f"*_{dom}")):
        lp = rd / "dev_logits.npy"
        if not lp.is_file():
            continue
        try:
            z = np.load(lp, mmap_mode="r")
            if z.ndim != 2 or z.shape[1] != K:
                print(f"Skipping {rd.name}: unexpected logits shape {z.shape}")
                continue
            candidates[rd.name[:-len("_" + dom)]] = np.asarray(z, dtype=np.float32)
        except Exception as e:
            print(f"Skipping {rd.name}: {e}")
    return candidates

def _score_probs(probs, exs, logT, logS, y):
    rows = []
    for lam in SMOOTH_VALUES:
        pred = smooth_split(probs, exs, logT, logS, lam)
        rows.append({
            "smooth": lam,
            "macro_f1": prf_report(y, pred, K)["macro_f1"],
            "accuracy": prf_report(y, pred, K)["acc"],
        })
    return rows

ensemble_results = []
best_settings = {}

for dom in ("CL", "IT"):
    print(f"\n{'='*18} {dom} {'='*18}")
    candidates = _validation_candidates(dom)

    if not candidates:
        print("No saved dev_logits.npy files found for this domain.")
        print("Attach the recovery Dataset and restore its run artifacts first.")
        continue

    # Use the same sentence order as the validation examples.
    # Context changes the input representation, not sentence/label ordering.
    exs = build_examples(DOCS[dom]["dev"], True)
    y = np.asarray([e["y"] for e in exs], dtype=int)
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K)

    usable = {}
    for name, logits in candidates.items():
        if len(logits) == len(y):
            usable[name] = _softmax_np(logits)
        else:
            print(f"Skipping {name}: {len(logits)} logits vs {len(y)} validation labels")

    if not usable:
        print("No candidate logits align with validation labels.")
        continue

    print("Candidates:", list(usable))
    combos = [(name,) for name in usable]

    # Pairwise probability averaging; no test data involved.
    names = list(usable)
    combos += [
        (names[i], names[j])
        for i in range(len(names))
        for j in range(i + 1, len(names))
    ]

    rows = []
    for combo in combos:
        probs = np.mean([usable[n] for n in combo], axis=0)
        for result in _score_probs(probs, exs, logT, logS, y):
            rows.append({
                "domain": dom,
                "models": " + ".join(combo),
                "n_models": len(combo),
                **result,
            })

    result_df = pd.DataFrame(rows).sort_values(
        ["macro_f1", "accuracy"], ascending=False
    ).reset_index(drop=True)

    ensemble_results.append(result_df)
    print("\nTop validation candidates:")
    print(result_df.head(12).round(4).to_string(index=False))

    best = result_df.iloc[0]
    best_settings[dom] = {
        "models": best["models"].split(" + "),
        "smooth": float(best["smooth"]),
        "dev_macro_f1": float(best["macro_f1"]),
    }
    print("\nBest setting for", dom, ":", best_settings[dom])

if ensemble_results:
    ensemble_table = pd.concat(ensemble_results, ignore_index=True)
    out = Path(WORK) / "validation_ensemble_sweep.csv"
    ensemble_table.to_csv(out, index=False)
    print("\nFull validation sweep saved to:", out)
    print("\nBest settings (validation only):")
    print(json.dumps(best_settings, indent=2))


In [ ]:

# CS781 — Per-class validation diagnostics for selected configurations
# Validation only; no training and no test labels used.

import numpy as np
import pandas as pd

SELECTED = {
    "CL": {"models": ["E0a_full_sent", "E6_kd_student"], "smooth": 0.5},
    "IT": {"models": ["E0a_full_sent"], "smooth": 1.0},
}

class_tables = {}

for dom, cfg in SELECTED.items():
    ex_dv = build_examples(DOCS[dom]["dev"], True)
    y_true = np.asarray([e["y"] for e in ex_dv], dtype=int)

    probs = ens_probs(cfg["models"], dom, "dev")
    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K
    )
    y_pred = predict_labels(
        probs, ex_dv, logT, logS, cfg["smooth"]
    )

    report = prf_report(y_true, y_pred, K)
    table = pd.DataFrame({
        "label": LABELS,
        "support": report["support"],
        "precision": report["prec"],
        "recall": report["rec"],
        "f1": report["f1"],
    }).sort_values(["f1", "support"], ascending=[True, False])

    class_tables[dom] = table
    print(f"\n{'='*15} {dom}: per-class validation results {'='*15}")
    print(table.round(3).to_string(index=False))
    print(
        f"\nMacro-F1: {report['macro_f1']:.4f}"
        f" | Accuracy: {report['acc']:.4f}"
    )

# Save diagnostics for later comparison.
out = WORK / "validation_per_class.csv"
pd.concat(
    [t.assign(domain=dom) for dom, t in class_tables.items()],
    ignore_index=True
).to_csv(out, index=False)

print("\nSaved:", out)


In [ ]:

# Diagnose the CL validation metric mismatch.
# Expected: both paths should produce the same macro-F1 if equivalent.

import numpy as np

dom = "CL"
cfg = {"models": ["E0a_full_sent", "E6_kd_student"], "smooth": 0.5}

exs = build_examples(DOCS[dom]["dev"], True)
y = np.asarray([e["y"] for e in exs], dtype=int)
logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K)

# Path A: the same candidate-loading and averaging approach as the sweep.
raw = _validation_candidates(dom)
missing = [n for n in cfg["models"] if n not in raw]
assert not missing, f"Missing saved logits: {missing}"

pA = np.mean(
    [_softmax_np(raw[n]) for n in cfg["models"]], axis=0
)
predA = smooth_split(pA, exs, logT, logS, cfg["smooth"])
repA = prf_report(y, predA, K)

# Path B: the helper used by the per-class diagnostic.
pB = ens_probs(cfg["models"], dom, "dev")
predB = predict_labels(pB, exs, logT, logS, cfg["smooth"])
repB = prf_report(y, predB, K)

print("Validation examples:", len(y))
print("Label count K:", K)
print("Path A probability shape:", pA.shape)
print("Path B output shape:", np.asarray(pB).shape)
print("Path A probability row sums:",
      np.round(pA.sum(axis=1)[:5], 4))
print("Path B row sums:",
      np.round(np.asarray(pB).sum(axis=1)[:5], 4))

print("\nPath A (sweep-style):")
print("Macro-F1:", repA["macro_f1"], "Accuracy:", repA["acc"])

print("\nPath B (diagnostic-style):")
print("Macro-F1:", repB["macro_f1"], "Accuracy:", repB["acc"])

print("\nPredictions identical:", np.array_equal(predA, predB))
print("Labels identical to example labels:",
      np.array_equal(y, np.asarray([e["y"] for e in exs])))

if not np.array_equal(predA, predB):
    print("Different predictions:", int(np.sum(predA != predB)))
    print("First differing indices:", np.flatnonzero(predA != predB)[:20])


In [ ]:

# CS781 — Isolate the CL prediction-path mismatch
import numpy as np

print("1) Probability matrix comparison")
print("Same probabilities:", np.allclose(pA, pB, atol=1e-6))
print("Max absolute difference:", float(np.max(np.abs(pA - pB))))
print("Mean absolute difference:", float(np.mean(np.abs(pA - pB))))

# Check predictions from the same probabilities through both helpers.
pred_A_A = smooth_split(pA, exs, logT, logS, 0.5)
pred_B_A = predict_labels(pA, exs, logT, logS, 0.5)

print("\n2) Same probabilities, different prediction helpers")
print("Predictions identical:", np.array_equal(pred_A_A, pred_B_A))
print("Different labels:", int(np.sum(pred_A_A != pred_B_A)))

# Check whether the mismatch is already present before smoothing.
pred_raw_A = np.argmax(pA, axis=1)
pred_raw_B = np.argmax(pB, axis=1)

print("\n3) Raw argmax comparison")
print("Raw argmax predictions identical:",
      np.array_equal(pred_raw_A, pred_raw_B))
print("Raw argmax differences:",
      int(np.sum(pred_raw_A != pred_raw_B)))

# Display a few examples where the final paths disagree.
diff = np.flatnonzero(predA != predB)[:10]
print("\n4) First disagreements")
for i in diff:
    print({
        "index": int(i),
        "gold_id": int(y[i]),
        "path_A_id": int(predA[i]),
        "path_B_id": int(predB[i]),
        "path_A_top3": np.argsort(pA[i])[-3:][::-1].tolist(),
        "path_B_top3": np.argsort(pB[i])[-3:][::-1].tolist(),
    })


In [ ]:

import inspect
import numpy as np

print("=== ens_probs implementation ===")
print(inspect.getsource(ens_probs))

print("\n=== _softmax_np implementation ===")
print(inspect.getsource(_softmax_np))

print("\n=== Raw logits summary ===")
for name in ["E0a_full_sent", "E6_kd_student"]:
    z = np.asarray(raw[name])
    print(
        name,
        "shape =", z.shape,
        "mean =", round(float(z.mean()), 5),
        "std =", round(float(z.std()), 5),
        "min =", round(float(z.min()), 5),
        "max =", round(float(z.max()), 5),
    )

print("\n=== Compare averaging probabilities vs logits ===")
z1 = np.asarray(raw["E0a_full_sent"], dtype=np.float64)
z2 = np.asarray(raw["E6_kd_student"], dtype=np.float64)

p_avg = (_softmax_np(z1) + _softmax_np(z2)) / 2
p_logit_avg = _softmax_np((z1 + z2) / 2)

print("Mean probability-avg vs logit-avg abs diff:",
      float(np.mean(np.abs(p_avg - p_logit_avg))))
print("Max probability-avg vs logit-avg abs diff:",
      float(np.max(np.abs(p_avg - p_logit_avg))))
print("Canonical helper vs probability average:",
      bool(np.allclose(pB, p_avg, atol=1e-6)))
print("Canonical helper vs logit average:",
      bool(np.allclose(pB, p_logit_avg, atol=1e-6)))


In [ ]:

# CS781 — Fair ensemble-method comparison
# Validation only. No training, no test-label use.

import numpy as np
import pandas as pd

def arithmetic_ens_probs(names, dom, split):
    ps = [
        softmax_np(load_arr(n, dom, f"{split}_logits"))
        for n in names
    ]
    return np.mean(ps, axis=0)

def geometric_ens_probs(names, dom, split):
    log_ps = [
        np.log(softmax_np(load_arr(n, dom, f"{split}_logits")) + 1e-9)
        for n in names
    ]
    return softmax_np(np.mean(log_ps, axis=0))

def macro_f1_from_preds(y, pred, K):
    return prf_report(y, pred, K)["macro_f1"]

comparison_rows = []

for dom in ("CL", "IT"):
    exs = build_examples(DOCS[dom]["dev"], True)
    y = np.asarray([e["y"] for e in exs], dtype=int)
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K)

    candidate_names = list(_validation_candidates(dom))
    model_sets = [(n,) for n in candidate_names]

    if len(candidate_names) >= 2:
        model_sets += [
            (candidate_names[i], candidate_names[j])
            for i in range(len(candidate_names))
            for j in range(i + 1, len(candidate_names))
        ]

    for names in model_sets:
        for method, make_probs in [
            ("arithmetic", arithmetic_ens_probs),
            ("geometric", geometric_ens_probs),
        ]:
            probs = make_probs(names, dom, "dev")
            assert len(probs) == len(y), (dom, names, probs.shape, y.shape)

            for smooth in [0.0, 0.25, 0.5, 1.0]:
                pred = predict_labels(
                    probs, exs, logT, logS, smooth
                )
                report = prf_report(y, pred, K)
                comparison_rows.append({
                    "domain": dom,
                    "models": " + ".join(names),
                    "method": method,
                    "smooth": smooth,
                    "macro_f1": report["macro_f1"],
                    "accuracy": report["acc"],
                })

comparison = pd.DataFrame(comparison_rows)

for dom in ("CL", "IT"):
    print(f"\n{'='*16} {dom} {'='*16}")
    part = comparison[comparison.domain == dom].sort_values(
        ["macro_f1", "accuracy"], ascending=False
    )
    print(part.head(12).round(4).to_string(index=False))

comparison.to_csv(
    WORK / "validation_ensemble_methods.csv", index=False
)
print("\nSaved:", WORK / "validation_ensemble_methods.csv")


In [ ]:

# CS781 — Per-class report for the selected ensemble configurations
# Validation only. Does not train or modify checkpoints.

import numpy as np
import pandas as pd

FINAL_CANDIDATES = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "arithmetic",
        "smooth": 0.50,
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "geometric",
        "smooth": 0.25,
    },
}

reports = []

for dom, cfg in FINAL_CANDIDATES.items():
    exs = build_examples(DOCS[dom]["dev"], True)
    y = np.asarray([e["y"] for e in exs], dtype=int)

    if cfg["method"] == "arithmetic":
        probs = arithmetic_ens_probs(cfg["models"], dom, "dev")
    else:
        probs = geometric_ens_probs(cfg["models"], dom, "dev")

    assert probs.shape == (len(y), K), (
        dom, probs.shape, len(y), K
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K
    )
    pred = predict_labels(
        probs, exs, logT, logS, cfg["smooth"]
    )
    rep = prf_report(y, pred, K)

    table = pd.DataFrame({
        "label": LABELS,
        "support": rep["support"],
        "precision": rep["prec"],
        "recall": rep["rec"],
        "f1": rep["f1"],
    }).sort_values(["f1", "support"], ascending=[True, False])

    print(f"\n{'='*16} {dom} {'='*16}")
    print("Models:", cfg["models"])
    print("Method:", cfg["method"], "| smooth:", cfg["smooth"])
    print(f"Macro-F1: {rep['macro_f1']:.4f} | Accuracy: {rep['acc']:.4f}")
    print(table.round(3).to_string(index=False))

    table["domain"] = dom
    reports.append(table)

out = WORK / "selected_ensemble_per_class.csv"
pd.concat(reports, ignore_index=True).to_csv(out, index=False)
print("\nSaved:", out)


In [ ]:

# CS781 — Small class-bias sweep on validation data only
# No training, checkpoint changes, or test-label use.

import numpy as np
import pandas as pd
from itertools import product

BIAS_VALUES = [-0.50, -0.25, 0.0, 0.25, 0.50, 0.75, 1.0]

CONFIGS = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "arithmetic",
        "smooth": 0.50,
        "targets": [
            "ArgumentRespondent",
            "RulingByLowerCourt",
            "PrecedentReliedUpon",
        ],
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "geometric",
        "smooth": 0.25,
        "targets": [
            "ArgumentRespondent",
            "RulingByLowerCourt",
            "PrecedentNotReliedUpon",
        ],
    },
}

def apply_class_bias(probs, bias_by_label):
    # Add biases in log-probability space, then renormalize.
    z = np.log(np.asarray(probs, dtype=np.float64) + 1e-12)
    bias = np.zeros(K, dtype=np.float64)

    for label, amount in bias_by_label.items():
        bias[LABEL2ID[label]] = amount

    z += bias[None, :]
    z -= z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

bias_results = []
bias_best = {}

for dom, cfg in CONFIGS.items():
    exs = build_examples(DOCS[dom]["dev"], True)
    y = np.asarray([e["y"] for e in exs], dtype=int)
    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K
    )

    if cfg["method"] == "arithmetic":
        base_probs = arithmetic_ens_probs(cfg["models"], dom, "dev")
    else:
        base_probs = geometric_ens_probs(cfg["models"], dom, "dev")

    # Baseline must reproduce the selected ensemble score.
    baseline_pred = predict_labels(
        base_probs, exs, logT, logS, cfg["smooth"]
    )
    baseline_report = prf_report(y, baseline_pred, K)
    baseline_f1 = float(baseline_report["macro_f1"])

    print(f"\n{'='*14} {dom} {'='*14}")
    print(f"Baseline macro-F1: {baseline_f1:.4f}")

    rows = []

    # One-at-a-time changes first; this limits overfitting.
    for label in cfg["targets"]:
        for amount in BIAS_VALUES:
            bias_map = {} if amount == 0 else {label: amount}
            adjusted = apply_class_bias(base_probs, bias_map)
            pred = predict_labels(
                adjusted, exs, logT, logS, cfg["smooth"]
            )
            report = prf_report(y, pred, K)

            rows.append({
                "domain": dom,
                "bias_label": label if amount != 0 else "(baseline)",
                "bias": amount,
                "macro_f1": float(report["macro_f1"]),
                "accuracy": float(report["acc"]),
                "target_f1": (
                    float(report["f1"][LABEL2ID[label]])
                    if amount != 0 else np.nan
                ),
            })

    df = pd.DataFrame(rows)
    bias_results.append(df)

    print("\nBest one-at-a-time settings:")
    print(
        df.sort_values(["macro_f1", "accuracy"], ascending=False)
          .head(10).round(4).to_string(index=False)
    )

    winner = df.sort_values(
        ["macro_f1", "accuracy"], ascending=False
    ).iloc[0]

    if winner["macro_f1"] > baseline_f1:
        bias_best[dom] = {
            "label": winner["bias_label"],
            "bias": float(winner["bias"]),
            "macro_f1": float(winner["macro_f1"]),
            "baseline_macro_f1": baseline_f1,
        }
    else:
        bias_best[dom] = {
            "label": None,
            "bias": 0.0,
            "macro_f1": baseline_f1,
            "baseline_macro_f1": baseline_f1,
        }

out = WORK / "validation_class_bias_sweep.csv"
pd.concat(bias_results, ignore_index=True).to_csv(out, index=False)

print("\nBest candidate per domain (validation only):")
print(bias_best)
print("Saved:", out)


In [ ]:

# CS781 — Baseline vs selected class-bias configurations
# Validation only. No training or test-label use.

import numpy as np
import pandas as pd

CHECKS = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "arithmetic",
        "smooth": 0.50,
        "target": "ArgumentRespondent",
        "bias": 1.0,
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "method": "geometric",
        "smooth": 0.25,
        "target": "RulingByLowerCourt",
        "bias": 1.0,
    },
}

for dom, cfg in CHECKS.items():
    exs = build_examples(DOCS[dom]["dev"], True)
    y = np.asarray([e["y"] for e in exs], dtype=int)
    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K
    )

    fn = (
        arithmetic_ens_probs
        if cfg["method"] == "arithmetic"
        else geometric_ens_probs
    )
    probs = fn(cfg["models"], dom, "dev")

    pred_base = predict_labels(
        probs, exs, logT, logS, cfg["smooth"]
    )
    pred_adj = predict_labels(
        apply_class_bias(probs, {cfg["target"]: cfg["bias"]}),
        exs, logT, logS, cfg["smooth"]
    )

    r0 = prf_report(y, pred_base, K)
    r1 = prf_report(y, pred_adj, K)
    k = LABEL2ID[cfg["target"]]

    print(f"\n===== {dom} =====")
    print(f"Baseline macro-F1: {r0['macro_f1']:.4f}")
    print(f"Adjusted macro-F1: {r1['macro_f1']:.4f}")
    print(f"Baseline accuracy: {r0['acc']:.4f}")
    print(f"Adjusted accuracy: {r1['acc']:.4f}")
    print("Changed predictions:", int(np.sum(pred_base != pred_adj)))
    print(f"\nTarget class: {cfg['target']}")
    print(
        f"Baseline precision/recall/F1: "
        f"{r0['prec'][k]:.3f} / {r0['rec'][k]:.3f} / {r0['f1'][k]:.3f}"
    )
    print(
        f"Adjusted precision/recall/F1: "
        f"{r1['prec'][k]:.3f} / {r1['rec'][k]:.3f} / {r1['f1'][k]:.3f}"
    )


In [ ]:

# Final inference configuration — selected using validation only

FINAL_CONFIG = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "arithmetic",
        "smooth": 0.50,
        "class_bias": {"ArgumentRespondent": 1.0},
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "geometric",
        "smooth": 0.25,
        "class_bias": {"RulingByLowerCourt": 1.0},
    },
}

def final_ensemble_probs(names, dom, split, method):
    if method == "arithmetic":
        return arithmetic_ens_probs(names, dom, split)
    elif method == "geometric":
        return geometric_ens_probs(names, dom, split)
    raise ValueError(f"Unknown ensemble method: {method}")

def final_predict(probs, examples, logT, logS, smooth, class_bias):
    adjusted_probs = apply_class_bias(probs, class_bias)
    return predict_labels(
        adjusted_probs, examples, logT, logS, smooth
    )

print("Final configurations ready.")
print(FINAL_CONFIG)


In [ ]:

# Preflight checks before final inference
from pathlib import Path

required_names = [
    "WORK", "DOCS", "LABEL2ID", "LABELS", "K",
    "build_examples", "transition_logprobs",
    "final_ensemble_probs", "final_predict",
    "prf_report", "load_arr",
]

missing = [name for name in required_names if name not in globals()]
print("Missing notebook definitions:", missing)

assert not missing, (
    "Run the notebook setup/data-loading cells first. "
    f"Missing: {missing}"
)

for dom, cfg in FINAL_CONFIG.items():
    print(f"\n{dom}:")
    for model in cfg["models"]:
        path = Path(WORK) / "runs" / f"{model}_{dom}"
        print(
            f"  {model}:",
            "directory =", path.exists(),
            "test logits =", (path / "test_logits.npy").exists(),
        )

print("\nPreflight complete. No files were modified.")


In [ ]:

# Inspect the existing prediction JSON schemas (read-only)
import json
from pathlib import Path

for p in [
    Path(WORK) / "prediction.json",
    Path(WORK) / "prediction_CL.json",
    Path(WORK) / "prediction_IT.json",
]:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print(f"\n{p.name}: type={type(obj).__name__}")
        if isinstance(obj, dict):
            print("keys:", list(obj.keys())[:20])
            for k, v in list(obj.items())[:2]:
                print("sample:", repr(k), repr(v)[:500])
        elif isinstance(obj, list):
            print("length:", len(obj))
            print("first item:", repr(obj[:1])[:1000])
        else:
            print("sample:", repr(obj)[:500])
    else:
        print(f"\nNot found: {p}")


In [ ]:

# CS781 — Final improved prediction generation and submission packaging
# Uses saved test logits only; no training.
# Keeps existing submission and recovery backup untouched.

import json
import zipfile
from pathlib import Path
import numpy as np

OUT_DIR = Path(WORK) / "final_improved"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def generate_domain_prediction(dom):
    cfg = FINAL_CONFIG[dom]
    examples = build_examples(DOCS[dom]["test"], False)

    probs = final_ensemble_probs(
        cfg["models"], dom, "test", cfg["ensemble"]
    )

    assert probs.shape == (len(examples), K), (
        f"{dom}: expected {(len(examples), K)}, got {probs.shape}"
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K
    )
    pred_ids = final_predict(
        probs, examples, logT, logS,
        cfg["smooth"], cfg["class_bias"]
    )

    # Group sentence predictions by their original document IDs.
    # Determine the ID field from the notebook's existing examples.
    if not examples:
        raise ValueError(f"No test examples for {dom}")

    possible_id_keys = [
        "doc_id", "document_id", "doc", "document", "id", "case_id"
    ]
    first = examples[0]
    id_key = next((k for k in possible_id_keys if k in first), None)

    if id_key is None:
        raise KeyError(
            f"Cannot identify document ID field in example keys: "
            f"{list(first.keys())}. Stop rather than guess."
        )

    output = {}
    for ex, pid in zip(examples, pred_ids):
        doc_id = str(ex[id_key])
        label = LABELS[int(pid)]
        output.setdefault(doc_id, []).append(label)

    return output

pred_cl = generate_domain_prediction("CL")
pred_it = generate_domain_prediction("IT")

# Preserve the original document order when available.
def order_like_existing(new_data, old_path):
    if not old_path.exists():
        return new_data
    with open(old_path, "r", encoding="utf-8") as f:
        old = json.load(f)
    ordered = {}
    for k in old:
        if k in new_data:
            ordered[k] = new_data[k]
    for k, v in new_data.items():
        if k not in ordered:
            ordered[k] = v
    return ordered

old_cl = Path(WORK) / "prediction_CL.json"
old_it = Path(WORK) / "prediction_IT.json"

pred_cl = order_like_existing(pred_cl, old_cl)
pred_it = order_like_existing(pred_it, old_it)

combined = dict(pred_cl)
overlap = set(combined) & set(pred_it)
assert not overlap, f"Document IDs appear in both CL and IT: {list(overlap)[:5]}"
combined.update(pred_it)

# Verify the expected document counts against the previous files.
def validate_against_old(new_data, old_path, domain):
    if not old_path.exists():
        print(f"{domain}: no previous file to compare; basic checks only")
        return
    with open(old_path, "r", encoding="utf-8") as f:
        old = json.load(f)

    assert set(new_data) == set(old), (
        f"{domain}: document IDs differ; "
        f"missing={set(old)-set(new_data)}, extra={set(new_data)-set(old)}"
    )
    mismatches = [
        (doc, len(old[doc]), len(new_data[doc]))
        for doc in old if len(old[doc]) != len(new_data[doc])
    ]
    assert not mismatches, (
        f"{domain}: sentence counts changed: {mismatches[:5]}"
    )

    print(f"{domain}: document IDs and sentence counts match previous output")

validate_against_old(pred_cl, old_cl, "CL")
validate_against_old(pred_it, old_it, "IT")

for name, data in [
    ("prediction_CL.json", pred_cl),
    ("prediction_IT.json", pred_it),
    ("prediction.json", combined),
]:
    path = OUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"Wrote {path.name}: {len(data)} documents, "
          f"{sum(map(len, data.values()))} sentences")

# Basic label and structure checks.
allowed = set(LABELS)
for filename, data in [
    ("prediction_CL.json", pred_cl),
    ("prediction_IT.json", pred_it),
    ("prediction.json", combined),
]:
    assert isinstance(data, dict)
    assert all(isinstance(k, str) and isinstance(v, list)
               for k, v in data.items())
    assert all(label in allowed for v in data.values() for label in v)
    with open(OUT_DIR / filename, "r", encoding="utf-8") as f:
        assert json.load(f) == data

zip_path = OUT_DIR / "submission_improved.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_DIR / "prediction.json", arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as z:
    assert z.testzip() is None
    assert "prediction.json" in z.namelist()
    with z.open("prediction.json") as f:
        zipped = json.load(f)
    assert zipped == combined

print("\nSUCCESS")
print("Improved submission:", zip_path)
print("CL documents:", len(pred_cl))
print("IT documents:", len(pred_it))
print("Total documents:", len(combined))
print("ZIP integrity: OK")


In [ ]:
from pathlib import Path
import inspect
import numpy as np

assert "WORK" in globals(), "WORK is missing; run your setup cells first."
assert "DOCS" in globals(), "DOCS is missing; run your dataset-loading cells first."
assert "LABELS" in globals(), "LABELS is missing; run your label setup cells first."
assert "FINAL_CONFIG" in globals(), "FINAL_CONFIG is missing; run your configuration cell first."

print("WORK:", WORK)
print("Labels:", LABELS)
print("Available document splits:")
for dom in ["CL", "IT"]:
    print(dom, list(DOCS[dom].keys()))

for fn_name in [
    "load_arr", "build_examples", "final_ensemble_probs",
    "final_predict", "prf_report", "transition_logprobs",
    "predict_labels"
]:
    if fn_name in globals():
        fn = globals()[fn_name]
        print(f"\n{fn_name}{inspect.signature(fn)}")

print("\nChecking saved validation logits:")
for dom in ["CL", "IT"]:
    for model in ["E0a_full_sent", "E6_kd_student"]:
        folder = Path(WORK) / "runs" / f"{model}_{dom}"
        matches = list(folder.glob("*val*logits*.npy"))
        matches += list(folder.glob("*valid*logits*.npy"))
        matches += list(folder.glob("*validation*logits*.npy"))
        print(f"{dom:2s} | {model:18s} |", [str(p) for p in matches])

In [ ]:
from pathlib import Path

for dom in ["CL", "IT"]:
    for model in ["E0a_full_sent", "E6_kd_student", "E6_student_nokd"]:
        folder = Path(WORK) / "runs" / f"{model}_{dom}"
        if folder.exists():
            print(f"\n{dom} / {model}")
            for p in sorted(folder.glob("*.npy")):
                print(" ", p.name, np.load(p, mmap_mode="r").shape)

In [ ]:
# Validation-only inspection. Does not access test labels.

for dom in ["CL", "IT"]:
    print("\n" + "=" * 70)
    print("DOMAIN:", dom)

    dev_examples = build_examples(DOCS[dom]["dev"], False)

    print("Number of validation examples:", len(dev_examples))
    print("Example type:", type(dev_examples[0]).__name__ if dev_examples else "EMPTY")

    if dev_examples:
        ex = dev_examples[0]
        if isinstance(ex, dict):
            print("Example keys:", list(ex.keys()))
            for key, value in ex.items():
                if key.lower() in {
                    "label", "labels", "y", "gold", "target",
                    "label_id", "label_ids", "doc_id",
                    "document_id", "doc", "sentence_id"
                }:
                    print(f"  {key}: {repr(value)[:250]}")
        else:
            print("First example:", repr(ex)[:1000])

    # Inspect the actual validation logits.
    for model in ["E0a_full_sent", "E6_kd_student"]:
        path = Path(WORK) / "runs" / f"{model}_{dom}" / "dev_logits.npy"
        arr = np.load(path)
        print(f"{model}: shape={arr.shape}, finite={np.isfinite(arr).all()}")

    # Inspect the source document structure.
    dev_docs = DOCS[dom]["dev"]
    print("Validation document container:", type(dev_docs).__name__)
    if isinstance(dev_docs, dict):
        print("Document IDs (sample):", list(dev_docs.keys())[:3])
        first_doc = next(iter(dev_docs.values()), None)
    else:
        print("Number of documents:", len(dev_docs))
        first_doc = dev_docs[0] if len(dev_docs) else None

    if first_doc is not None:
        print("First document type:", type(first_doc).__name__)
        if isinstance(first_doc, dict):
            print("First document keys:", list(first_doc.keys()))
            for key, value in first_doc.items():
                if isinstance(value, (list, tuple)):
                    print(f"  {key}: {len(value)} items; sample={repr(value[:2])[:250]}")

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

EXP_DIR = Path(WORK) / "experiments" / "weighted_ensemble_v1"
EXP_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["E0a_full_sent", "E6_kd_student"]

# Weight assigned to E0a_full_sent.
# The remaining weight (1 - w) goes to E6_kd_student.
WEIGHTS = [0.0, 0.25, 0.40, 0.50, 0.60, 0.75, 1.0]

# Use the existing candidate configuration as the fixed baseline.
assert "FINAL_CONFIG" in globals()

for dom in ["CL", "IT"]:
    for model in MODELS:
        path = Path(WORK) / "runs" / f"{model}_{dom}" / "dev_logits.npy"
        assert path.exists(), f"Missing validation logits: {path}"

    assert len(DOCS[dom]["dev"]) > 0

print("Experiment folder:", EXP_DIR)
print("Split: dev only")
print("Models:", MODELS)
print("Weights:", WEIGHTS)
print("Domains: CL, IT")

In [ ]:
def weighted_probs(logits_a, logits_b, weight_a, method="arithmetic"):
    """Combine two models' logits as weighted probabilities."""
    assert logits_a.shape == logits_b.shape
    assert 0.0 <= weight_a <= 1.0

    def softmax_stable(x):
        x = np.asarray(x, dtype=np.float64)
        x = x - np.max(x, axis=-1, keepdims=True)
        ex = np.exp(x)
        return ex / np.maximum(ex.sum(axis=-1, keepdims=True), 1e-300)

    pa = softmax_stable(logits_a)
    pb = softmax_stable(logits_b)
    weight_b = 1.0 - weight_a

    if method == "arithmetic":
        p = weight_a * pa + weight_b * pb
    elif method == "geometric":
        logp = (
            weight_a * np.log(np.clip(pa, 1e-12, 1.0))
            + weight_b * np.log(np.clip(pb, 1e-12, 1.0))
        )
        logp -= np.max(logp, axis=-1, keepdims=True)
        p = np.exp(logp)
    else:
        raise ValueError(f"Unknown ensemble method: {method}")

    p /= np.maximum(p.sum(axis=-1, keepdims=True), 1e-300)
    return p


results = []
saved_predictions = {}

for dom in ["CL", "IT"]:
    cfg = FINAL_CONFIG[dom]
    examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.asarray([int(ex["y"]) for ex in examples], dtype=np.int64)

    log_a = np.load(
        Path(WORK) / "runs" / f"{MODELS[0]}_{dom}" / "dev_logits.npy"
    )
    log_b = np.load(
        Path(WORK) / "runs" / f"{MODELS[1]}_{dom}" / "dev_logits.npy"
    )

    assert len(examples) == len(y_true) == len(log_a) == len(log_b)
    assert log_a.shape[1] == log_b.shape[1] == len(LABELS)
    assert np.isfinite(log_a).all() and np.isfinite(log_b).all()

    # FIX: transition_logprobs returns (logT, logS).
    logT, _ = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS)
    )
    _, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS)
    )

    smooth = float(cfg["smooth"])
    class_bias = cfg.get("class_bias", {})

    for method in ["arithmetic", "geometric"]:
        for w in WEIGHTS:
            probs = weighted_probs(log_a, log_b, w, method=method)

            y_pred = np.asarray(
                final_predict(
                    probs, examples, logT, logS,
                    smooth=smooth,
                    class_bias=class_bias,
                ),
                dtype=np.int64,
            )

            assert len(y_pred) == len(y_true)

            macro_f1 = f1_score(
                y_true,
                y_pred,
                labels=np.arange(len(LABELS)),
                average="macro",
                zero_division=0,
            )
            accuracy = accuracy_score(y_true, y_pred)

            saved_predictions[(dom, method, w)] = y_pred

            results.append({
                "domain": dom,
                "method": method,
                "weight_E0a": w,
                "weight_E6_kd": 1.0 - w,
                "smooth": smooth,
                "class_bias": json.dumps(class_bias, sort_keys=True),
                "macro_f1": float(macro_f1),
                "accuracy": float(accuracy),
            })

results_df = pd.DataFrame(results).sort_values(
    ["domain", "macro_f1"], ascending=[True, False]
).reset_index(drop=True)

csv_path = EXP_DIR / "weighted_ensemble_dev_results.csv"
results_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
display(results_df.round(5))

In [ ]:
for dom in ["CL", "IT"]:
    subset = results_df[results_df["domain"] == dom].copy()

    equal = subset[subset["weight_E0a"] == 0.5].sort_values(
        "macro_f1", ascending=False
    ).iloc[0]

    best = subset.sort_values("macro_f1", ascending=False).iloc[0]

    print(f"\n{'=' * 60}\n{dom}")
    print(
        f"Best: {best['method']} | "
        f"E0a={best['weight_E0a']:.2f}, "
        f"E6={best['weight_E6_kd']:.2f} | "
        f"macro-F1={best['macro_f1']:.5f} | "
        f"accuracy={best['accuracy']:.5f}"
    )
    print(
        f"Equal weight, best averaging method: {equal['method']} | "
        f"macro-F1={equal['macro_f1']:.5f}"
    )
    print(f"Difference vs equal weight: {best['macro_f1'] - equal['macro_f1']:+.5f}")

In [ ]:
best_configs = {}

for dom in ["CL", "IT"]:
    subset = results_df[results_df["domain"] == dom]
    best = subset.sort_values("macro_f1", ascending=False).iloc[0]

    best_configs[dom] = {
        "models": MODELS,
        "ensemble": str(best["method"]),
        "weight_E0a": float(best["weight_E0a"]),
        "weight_E6_kd": float(best["weight_E6_kd"]),
        "smooth": float(best["smooth"]),
        "class_bias": json.loads(best["class_bias"]),
        "validation_macro_f1": float(best["macro_f1"]),
        "validation_accuracy": float(best["accuracy"]),
    }

config_path = EXP_DIR / "best_weighted_configs.json"
with open(config_path, "w") as f:
    json.dump(best_configs, f, indent=2)

print("Saved:", config_path)
print(json.dumps(best_configs, indent=2))
print("\nOriginal predictions and submission remain untouched.")

In [ ]:
# Experiment 1 diagnostic: verify score and prediction consistency.
# Uses dev data only. Does not modify any model or submission files.

from sklearn.metrics import f1_score, accuracy_score
import numpy as np

for dom in ["CL", "IT"]:
    cfg = FINAL_CONFIG[dom]
    examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.asarray([int(ex["y"]) for ex in examples])

    log_a = load_arr("E0a_full_sent", dom, "dev_logits")
    log_b = load_arr("E6_kd_student", dom, "dev_logits")

    print(f"\n{'=' * 60}\n{dom}")
    print("Rows:", len(y_true), "classes:", len(LABELS))
    print("Gold class counts:", np.bincount(y_true, minlength=len(LABELS)))

    # Direct single-model baseline.
    for name, logits in [
        ("E0a_full_sent", log_a),
        ("E6_kd_student", log_b),
    ]:
        probs = weighted_probs(logits, logits, 1.0, "arithmetic")
        pred = probs.argmax(axis=1)
        print(
            name,
            "| macro-F1:", round(f1_score(
                y_true, pred, labels=np.arange(len(LABELS)),
                average="macro", zero_division=0
            ), 5),
            "| accuracy:", round(accuracy_score(y_true, pred), 5)
        )

    # Check helper-generated equal-weight ensemble probabilities.
    helper_probs = final_ensemble_probs(
        ["E0a_full_sent", "E6_kd_student"],
        dom, "dev", "arithmetic"
    )
    manual_probs = weighted_probs(log_a, log_b, 0.5, "arithmetic")

    print("Helper/manual probability max difference:",
          float(np.max(np.abs(helper_probs - manual_probs))))

    # Confirm dev labels align with document/position ordering.
    print("First examples:", [
        (ex["doc"], ex["i"], ex["y"]) for ex in examples[:5]
    ])

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

print("LABEL2ID:", LABEL2ID)
print("Number of labels:", len(LABELS))

for dom in ["CL", "IT"]:
    print(f"\n{'=' * 65}\n{dom}")

    train_labels = {
        label for doc in DOCS[dom]["train"] for label in doc["labels"]
    }
    dev_labels = {
        label for doc in DOCS[dom]["dev"] for label in doc["labels"]
    }

    print("Labels in training:", sorted(train_labels))
    print("Labels in dev:", sorted(dev_labels))
    print("Dev labels missing from training:", sorted(dev_labels - train_labels))
    print("Training labels missing from dev:", sorted(train_labels - dev_labels))

    examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.array([int(ex["y"]) for ex in examples])

    print("\nGold counts by label:")
    for i, label in enumerate(LABELS):
        print(f"{label:28s} {int((y_true == i).sum()):5d}")

    logits = np.load(
        Path(WORK) / "runs" / f"E0a_full_sent_{dom}" / "dev_logits.npy"
    )
    pred = logits.argmax(axis=1)

    print("\nMacro-F1, all 13 labels, zero_division=0:",
          f1_score(y_true, pred, labels=np.arange(len(LABELS)),
                   average="macro", zero_division=0))
    print("Macro-F1, observed dev labels only:",
          f1_score(y_true, pred, labels=np.unique(y_true),
                   average="macro", zero_division=0))

    print("\nNotebook prf_report output:")
    print(prf_report(y_true, pred, len(LABELS)))

    print("\nClassification report:")
    print(classification_report(
        y_true, pred, labels=np.arange(len(LABELS)),
        target_names=LABELS, zero_division=0
    ))

In [ ]:
import inspect
print(inspect.getsource(prf_report))

In [ ]:
# Compare both scoring conventions for all completed weighted experiments.
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

comparison = []

for dom in ["CL", "IT"]:
    examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.asarray([int(ex["y"]) for ex in examples])
    present = np.unique(y_true)

    for row in results_df[results_df["domain"] == dom].itertuples():
        key = (dom, row.method, row.weight_E0a)
        y_pred = saved_predictions[key]

        report = prf_report(y_true, y_pred, len(LABELS))

        comparison.append({
            "domain": dom,
            "method": row.method,
            "weight_E0a": row.weight_E0a,
            "macro_f1_all_13": f1_score(
                y_true, y_pred,
                labels=np.arange(len(LABELS)),
                average="macro", zero_division=0
            ),
            "macro_f1_present_gold": f1_score(
                y_true, y_pred,
                labels=present,
                average="macro", zero_division=0
            ),
            "notebook_macro_f1": report["macro_f1"],
            "accuracy": row.accuracy,
        })

comparison_df = pd.DataFrame(comparison)
display(
    comparison_df.sort_values(
        ["domain", "macro_f1_all_13"], ascending=[True, False]
    ).round(5)
)

In [ ]:
# Experiment 2: joint class-bias tuning setup
# Validation/dev only. Does not change FINAL_CONFIG or test predictions.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

BIAS_EXP_DIR = Path(WORK) / "experiments" / "joint_class_bias_v1"
BIAS_EXP_DIR.mkdir(parents=True, exist_ok=True)

BIAS_CONFIG = {
    "CL": {
        "method": "arithmetic",
        "weight_E0a": 0.60,
        "smooth": 0.50,
        "base_bias": {"ArgumentRespondent": 1.0},
        "classes_to_test": [
            "Issue",
            "ArgumentPetitioner",
            "ArgumentRespondent",
            "PrecedentReliedUpon",
            "RulingByLowerCourt",
            "RatioOfTheDecision",
            "RulingByPresentCourt",
        ],
    },
    "IT": {
        "method": "geometric",
        "weight_E0a": 0.50,
        "smooth": 0.25,
        "base_bias": {"RulingByLowerCourt": 1.0},
        "classes_to_test": [
            "ArgumentPetitioner",
            "ArgumentRespondent",
            "PrecedentReliedUpon",
            "PrecedentNotReliedUpon",
            "RulingByLowerCourt",
            "RatioOfTheDecision",
            "RulingByPresentCourt",
            "None",
        ],
    },
}

for dom, cfg in BIAS_CONFIG.items():
    print(f"\n{dom}:")
    print("Method:", cfg["method"])
    print("E0a weight:", cfg["weight_E0a"])
    print("Smoothing:", cfg["smooth"])
    print("Base bias:", cfg["base_bias"])
    print("Classes:", cfg["classes_to_test"])
    print("Class IDs:", {c: LABEL2ID[c] for c in cfg["classes_to_test"]})

print("\nOutput directory:", BIAS_EXP_DIR)

In [ ]:
# Experiment 2A: one-class-at-a-time bias sweep
# Keep all other settings fixed.

BIAS_VALUES = [-0.5, 0.0, 0.5, 1.0]
bias_results = []
bias_predictions = {}

for dom, cfg in BIAS_CONFIG.items():
    examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.asarray([int(ex["y"]) for ex in examples], dtype=np.int64)

    logits_a = np.load(
        Path(WORK) / "runs" / f"E0a_full_sent_{dom}" / "dev_logits.npy"
    )
    logits_b = np.load(
        Path(WORK) / "runs" / f"E6_kd_student_{dom}" / "dev_logits.npy"
    )

    probs = weighted_probs(
        logits_a, logits_b,
        cfg["weight_E0a"],
        method=cfg["method"],
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS)
    )

    # Include the existing baseline and test each selected class independently.
    candidates = [("base", dict(cfg["base_bias"]))]

    for class_name in cfg["classes_to_test"]:
        for value in BIAS_VALUES:
            trial = dict(cfg["base_bias"])
            if value == 0.0:
                trial.pop(class_name, None)
            else:
                trial[class_name] = value
            candidates.append((f"{class_name}={value:+.1f}", trial))

    seen = set()
    for name, bias in candidates:
        signature = tuple(sorted(bias.items()))
        if signature in seen:
            continue
        seen.add(signature)

        y_pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                smooth=cfg["smooth"],
                class_bias=bias,
            ),
            dtype=np.int64,
        )

        report = prf_report(y_true, y_pred, len(LABELS))
        key = (dom, name)
        bias_predictions[key] = y_pred

        bias_results.append({
            "domain": dom,
            "trial": name,
            "class_bias": repr(bias),
            "macro_f1": report["macro_f1"],
            "macro_f1_all_13": report["macro_f1_all"],
            "accuracy": report["acc"],
        })

bias_df = pd.DataFrame(bias_results).sort_values(
    ["domain", "macro_f1"], ascending=[True, False]
).reset_index(drop=True)

bias_csv = BIAS_EXP_DIR / "one_class_bias_sweep.csv"
bias_df.to_csv(bias_csv, index=False)

print("Saved:", bias_csv)
display(bias_df.round(5))

In [ ]:
FINAL_CONFIG = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "arithmetic",
        "weight_E0a": 0.60,
        "smooth": 0.50,
        "class_bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.5,
        },
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "geometric",
        "weight_E0a": 0.50,
        "smooth": 0.25,
        "class_bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import json

JOINT_DIR = Path(WORK) / "experiments" / "joint_bias_refinement_v1"
JOINT_DIR.mkdir(parents=True, exist_ok=True)

JOINT_CONFIG = {
    "CL": {
        "method": "arithmetic",
        "weight_E0a": 0.60,
        "smooth": 0.50,
        "fixed_bias": {"ArgumentRespondent": 1.0},
        "tune_class": "ArgumentPetitioner",
        "values": [0.25, 0.50, 0.75],
    },
    "IT": {
        "method": "geometric",
        "weight_E0a": 0.50,
        "smooth": 0.25,
        "fixed_bias": {"RulingByLowerCourt": 1.0},
        "tune_class": "RulingByPresentCourt",
        "values": [0.50, 0.75, 1.00, 1.25],
    },
}

joint_rows = []
joint_predictions = {}

for dom, cfg in JOINT_CONFIG.items():
    # IMPORTANT: include "_logits" in the filename stem.
    logits_a = load_arr("E0a_full_sent", dom, "dev_logits")
    logits_b = load_arr("E6_kd_student", dom, "dev_logits")

    probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_E0a"],
        method=cfg["method"],
    )

    dev_examples = build_examples(DOCS[dom]["dev"], False)
    y_true = np.asarray([int(ex["y"]) for ex in dev_examples])

    assert len(y_true) == len(logits_a) == len(logits_b), (
        f"{dom}: dev examples and logits have different lengths"
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS)
    )

    for value in cfg["values"]:
        class_bias = dict(cfg["fixed_bias"])
        class_bias[cfg["tune_class"]] = float(value)

        y_pred = np.asarray(
            final_predict(
                probs, dev_examples, logT, logS,
                cfg["smooth"], class_bias
            ),
            dtype=int,
        )

        report = prf_report(y_true, y_pred, len(LABELS))
        trial = f'{cfg["tune_class"]}={value:+.2f}'

        joint_rows.append({
            "domain": dom,
            "trial": trial,
            "tuned_value": float(value),
            "class_bias": json.dumps(class_bias, sort_keys=True),
            "macro_f1": report["macro_f1"],
            "macro_f1_all_13": report["macro_f1_all"],
            "accuracy": report["acc"],
        })

        joint_predictions[(dom, float(value))] = y_pred

joint_df = pd.DataFrame(joint_rows).sort_values(
    ["domain", "macro_f1"], ascending=[True, False]
).reset_index(drop=True)

csv_path = JOINT_DIR / "joint_bias_results.csv"
joint_df.to_csv(csv_path, index=False)

display(joint_df.round(5))
print("Saved:", csv_path)


In [ ]:

FINAL_CONFIG_REFINED = {
    "CL": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "arithmetic",
        "weight_E0a": 0.60,
        "smooth": 0.50,
        "class_bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "models": ["E0a_full_sent", "E6_kd_student"],
        "ensemble": "geometric",
        "weight_E0a": 0.50,
        "smooth": 0.25,
        "class_bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

from pathlib import Path
import json

FINAL_DIR = Path(WORK) / "final_refined"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

config_path = FINAL_DIR / "selected_final_config.json"
with open(config_path, "w") as f:
    json.dump(FINAL_CONFIG_REFINED, f, indent=2)

print(json.dumps(FINAL_CONFIG_REFINED, indent=2))
print("Config saved:", config_path)


In [ ]:

import numpy as np
import json
from pathlib import Path

def predict_with_config(dom, split="test"):
    cfg = FINAL_CONFIG_REFINED[dom]

    logits_a = load_arr(cfg["models"][0], dom, f"{split}_logits")
    logits_b = load_arr(cfg["models"][1], dom, f"{split}_logits")

    probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_E0a"],
        method=cfg["ensemble"],
    )

    examples = build_examples(DOCS[dom][split], False)

    assert len(examples) == len(probs), (
        f"{dom}: examples={len(examples)}, logits={len(probs)}"
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS)
    )

    pred_ids = np.asarray(
        final_predict(
            probs,
            examples,
            logT,
            logS,
            cfg["smooth"],
            cfg["class_bias"],
        ),
        dtype=int,
    )

    assert len(pred_ids) == len(examples)
    assert np.all((pred_ids >= 0) & (pred_ids < len(LABELS)))

    return examples, pred_ids

test_outputs = {}

for dom in ["CL", "IT"]:
    examples, pred_ids = predict_with_config(dom, "test")

    records = [
        {
            "doc": str(ex["doc"]),
            "i": int(ex["i"]),
            "prediction": LABELS[int(pid)],
        }
        for ex, pid in zip(examples, pred_ids)
    ]

    out_path = FINAL_DIR / f"{dom.lower()}_predictions.json"
    with open(out_path, "w") as f:
        json.dump(records, f, indent=2)

    test_outputs[dom] = {
        "examples": examples,
        "pred_ids": pred_ids,
        "records": records,
    }

    print(f"{dom}: {len(records)} predictions saved to {out_path}")

print("Prediction directory:", FINAL_DIR)


In [ ]:

for dom in ["CL", "IT"]:
    path = FINAL_DIR / f"{dom.lower()}_predictions.json"

    with open(path) as f:
        records = json.load(f)

    print(f"\n{dom}")
    print("Exists:", path.exists())
    print("Rows:", len(records))
    print("First 3 records:")
    print(records[:3])

    assert len(records) == len(test_outputs[dom]["examples"])
    assert all(r["prediction"] in LABEL2ID for r in records)

print("\nBoth domain prediction files passed basic validation.")


In [ ]:

from pathlib import Path
import json
import os

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

keywords = (
    "sample_submission", "submission", "prediction",
    "test.json", "test.jsonl", "readme", "instruction"
)

matches = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_file() and any(k in p.name.lower() for k in keywords):
            matches.append(p)

print("=== Relevant files ===")
for p in matches[:250]:
    try:
        size = p.stat().st_size
    except OSError:
        size = -1
    print(f"{p}  [{size:,} bytes]")

print("\n=== Existing prediction JSON previews ===")
for p in matches:
    if "prediction" not in p.name.lower() and "sample_submission" not in p.name.lower():
        continue
    if p.suffix.lower() != ".json":
        continue

    print(f"\n--- {p} ---")
    try:
        with open(p, encoding="utf-8") as f:
            obj = json.load(f)
        print("Top-level type:", type(obj).__name__)
        if isinstance(obj, dict):
            print("Top-level keys:", list(obj.keys())[:40])
            for k, v in list(obj.items())[:3]:
                print(f"  {k!r}: {repr(v)[:500]}")
        elif isinstance(obj, list):
            print("Number of rows:", len(obj))
            print("First 3 rows:", repr(obj[:3])[:2000])
        else:
            print(repr(obj)[:1000])
    except Exception as e:
        print("Could not parse as JSON:", repr(e))

print("\n=== Kaggle input folders ===")
input_root = Path("/kaggle/input")
if input_root.exists():
    for p in input_root.iterdir():
        print(p)


In [ ]:

from pathlib import Path
import json

FINAL_DIR = Path(WORK) / "final_refined"
prediction_files = {
    "CL": FINAL_DIR / "cl_predictions.json",
    "IT": FINAL_DIR / "it_predictions.json",
}

validated = {}

for dom, path in prediction_files.items():
    assert path.exists(), f"Missing prediction file: {path}"

    with open(path, encoding="utf-8") as f:
        rows = json.load(f)

    assert isinstance(rows, list), f"{dom}: expected a list of records"
    assert rows, f"{dom}: prediction file is empty"

    required_keys = {"doc", "i", "prediction"}
    for idx, row in enumerate(rows):
        assert isinstance(row, dict), f"{dom}: row {idx} is not a dict"
        assert required_keys.issubset(row), (
            f"{dom}: row {idx} is missing fields "
            f"{required_keys - set(row)}"
        )
        assert row["prediction"] in LABEL2ID, (
            f"{dom}: unknown label at row {idx}: {row['prediction']}"
        )

    keys = [(str(r["doc"]), int(r["i"])) for r in rows]
    assert len(keys) == len(set(keys)), f"{dom}: duplicate document-position pairs"

    validated[dom] = rows
    print(f"{dom}: {len(rows)} valid predictions")

print("\nDomain prediction files passed structural validation.")


In [ ]:

from pathlib import Path
import zipfile
import json

WORK_PATH = Path("/kaggle/working/phase3_rr")
old_zip_candidates = [
    Path("/kaggle/working/submission.zip"),
    Path("/kaggle/working/phase3_publish/submission.zip"),
    WORK_PATH / "final_improved" / "submission_improved.zip",
]

for zp in old_zip_candidates:
    if not zp.exists():
        continue

    print("\nArchive:", zp)
    with zipfile.ZipFile(zp, "r") as zf:
        print("Members:", zf.namelist())
        for name in zf.namelist():
            if name.lower().endswith(".json"):
                obj = json.loads(zf.read(name).decode("utf-8"))
                print("JSON member:", name)
                print("Top-level type:", type(obj).__name__)
                if isinstance(obj, dict):
                    print("Number of documents:", len(obj))
                    print("Sample document:", next(iter(obj.items()), None))
                elif isinstance(obj, list):
                    print("Number of records:", len(obj))


In [ ]:

from pathlib import Path
from collections import Counter
import json
import zipfile

WORK_PATH = Path("/kaggle/working/phase3_rr")
FINAL_DIR = WORK_PATH / "final_refined"

with open(FINAL_DIR / "cl_predictions.json", encoding="utf-8") as f:
    cl_rows = json.load(f)

with open(FINAL_DIR / "it_predictions.json", encoding="utf-8") as f:
    it_rows = json.load(f)

# Convert each domain's records into document_id -> ordered label list.
def records_to_documents(rows, domain):
    docs = {}
    seen = set()

    for row in rows:
        doc_id = str(row["doc"])
        position = int(row["i"])
        label = row["prediction"]

        assert label in LABEL2ID, f"Invalid label: {label}"
        assert position >= 0, f"Negative position: {doc_id}/{position}"

        if doc_id not in docs:
            docs[doc_id] = {}

        key = (doc_id, position)
        assert key not in seen, f"Duplicate prediction: {key}"
        seen.add(key)
        docs[doc_id][position] = label

    result = {}
    for doc_id, indexed_labels in docs.items():
        positions = sorted(indexed_labels)
        assert positions == list(range(len(positions))), (
            f"{domain}: non-contiguous positions for {doc_id}: "
            f"{positions[:10]}"
        )
        result[doc_id] = [indexed_labels[i] for i in positions]

    return result

cl_docs = records_to_documents(cl_rows, "CL")
it_docs = records_to_documents(it_rows, "IT")

overlap = set(cl_docs) & set(it_docs)
assert not overlap, f"Documents appear in both domains: {list(overlap)[:5]}"

prediction = {**cl_docs, **it_docs}

# Check document and sentence counts against the notebook's loaded test splits.
def expected_documents(domain):
    expected = {}
    for doc in DOCS[domain]["test"]:
        if isinstance(doc, dict):
            doc_id = str(
                doc.get("id", doc.get("doc_id", doc.get("name", "")))
            )
            # If IDs are stored elsewhere in the document, the check below
            # will reveal the mismatch rather than silently changing them.
            if not doc_id:
                raise ValueError(
                    f"Could not infer document ID for a {domain} test document. "
                    "Inspect DOCS[domain]['test'][0] before proceeding."
                )
            expected[doc_id] = len(doc.get("sentences", doc.get("labels", [])))
        else:
            raise TypeError(
                f"Unexpected test document type: {type(doc).__name__}. "
                "Inspect DOCS[domain]['test'][0] before proceeding."
            )
    return expected

# Validate document IDs against the example-level document IDs, which are
# derived by the notebook's established preprocessing pipeline.
for domain, rows in [("CL", cl_rows), ("IT", it_rows)]:
    examples = build_examples(DOCS[domain]["test"], False)
    expected_counts = Counter(str(ex["doc"]) for ex in examples)
    actual_counts = Counter(str(r["doc"]) for r in rows)

    assert expected_counts == actual_counts, (
        f"{domain}: prediction document IDs/counts don't match test examples"
    )

    for doc_id, count in expected_counts.items():
        assert len(prediction[doc_id]) == count, (
            f"{domain}: sentence count mismatch for {doc_id}"
        )

assert len(prediction) == len(cl_docs) + len(it_docs)
assert all(isinstance(v, list) and len(v) > 0 for v in prediction.values())

# Save to a new location; preserve all existing submissions.
out_path = FINAL_DIR / "prediction.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False)

print("Final prediction:", out_path)
print("Documents:", len(prediction))
print("CL documents:", len(cl_docs), "| CL predictions:", sum(map(len, cl_docs.values())))
print("IT documents:", len(it_docs), "| IT predictions:", sum(map(len, it_docs.values())))
print("Total sentence predictions:", sum(map(len, prediction.values())))
print("Sample:", next(iter(prediction.items())))


In [ ]:

import json
import zipfile
from pathlib import Path

FINAL_DIR = Path(WORK) / "final_refined"
prediction_path = FINAL_DIR / "prediction.json"
zip_path = FINAL_DIR / "submission_refined.zip"

with open(prediction_path, encoding="utf-8") as f:
    final_obj = json.load(f)

assert isinstance(final_obj, dict)
assert len(final_obj) > 0
assert all(isinstance(k, str) and isinstance(v, list)
           for k, v in final_obj.items())
assert all(label in LABEL2ID for labels in final_obj.values() for label in labels)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")

# Read back the archive and confirm the contents are identical.
with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    zipped_obj = json.loads(zf.read("prediction.json").decode("utf-8"))
    assert zipped_obj == final_obj

print("Submission ZIP verified:", zip_path)
print("Prediction JSON:", prediction_path)
print("Documents:", len(final_obj))
print("Sentence predictions:", sum(len(v) for v in final_obj.values()))


In [ ]:

from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd

WORK_PATH = Path("/kaggle/working/phase3_rr")

CANDIDATES = {
    "previous": WORK_PATH / "prediction.json",
    "previous_improved": WORK_PATH / "final_improved" / "prediction.json",
    "refined": WORK_PATH / "final_refined" / "prediction.json",
}

predictions = {}
for name, path in CANDIDATES.items():
    if path.exists():
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, dict):
            predictions[name] = obj
            print(f"{name}: {len(obj)} documents | {path}")
        else:
            print(f"Skipping {name}: expected document -> labels dictionary")
    else:
        print(f"Not found: {path}")

assert "refined" in predictions, "Refined prediction.json not found."

# Check label frequencies.
dist_rows = []
for name, obj in predictions.items():
    counts = Counter(
        label for labels in obj.values() for label in labels
    )
    total = sum(counts.values())
    for label in LABELS:
        dist_rows.append({
            "submission": name,
            "label": label,
            "count": counts.get(label, 0),
            "fraction": counts.get(label, 0) / max(total, 1),
        })

dist_df = pd.DataFrame(dist_rows)
display(
    dist_df.pivot(index="label", columns="submission", values="fraction")
    .fillna(0).round(4)
)

# Compare document-level predictions against each available older submission.
for old_name in [n for n in predictions if n != "refined"]:
    old = predictions[old_name]
    new = predictions["refined"]

    print(f"\n=== {old_name} vs refined ===")
    print("Documents only in old:", len(set(old) - set(new)))
    print("Documents only in refined:", len(set(new) - set(old)))

    common = sorted(set(old) & set(new))
    rows = []

    for doc_id in common:
        a = old[doc_id]
        b = new[doc_id]

        if len(a) != len(b):
            rows.append({
                "doc": doc_id,
                "old_n": len(a),
                "new_n": len(b),
                "changed": None,
                "change_rate": None,
            })
            continue

        changed = sum(x != y for x, y in zip(a, b))
        rows.append({
            "doc": doc_id,
            "old_n": len(a),
            "new_n": len(b),
            "changed": changed,
            "change_rate": changed / max(len(a), 1),
        })

    compare_df = pd.DataFrame(rows)
    display(compare_df.sort_values(
        "change_rate", ascending=False, na_position="first"
    ).round(4))

    valid = compare_df.dropna(subset=["change_rate"])
    if len(valid):
        total_sentences = int(valid["old_n"].sum())
        total_changed = int(valid["changed"].sum())
        print("Matched documents:", len(valid))
        print("Sentences compared:", total_sentences)
        print("Changed predictions:", total_changed)
        print("Overall change rate:",
              round(total_changed / max(total_sentences, 1), 4))

    print("Document-key sets identical:", set(old) == set(new))


In [ ]:

# EXPERIMENT A1 — Validate existing prediction files
from pathlib import Path
from collections import Counter
import json
import numpy as np

WORK = Path("/kaggle/working/phase3_rr")
OUT = WORK / "diagnostics_20261014"
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "previous": WORK / "prediction.json",
    "previous_improved": WORK / "final_improved" / "prediction.json",
    "refined": WORK / "final_refined" / "prediction.json",
}

LABELS = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent",
    "Statute", "Dissent", "PrecedentReliedUpon",
    "PrecedentNotReliedUpon", "PrecedentOverruled",
    "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]
valid = set(LABELS)
preds = {}

for name, path in FILES.items():
    assert path.exists(), f"Missing file: {path}"
    with open(path) as f:
        obj = json.load(f)

    assert isinstance(obj, dict), f"{name}: expected dict doc_id -> labels"
    assert all(isinstance(v, list) for v in obj.values()), \
        f"{name}: some document values are not lists"

    bad = {
        str(doc): sorted(set(labels) - valid)
        for doc, labels in obj.items()
        if set(labels) - valid
    }
    assert not bad, f"{name}: unknown labels found: {bad}"

    preds[name] = obj
    counts = Counter(label for seq in obj.values() for label in seq)
    total = sum(counts.values())

    print(f"\n{name}: {len(obj)} docs, {total} sentences")
    print("Label counts:", dict(counts))

base = preds["previous_improved"]
for name, obj in preds.items():
    print(f"\n{name} vs previous_improved")
    print("Same document IDs:", set(obj) == set(base))
    if set(obj) == set(base):
        print("Same sentence counts:",
              all(len(obj[d]) == len(base[d]) for d in base))
        print("Total sentences:", sum(map(len, obj.values())))

# Save a concise structural report.
report = {
    name: {
        "documents": len(obj),
        "sentences": sum(map(len, obj.values())),
        "label_counts": dict(Counter(x for seq in obj.values() for x in seq)),
    }
    for name, obj in preds.items()
}
with open(OUT / "experiment_A1_structure.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved:", OUT / "experiment_A1_structure.json")


In [ ]:

# EXPERIMENT B0 — Discover the existing development-evaluation API
import inspect

for name in [
    "build_examples", "load_arr", "weighted_probs",
    "final_predict", "transition_logprobs", "prf_report"
]:
    obj = globals().get(name)
    if obj is None:
        print(f"\n{name}: NOT FOUND")
    else:
        try:
            print(f"\n{name}{inspect.signature(obj)}")
        except Exception:
            print(f"\n{name}: found")

print("\nDOCS:", type(globals().get("DOCS")).__name__)
if isinstance(globals().get("DOCS"), dict):
    for k, v in DOCS.items():
        print("DOCS key:", k, "| type:", type(v).__name__,
              "| keys:", list(v.keys())[:20] if isinstance(v, dict) else "")

print("\nLikely dev/example variables:")
for k, v in globals().copy().items():
    kl = k.lower()
    if any(t in kl for t in ["dev_examples", "examples_dev", "dev_logits",
                             "dev_data", "dev_docs", "label2id"]):
        try:
            print(k, "|", type(v).__name__, "| len =", len(v))
        except Exception:
            print(k, "|", type(v).__name__)


In [ ]:

# EXPERIMENT B1 — Inspect development data and saved logits
import inspect
import numpy as np
from pathlib import Path

WORK = Path("/kaggle/working/phase3_rr")

print("=== Existing prediction helpers ===")
for name in ["final_predict", "transition_logprobs", "prf_report",
             "weighted_probs", "build_examples"]:
    fn = globals().get(name)
    if fn is not None:
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(fn))
        except Exception as e:
            print("Source unavailable:", e)

print("\n=== Document structure ===")
for dom in ["CL", "IT"]:
    print(f"\n{dom}")
    for split in ["train", "dev", "test"]:
        docs = DOCS[dom][split]
        print(split, "type =", type(docs).__name__,
              "length =", len(docs))
        if len(docs):
            first = docs[0]
            if isinstance(first, dict):
                print(" first doc keys:", list(first.keys()))
                for k, v in first.items():
                    if isinstance(v, (str, int, float, bool)):
                        print(f"   {k}: {str(v)[:100]}")
                    elif isinstance(v, list):
                        print(f"   {k}: list length {len(v)}; sample:",
                              str(v[:2])[:200])
            else:
                print(" first doc type:", type(first).__name__,
                      "sample:", str(first)[:300])

print("\n=== Saved dev logits ===")
for model_name in ["E0a_full_sent", "E6_kd_student",
                   "E6_student_nokd"]:
    for dom in ["CL", "IT"]:
        p = WORK / "runs" / f"{model_name}_{dom}" / "dev_logits.npy"
        if p.exists():
            a = np.load(p)
            print(model_name, dom, a.shape, a.dtype)
        else:
            print("MISSING:", p)

print("\n=== Relevant existing variables ===")
for name in sorted(list(globals())):
    low = name.lower()
    if any(s in low for s in [
        "tokenizer", "context", "ctx", "dev_examples",
        "dev_docs", "logt", "logs", "label2id"
    ]):
        obj = globals()[name]
        try:
            info = f"{type(obj).__name__}, len={len(obj)}"
        except Exception:
            info = type(obj).__name__
        print(name, "->", info)


In [ ]:

# EXPERIMENT B1 — Evaluate previous-improved vs refined on dev
import json
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/phase3_rr")
OUT = WORK / "diagnostics_20261014"
OUT.mkdir(parents=True, exist_ok=True)

LABELS = list(LABEL2ID.keys())
K = len(LABELS)

CONFIGS = {
    "previous_improved": {
        "CL": {
            "weight_E0a": 0.60, "method": "arithmetic",
            "smooth": 0.50, "bias": {}
        },
        "IT": {
            "weight_E0a": 0.60, "method": "arithmetic",
            "smooth": 0.50, "bias": {}
        }
    },
    "refined": {
        "CL": {
            "weight_E0a": 0.60, "method": "arithmetic",
            "smooth": 0.50,
            "bias": {"ArgumentRespondent": 1.0,
                     "ArgumentPetitioner": 0.75}
        },
        "IT": {
            "weight_E0a": 0.50, "method": "geometric",
            "smooth": 0.25,
            "bias": {"RulingByLowerCourt": 1.0,
                     "RulingByPresentCourt": 1.0}
        }
    }
}

# Confirm saved example order matches the rebuilt development examples.
# ctx=True and all other values are the settings used by the project helper.
all_results = []
per_class_rows = []
dev_cache = {}

for dom in ["CL", "IT"]:
    docs = DOCS[dom]["dev"]
    examples = build_examples(
        docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    dev_cache[dom] = (docs, examples)

    y_true = np.asarray([e["y"] for e in examples], dtype=int)
    assert np.all(y_true >= 0), f"{dom}: missing dev labels"

    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")
    assert z_a.shape == z_b.shape == (len(examples), K), (
        dom, z_a.shape, z_b.shape, len(examples)
    )

    logT, logS = transition_logprobs(docs, LABEL2ID, K)

    for config_name, config in CONFIGS.items():
        c = config[dom]
        probs = weighted_probs(
            z_a, z_b, c["weight_E0a"], method=c["method"]
        )
        pred = final_predict(
            probs, examples, logT, logS,
            c["smooth"], c["bias"]
        )
        pred = np.asarray(pred, dtype=int)
        report = prf_report(y_true, pred, K)

        all_results.append({
            "configuration": config_name,
            "domain": dom,
            "macro_f1": report["macro_f1"],
            "macro_f1_all": report["macro_f1_all"],
            "macro_precision": report["macro_p"],
            "macro_recall": report["macro_r"],
            "accuracy": report["acc"],
            "n_present": report["n_present"],
        })

        for i, label in enumerate(LABELS):
            per_class_rows.append({
                "configuration": config_name,
                "domain": dom,
                "label": label,
                "precision": report["prec"][i],
                "recall": report["rec"][i],
                "f1": report["f1"][i],
                "support": int(report["support"][i]),
            })

        print(
            f"{dom} | {config_name}: "
            f"macro-F1={report['macro_f1']:.5f}, "
            f"all13={report['macro_f1_all']:.5f}, "
            f"accuracy={report['acc']:.5f}"
        )

summary = pd.DataFrame(all_results)
per_class = pd.DataFrame(per_class_rows)

display(summary)
display(per_class.pivot_table(
    index=["domain", "label"],
    columns="configuration",
    values="f1"
).reset_index())

summary.to_csv(OUT / "experiment_B1_summary.csv", index=False)
per_class.to_csv(OUT / "experiment_B1_per_class.csv", index=False)

print("Saved experiment B results to", OUT)


In [ ]:

# EXPERIMENT C1 — Controlled smoothing ablation on dev
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/phase3_rr/diagnostics_20261014")
OUT.mkdir(parents=True, exist_ok=True)

SMOOTH_VALUES = [0.0, 0.10, 0.25, 0.50, 0.75]

SETTINGS = {
    "CL": {
        "weight_E0a": 0.60,
        "method": "arithmetic",
        "biases": {
            "none": {},
            "refined": {
                "ArgumentRespondent": 1.0,
                "ArgumentPetitioner": 0.75
            }
        }
    },
    "IT": {
        "weight_E0a": 0.50,
        "method": "geometric",
        "biases": {
            "none": {},
            "refined": {
                "RulingByLowerCourt": 1.0,
                "RulingByPresentCourt": 1.0
            }
        }
    }
}

rows = []

for dom in ["CL", "IT"]:
    docs, examples = dev_cache[dom]
    y_true = np.asarray([e["y"] for e in examples], dtype=int)
    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")

    cfg = SETTINGS[dom]
    probs = weighted_probs(
        z_a, z_b, cfg["weight_E0a"], method=cfg["method"]
    )
    logT, logS = transition_logprobs(docs, LABEL2ID, K)

    for bias_name, bias in cfg["biases"].items():
        for smooth in SMOOTH_VALUES:
            pred = np.asarray(
                final_predict(probs, examples, logT, logS, smooth, bias),
                dtype=int
            )
            r = prf_report(y_true, pred, K)
            rows.append({
                "domain": dom,
                "bias_setting": bias_name,
                "smooth": smooth,
                "macro_f1": r["macro_f1"],
                "macro_f1_all": r["macro_f1_all"],
                "macro_precision": r["macro_p"],
                "macro_recall": r["macro_r"],
                "accuracy": r["acc"],
            })

results_C = pd.DataFrame(rows).sort_values(
    ["domain", "bias_setting", "macro_f1"],
    ascending=[True, True, False]
)
display(results_C)

print("\nBest smoothing per domain and bias setting:")
display(
    results_C.loc[
        results_C.groupby(["domain", "bias_setting"])["macro_f1"].idxmax()
    ].sort_values(["domain", "bias_setting"])
)

results_C.to_csv(OUT / "experiment_C1_smoothing.csv", index=False)
print("Saved:", OUT / "experiment_C1_smoothing.csv")


In [ ]:

# EXPERIMENT D1 — Per-document robustness of refined ensemble
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/phase3_rr/diagnostics_20261014")

rows = []

for dom in ["CL", "IT"]:
    docs, examples = dev_cache[dom]
    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")
    cfg = SETTINGS[dom]

    probs = weighted_probs(
        z_a, z_b, cfg["weight_E0a"], method=cfg["method"]
    )
    logT, logS = transition_logprobs(docs, LABEL2ID, K)

    for bias_name, bias in cfg["biases"].items():
        for smooth in SMOOTH_VALUES:
            pred = np.asarray(
                final_predict(probs, examples, logT, logS, smooth, bias),
                dtype=int
            )

            # Use each example's document ID to separate scores.
            for doc_id in dict.fromkeys(e["doc"] for e in examples):
                idx = np.asarray([
                    j for j, e in enumerate(examples) if e["doc"] == doc_id
                ], dtype=int)
                y_doc = np.asarray([examples[j]["y"] for j in idx], dtype=int)
                p_doc = pred[idx]
                r = prf_report(y_doc, p_doc, K)

                rows.append({
                    "domain": dom,
                    "doc_id": str(doc_id),
                    "bias_setting": bias_name,
                    "smooth": smooth,
                    "sentences": len(idx),
                    "macro_f1": r["macro_f1"],
                    "macro_f1_all": r["macro_f1_all"],
                    "accuracy": r["acc"],
                })

results_D = pd.DataFrame(rows)
results_D.to_csv(OUT / "experiment_D1_document_scores.csv", index=False)

# Compare the mean document score and its spread across the five documents.
robustness = (
    results_D.groupby(["domain", "bias_setting", "smooth"])
    .agg(
        mean_doc_macro_f1=("macro_f1", "mean"),
        std_doc_macro_f1=("macro_f1", "std"),
        min_doc_macro_f1=("macro_f1", "min"),
        max_doc_macro_f1=("macro_f1", "max"),
    )
    .reset_index()
    .sort_values(["domain", "bias_setting", "mean_doc_macro_f1"],
                 ascending=[True, True, False])
)
display(robustness)
robustness.to_csv(OUT / "experiment_D1_robustness.csv", index=False)

print("Saved:", OUT)


In [ ]:
logT, logS = transition_logprobs(docs, LABEL2ID, K)

In [ ]:

# EXPERIMENT B2 — Leakage-free development evaluation
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/phase3_rr")
OUT = WORK / "diagnostics_20261014"
OUT.mkdir(parents=True, exist_ok=True)

LABELS = list(LABEL2ID.keys())
K = len(LABELS)

CONFIGS = {
    "baseline_no_bias": {
        "CL": dict(weight=0.60, method="arithmetic", smooth=0.50, bias={}),
        "IT": dict(weight=0.60, method="arithmetic", smooth=0.50, bias={}),
    },
    "refined": {
        "CL": dict(
            weight=0.60, method="arithmetic", smooth=0.50,
            bias={"ArgumentRespondent": 1.0, "ArgumentPetitioner": 0.75}
        ),
        "IT": dict(
            weight=0.50, method="geometric", smooth=0.25,
            bias={"RulingByLowerCourt": 1.0, "RulingByPresentCourt": 1.0}
        ),
    },
    "refined_more_CL_smoothing": {
        "CL": dict(
            weight=0.60, method="arithmetic", smooth=0.75,
            bias={"ArgumentRespondent": 1.0, "ArgumentPetitioner": 0.75}
        ),
        "IT": dict(
            weight=0.50, method="geometric", smooth=0.25,
            bias={"RulingByLowerCourt": 1.0, "RulingByPresentCourt": 1.0}
        ),
    },
}

rows = []
class_rows = []
doc_rows = []
dev_cache_clean = {}

for dom in ["CL", "IT"]:
    train_docs = DOCS[dom]["train"]
    dev_docs = DOCS[dom]["dev"]

    examples = build_examples(
        dev_docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    dev_cache_clean[dom] = (dev_docs, examples)

    y_true = np.asarray([e["y"] for e in examples], dtype=int)
    assert np.all(y_true >= 0)

    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")
    assert z_a.shape == z_b.shape == (len(examples), K)

    # CRITICAL: transition counts from TRAIN only, never dev/test.
    logT, logS = transition_logprobs(train_docs, LABEL2ID, K)

    for name, cfgs in CONFIGS.items():
        c = cfgs[dom]
        probs = weighted_probs(
            z_a, z_b, c["weight"], method=c["method"]
        )
        pred = np.asarray(
            final_predict(probs, examples, logT, logS,
                          c["smooth"], c["bias"]),
            dtype=int
        )
        r = prf_report(y_true, pred, K)

        rows.append({
            "configuration": name,
            "domain": dom,
            "macro_f1": r["macro_f1"],
            "macro_f1_all": r["macro_f1_all"],
            "macro_precision": r["macro_p"],
            "macro_recall": r["macro_r"],
            "accuracy": r["acc"],
            "n_present": r["n_present"],
        })

        for j, label in enumerate(LABELS):
            class_rows.append({
                "configuration": name,
                "domain": dom,
                "label": label,
                "f1": r["f1"][j],
                "precision": r["prec"][j],
                "recall": r["rec"][j],
                "support": int(r["support"][j]),
            })

        for doc_id in dict.fromkeys(e["doc"] for e in examples):
            idx = np.array(
                [i for i, e in enumerate(examples) if e["doc"] == doc_id],
                dtype=int
            )
            rd = prf_report(y_true[idx], pred[idx], K)
            doc_rows.append({
                "configuration": name,
                "domain": dom,
                "doc_id": str(doc_id),
                "sentences": len(idx),
                "macro_f1": rd["macro_f1"],
                "macro_f1_all": rd["macro_f1_all"],
                "accuracy": rd["acc"],
            })

summary_clean = pd.DataFrame(rows)
classes_clean = pd.DataFrame(class_rows)
docs_clean = pd.DataFrame(doc_rows)

print("=== Aggregate scores (training-only transition statistics) ===")
display(summary_clean)

print("=== Per-document scores ===")
display(docs_clean)

print("=== Per-class F1 ===")
display(classes_clean.pivot_table(
    index=["domain", "label"],
    columns="configuration",
    values="f1"
).reset_index())

summary_clean.to_csv(OUT / "experiment_B2_summary.csv", index=False)
classes_clean.to_csv(OUT / "experiment_B2_per_class.csv", index=False)
docs_clean.to_csv(OUT / "experiment_B2_per_document.csv", index=False)

print("Saved corrected evaluation to:", OUT)


In [ ]:

# EXPERIMENT C2 — Leakage-free smoothing sweep
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/phase3_rr")
OUT = WORK / "diagnostics_20261014"
OUT.mkdir(parents=True, exist_ok=True)

SMOOTH_VALUES = [0.0, 0.10, 0.25, 0.50, 0.75, 1.0]

SETTINGS = {
    "CL": {
        "weight": 0.60,
        "method": "arithmetic",
        "biases": {
            "none": {},
            "refined": {
                "ArgumentRespondent": 1.0,
                "ArgumentPetitioner": 0.75,
            },
        },
    },
    "IT": {
        "weight": 0.50,
        "method": "geometric",
        "biases": {
            "none": {},
            "refined": {
                "RulingByLowerCourt": 1.0,
                "RulingByPresentCourt": 1.0,
            },
        },
    },
}

rows = []
class_rows = []
doc_rows = []

for dom in ["CL", "IT"]:
    train_docs = DOCS[dom]["train"]
    dev_docs = DOCS[dom]["dev"]

    examples = build_examples(
        dev_docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    y = np.asarray([e["y"] for e in examples], dtype=int)

    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")
    assert z_a.shape == z_b.shape == (len(examples), len(LABEL2ID))

    probs = weighted_probs(
        z_a, z_b, SETTINGS[dom]["weight"],
        method=SETTINGS[dom]["method"]
    )

    # Never use dev labels to estimate transition probabilities.
    logT, logS = transition_logprobs(train_docs, LABEL2ID, K)

    for bias_name, bias in SETTINGS[dom]["biases"].items():
        for smooth in SMOOTH_VALUES:
            pred = np.asarray(
                final_predict(probs, examples, logT, logS, smooth, bias),
                dtype=int
            )
            report = prf_report(y, pred, K)

            rows.append({
                "domain": dom,
                "bias_setting": bias_name,
                "smooth": smooth,
                "macro_f1": report["macro_f1"],
                "macro_f1_all": report["macro_f1_all"],
                "macro_precision": report["macro_p"],
                "macro_recall": report["macro_r"],
                "accuracy": report["acc"],
            })

            for j, label in enumerate(LABELS):
                class_rows.append({
                    "domain": dom,
                    "bias_setting": bias_name,
                    "smooth": smooth,
                    "label": label,
                    "f1": report["f1"][j],
                    "precision": report["prec"][j],
                    "recall": report["rec"][j],
                    "support": int(report["support"][j]),
                })

            # Document scores: diagnostic only; short documents can be noisy.
            for doc_id in dict.fromkeys(e["doc"] for e in examples):
                ix = np.array(
                    [i for i, e in enumerate(examples)
                     if e["doc"] == doc_id],
                    dtype=int
                )
                rd = prf_report(y[ix], pred[ix], K)
                doc_rows.append({
                    "domain": dom,
                    "doc_id": str(doc_id),
                    "bias_setting": bias_name,
                    "smooth": smooth,
                    "sentences": len(ix),
                    "macro_f1": rd["macro_f1"],
                    "macro_f1_all": rd["macro_f1_all"],
                    "accuracy": rd["acc"],
                })

results = pd.DataFrame(rows)
class_results = pd.DataFrame(class_rows)
doc_results = pd.DataFrame(doc_rows)

print("=== Aggregate results, ranked within each domain/bias setting ===")
display(results.sort_values(
    ["domain", "bias_setting", "macro_f1"],
    ascending=[True, True, False]
))

print("=== Best setting per domain and bias configuration ===")
best = results.loc[
    results.groupby(["domain", "bias_setting"])["macro_f1"].idxmax()
].sort_values(["domain", "bias_setting"])
display(best)

print("=== Per-document robustness summary ===")
robust = (
    doc_results.groupby(["domain", "bias_setting", "smooth"])
    .agg(
        mean_doc_f1=("macro_f1", "mean"),
        std_doc_f1=("macro_f1", "std"),
        min_doc_f1=("macro_f1", "min"),
        max_doc_f1=("macro_f1", "max"),
    )
    .reset_index()
)
display(robust.sort_values(
    ["domain", "bias_setting", "mean_doc_f1"],
    ascending=[True, True, False]
))

results.to_csv(OUT / "experiment_C2_summary.csv", index=False)
class_results.to_csv(OUT / "experiment_C2_per_class.csv", index=False)
doc_results.to_csv(OUT / "experiment_C2_per_document.csv", index=False)
robust.to_csv(OUT / "experiment_C2_robustness.csv", index=False)

print("Saved leakage-free C2 results to:", OUT)


In [ ]:

# EXPERIMENT E1 — Transition-prior alpha sensitivity
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/phase3_rr/diagnostics_20261014")
ALPHAS = [0.25, 1.0, 2.0, 5.0]

CURRENT = {
    "CL": {
        "weight": 0.60, "method": "arithmetic", "smooth": 0.50,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "weight": 0.50, "method": "geometric", "smooth": 0.25,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

rows = []

for dom in ["CL", "IT"]:
    train_docs = DOCS[dom]["train"]
    dev_docs = DOCS[dom]["dev"]

    examples = build_examples(
        dev_docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    y = np.asarray([e["y"] for e in examples], dtype=int)

    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")

    cfg = CURRENT[dom]
    probs = weighted_probs(
        z_a, z_b, cfg["weight"], method=cfg["method"]
    )

    for alpha in ALPHAS:
        # Train-only transition estimates; alpha changes the prior.
        logT, logS = transition_logprobs(
            train_docs, LABEL2ID, K, alpha=alpha
        )
        pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                cfg["smooth"], cfg["bias"]
            ),
            dtype=int
        )
        r = prf_report(y, pred, K)

        rows.append({
            "domain": dom,
            "alpha": alpha,
            "macro_f1": r["macro_f1"],
            "macro_f1_all": r["macro_f1_all"],
            "macro_precision": r["macro_p"],
            "macro_recall": r["macro_r"],
            "accuracy": r["acc"],
        })

results_E = pd.DataFrame(rows)
display(results_E.sort_values(["domain", "macro_f1"], ascending=[True, False]))
results_E.to_csv(OUT / "experiment_E1_alpha.csv", index=False)
print("Saved:", OUT / "experiment_E1_alpha.csv")


In [ ]:

# EXPERIMENT F1 — Small ensemble-weight sensitivity check
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/phase3_rr/diagnostics_20261014")
WEIGHTS = {
    "CL": [0.50, 0.55, 0.60, 0.65, 0.70],
    "IT": [0.40, 0.45, 0.50, 0.55, 0.60],
}

CFG = {
    "CL": {
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

rows = []

for dom in ["CL", "IT"]:
    train_docs = DOCS[dom]["train"]
    dev_docs = DOCS[dom]["dev"]
    c = CFG[dom]

    examples = build_examples(
        dev_docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    y = np.asarray([e["y"] for e in examples], dtype=int)

    z_a = load_arr("E0a_full_sent", dom, "dev_logits")
    z_b = load_arr("E6_kd_student", dom, "dev_logits")
    logT, logS = transition_logprobs(
        train_docs, LABEL2ID, K, alpha=c["alpha"]
    )

    for weight in WEIGHTS[dom]:
        probs = weighted_probs(
            z_a, z_b, weight, method=c["method"]
        )
        pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                c["smooth"], c["bias"]
            ),
            dtype=int
        )
        r = prf_report(y, pred, K)
        rows.append({
            "domain": dom,
            "weight_E0a": weight,
            "weight_KD": round(1 - weight, 2),
            "macro_f1": r["macro_f1"],
            "macro_f1_all": r["macro_f1_all"],
            "accuracy": r["acc"],
        })

results_F = pd.DataFrame(rows)
display(results_F.sort_values(
    ["domain", "macro_f1"], ascending=[True, False]
))
results_F.to_csv(OUT / "experiment_F1_weight_sensitivity.csv", index=False)
print("Saved:", OUT / "experiment_F1_weight_sensitivity.csv")


In [ ]:

# FINAL CANDIDATE — Build predictions using selected configurations
import json
import zipfile
import numpy as np
from pathlib import Path
from collections import Counter

WORK = Path("/kaggle/working/phase3_rr")
OUT = WORK / "final_candidate_20261014"
OUT.mkdir(parents=True, exist_ok=True)

LABELS = list(LABEL2ID.keys())
K = len(LABELS)

FINAL_CFG = {
    "CL": {
        "weight": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "weight": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

all_records = []
domain_counts = {}

for dom in ["CL", "IT"]:
    docs = DOCS[dom]["test"]
    cfg = FINAL_CFG[dom]

    examples = build_examples(
        docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )

    z_a = load_arr("E0a_full_sent", dom, "test_logits")
    z_b = load_arr("E6_kd_student", dom, "test_logits")

    assert z_a.shape == z_b.shape == (len(examples), K), (
        dom, z_a.shape, z_b.shape, len(examples)
    )

    # Transition statistics are estimated only from labeled training data.
    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K, alpha=cfg["alpha"]
    )

    probs = weighted_probs(
        z_a, z_b, cfg["weight"], method=cfg["method"]
    )

    pred = np.asarray(
        final_predict(
            probs, examples, logT, logS,
            cfg["smooth"], cfg["bias"]
        ),
        dtype=int
    )
    assert len(pred) == len(examples)
    assert np.all((pred >= 0) & (pred < K))

    records = [
        {
            "doc": str(ex["doc"]),
            "i": int(ex["i"]),
            "prediction": LABELS[int(p)],
        }
        for ex, p in zip(examples, pred)
    ]
    all_records.extend(records)
    domain_counts[dom] = {
        "documents": len(docs),
        "sentences": len(examples),
        "prediction_counts": dict(Counter(r["prediction"] for r in records)),
    }

    with open(OUT / f"{dom.lower()}_predictions.json", "w") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

# Assemble one ordered list of labels per document.
grouped = {}
for r in all_records:
    grouped.setdefault(r["doc"], []).append((r["i"], r["prediction"]))

prediction = {
    doc: [label for _, label in sorted(items)]
    for doc, items in grouped.items()
}

# Validate document-level sequence lengths against the original test docs.
expected = {}
for dom in ["CL", "IT"]:
    for d in DOCS[dom]["test"]:
        doc_id = str(d["id"])
        if doc_id in expected:
            raise ValueError(f"Duplicate document ID across domains: {doc_id}")
        expected[doc_id] = len(d["sents"])

assert set(prediction) == set(expected), (
    set(expected) - set(prediction), set(prediction) - set(expected)
)
for doc, n in expected.items():
    assert len(prediction[doc]) == n, (doc, len(prediction[doc]), n)

prediction_path = OUT / "prediction.json"
with open(prediction_path, "w") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)

# Package exactly prediction.json at the archive root.
zip_path = OUT / "submission_candidate.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(prediction_path, arcname="prediction.json")

with open(OUT / "config_and_counts.json", "w") as f:
    json.dump(
        {"config": FINAL_CFG, "domain_counts": domain_counts,
         "total_documents": len(prediction),
         "total_sentences": sum(map(len, prediction.values()))},
        f, ensure_ascii=False, indent=2
    )

print("Candidate created:", prediction_path)
print("ZIP created:", zip_path)
print("Documents:", len(prediction))
print("Sentences:", sum(map(len, prediction.values())))
print(json.dumps(domain_counts, indent=2))


In [ ]:

# Experiment 1 — baseline and artifact checks
# Safe: does not modify checkpoints or existing submissions.

from pathlib import Path
import numpy as np
import json

EXP_DIR = Path("/kaggle/working/phase3_rr/experiments/exp1_model_comparison")
EXP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = [
    "E0a_full_sent",
    "E6_kd_student",
    "E6_student_nokd",
]
DOMAINS = ["CL", "IT"]

print("Experiment folder:", EXP_DIR)
print("Available domains:", DOMAINS)
print("Available models:", MODEL_NAMES)

# Check the existing notebook state before running any scoring.
for name in ["DOCS", "LABEL2ID", "load_arr", "build_examples",
             "transition_logprobs", "final_predict", "prf_report"]:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

if "DOCS" in globals():
    for dom in DOMAINS:
        print(f"\n--- {dom} ---")
        for split in ["train", "dev", "test"]:
            docs = DOCS[dom][split]
            print(f"{split}: {len(docs)} documents")
            if docs:
                first = docs[0]
                print("  first document type:", type(first).__name__)
                if isinstance(first, dict):
                    print("  first document keys:", list(first.keys()))

# Check that all three models have development and test logits.
if "load_arr" in globals():
    for dom in DOMAINS:
        for model in MODEL_NAMES:
            for split in ["dev", "test"]:
                try:
                    arr = load_arr(model, dom, f"{split}_logits.npy")
                    arr = np.asarray(arr)
                    print(
                        f"{dom:2s} | {model:18s} | {split:4s} | "
                        f"shape={arr.shape} | finite={np.isfinite(arr).all()}"
                    )
                except Exception as e:
                    print(f"MISSING/ERROR: {dom} | {model} | {split}: {e}")


In [ ]:

# EXPERIMENT 1 — Standalone models + broad ensemble comparison
# Read-only with respect to checkpoints and submissions.
# All transition statistics come from TRAIN documents only.

from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import json

EXP1 = Path("/kaggle/working/phase3_rr/experiments/exp1_model_comparison")
EXP1.mkdir(parents=True, exist_ok=True)

DOMAINS = ["CL", "IT"]
MODELS = ["E0a_full_sent", "E6_kd_student", "E6_student_nokd"]
WEIGHTS = [0.00, 0.25, 0.40, 0.50, 0.60, 0.75, 1.00]

# Freeze the current post-processing configuration for a fair comparison.
POST_CFG = {
    "CL": {
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {"ArgumentRespondent": 1.0, "ArgumentPetitioner": 0.75},
    },
    "IT": {
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {"RulingByLowerCourt": 1.0, "RulingByPresentCourt": 1.0},
    },
}

def exp1_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.maximum(e.sum(axis=1, keepdims=True), 1e-300)

def exp1_score(y, pred):
    r = prf_report(y, pred, len(LABEL2ID))
    return {
        "macro_f1": r["macro_f1"],
        "macro_f1_all13": r["macro_f1_all"],
        "macro_precision": r["macro_p"],
        "macro_recall": r["macro_r"],
        "accuracy": r["acc"],
    }

rows = []
class_rows = []
doc_rows = []
# Keep these in memory for Experiment 5.
EXP1_CACHE = {}

for dom in DOMAINS:
    print(f"\nPreparing {dom}...")
    train_docs = DOCS[dom]["train"]
    dev_docs = DOCS[dom]["dev"]

    examples = build_examples(
        dev_docs, ctx=True, k_left=3, k_right=1,
        t_budget=96, l_budget=160, r_budget=64, max_sent=96
    )
    y = np.asarray([e["y"] for e in examples], dtype=int)
    assert np.all(y >= 0), f"{dom}: missing dev labels"

    probs_by_model = {}
    for model in MODELS:
        logits = np.asarray(load_arr(model, dom, "dev_logits"))
        assert logits.shape == (len(examples), len(LABEL2ID)), (
            dom, model, logits.shape, len(examples)
        )
        assert np.isfinite(logits).all(), f"Non-finite logits: {dom}/{model}"
        probs_by_model[model] = exp1_softmax(logits)

    # Transitions estimated exclusively from training labels.
    cfg = POST_CFG[dom]
    logT, logS = transition_logprobs(
        train_docs, LABEL2ID, len(LABEL2ID), alpha=cfg["alpha"]
    )

    candidates = []

    # 1. Each standalone model.
    for model in MODELS:
        candidates.append((
            f"standalone::{model}",
            probs_by_model[model],
            {"kind": "standalone", "model": model},
        ))

    # 2. All pairwise ensembles, both methods and broad weight extremes.
    for model_a, model_b in combinations(MODELS, 2):
        pa, pb = probs_by_model[model_a], probs_by_model[model_b]
        for method in ["arithmetic", "geometric"]:
            for w in WEIGHTS:
                p = weighted_probs(
                    # weighted_probs expects logits, so use original logits
                    load_arr(model_a, dom, "dev_logits"),
                    load_arr(model_b, dom, "dev_logits"),
                    weight_a=w,
                    method=method,
                )
                candidates.append((
                    f"pair::{model_a}+{model_b}::{method}::{w:.2f}",
                    p,
                    {
                        "kind": "pair",
                        "model_a": model_a,
                        "model_b": model_b,
                        "method": method,
                        "weight_a": w,
                        "weight_b": 1.0 - w,
                    },
                ))

    # 3. Equal-weight arithmetic ensemble of all three models.
    p3 = sum(probs_by_model[m] for m in MODELS) / len(MODELS)
    p3 /= np.maximum(p3.sum(axis=1, keepdims=True), 1e-300)
    candidates.append((
        "three_model::arithmetic_equal",
        p3,
        {"kind": "three_model", "models": MODELS, "weights": [1/3]*3},
    ))

    # Evaluate raw and post-processed predictions with the same dev labels.
    for name, probs, metadata in candidates:
        raw_pred = probs.argmax(axis=1)
        post_pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                cfg["smooth"], cfg["bias"]
            ),
            dtype=int,
        )
        assert len(raw_pred) == len(y) == len(post_pred)

        for stage, pred in [("raw_argmax", raw_pred),
                            ("current_postprocessing", post_pred)]:
            score = exp1_score(y, pred)
            rows.append({
                "domain": dom,
                "candidate": name,
                "stage": stage,
                **metadata,
                **score,
            })

            report = prf_report(y, pred, len(LABEL2ID))
            for j, label in enumerate(LABEL2ID.keys()):
                class_rows.append({
                    "domain": dom,
                    "candidate": name,
                    "stage": stage,
                    "label": label,
                    "f1": float(report["f1"][j]),
                    "precision": float(report["prec"][j]),
                    "recall": float(report["rec"][j]),
                    "support": int(report["support"][j]),
                })

            for doc_id in dict.fromkeys(e["doc"] for e in examples):
                ix = np.asarray(
                    [i for i, e in enumerate(examples) if e["doc"] == doc_id],
                    dtype=int,
                )
                dr = exp1_score(y[ix], pred[ix])
                doc_rows.append({
                    "domain": dom,
                    "candidate": name,
                    "stage": stage,
                    "doc_id": str(doc_id),
                    "n_sentences": len(ix),
                    **dr,
                })

    EXP1_CACHE[dom] = {
        "examples": examples,
        "y": y,
        "probs": probs_by_model,
        "logT": logT,
        "logS": logS,
    }

results_exp1 = pd.DataFrame(rows)
classes_exp1 = pd.DataFrame(class_rows)
docs_exp1 = pd.DataFrame(doc_rows)

results_exp1.to_csv(EXP1 / "all_model_ensemble_scores.csv", index=False)
classes_exp1.to_csv(EXP1 / "per_class_scores.csv", index=False)
docs_exp1.to_csv(EXP1 / "per_document_scores.csv", index=False)

for dom in DOMAINS:
    print(f"\n{'='*20} {dom}: RAW TOP 12 {'='*20}")
    display(
        results_exp1[
            (results_exp1.domain == dom) &
            (results_exp1.stage == "raw_argmax")
        ].sort_values(
            ["macro_f1", "macro_f1_all13", "accuracy"],
            ascending=False,
        ).head(12).round(5)
    )

    print(f"\n{'='*20} {dom}: POST-PROCESSED TOP 12 {'='*20}")
    display(
        results_exp1[
            (results_exp1.domain == dom) &
            (results_exp1.stage == "current_postprocessing")
        ].sort_values(
            ["macro_f1", "macro_f1_all13", "accuracy"],
            ascending=False,
        ).head(12).round(5)
    )

print("\nSaved results under:", EXP1)
print("No checkpoints or existing submission files were changed.")


In [ ]:

# EXPERIMENT 5 — Error overlap and complementarity
# Requires Cell 1's EXP1_CACHE.
# Uses dev labels only; does not touch test predictions.

from itertools import combinations
import numpy as np
import pandas as pd

EXP5 = EXP1 / "error_overlap"
EXP5.mkdir(parents=True, exist_ok=True)

overlap_rows = []
class_rows = []
sentence_rows = []

for dom in DOMAINS:
    cache = EXP1_CACHE[dom]
    examples = cache["examples"]
    y = cache["y"]
    probs_by_model = cache["probs"]

    preds = {
        m: probs_by_model[m].argmax(axis=1)
        for m in MODELS
    }
    correct = {
        m: (preds[m] == y)
        for m in MODELS
    }

    # Pairwise overlap and rescue analysis.
    for model_a, model_b in combinations(MODELS, 2):
        ca, cb = correct[model_a], correct[model_b]
        pa, pb = preds[model_a], preds[model_b]
        both_correct = ca & cb
        a_only = ca & ~cb
        b_only = ~ca & cb
        both_wrong = ~ca & ~cb
        disagree = pa != pb

        overlap_rows.append({
            "domain": dom,
            "model_a": model_a,
            "model_b": model_b,
            "n_sentences": len(y),
            "a_accuracy": float(ca.mean()),
            "b_accuracy": float(cb.mean()),
            "both_correct": int(both_correct.sum()),
            "a_only_correct": int(a_only.sum()),
            "b_only_correct": int(b_only.sum()),
            "both_wrong": int(both_wrong.sum()),
            "disagree_count": int(disagree.sum()),
            "disagreement_rate": float(disagree.mean()),
            "b_rescue_rate_on_a_errors": (
                float(b_only.sum() / (~ca).sum()) if (~ca).sum() else np.nan
            ),
            "a_rescue_rate_on_b_errors": (
                float(a_only.sum() / (~cb).sum()) if (~cb).sum() else np.nan
            ),
        })

        # Preserve sentence-level disagreement details for inspection.
        for i in np.flatnonzero(disagree | a_only | b_only | both_wrong):
            sentence_rows.append({
                "domain": dom,
                "doc_id": str(examples[i]["doc"]),
                "sentence_index": int(examples[i]["i"]),
                "gold": list(LABEL2ID.keys())[int(y[i])],
                "model_a": model_a,
                "pred_a": list(LABEL2ID.keys())[int(pa[i])],
                "model_a_correct": bool(ca[i]),
                "model_b": model_b,
                "pred_b": list(LABEL2ID.keys())[int(pb[i])],
                "model_b_correct": bool(cb[i]),
                "predictions_disagree": bool(disagree[i]),
            })

    # Per-class recall/F1 and correct counts for each standalone model.
    for model in MODELS:
        report = prf_report(y, preds[model], len(LABEL2ID))
        for j, label in enumerate(LABEL2ID.keys()):
            class_rows.append({
                "domain": dom,
                "model": model,
                "label": label,
                "support": int(report["support"][j]),
                "correct_count": int(
                    ((y == j) & (preds[model] == j)).sum()
                ),
                "precision": float(report["prec"][j]),
                "recall": float(report["rec"][j]),
                "f1": float(report["f1"][j]),
            })

overlap_df = pd.DataFrame(overlap_rows)
class_df = pd.DataFrame(class_rows)
sentence_df = pd.DataFrame(sentence_rows)

overlap_df.to_csv(EXP5 / "pairwise_error_overlap.csv", index=False)
class_df.to_csv(EXP5 / "standalone_per_class.csv", index=False)
sentence_df.to_csv(EXP5 / "sentence_disagreements.csv", index=False)

print("=== Pairwise error overlap ===")
display(overlap_df.round(4))

print("=== Standalone model scores by class ===")
display(
    class_df.pivot_table(
        index=["domain", "label"],
        columns="model",
        values="f1",
    ).reset_index().round(4)
)

print("=== Where the non-KD student rescues E0a/KD errors ===")
for dom in DOMAINS:
    print(f"\n{dom}")
    subset = overlap_df[
        (overlap_df.domain == dom) &
        (
            overlap_df.model_a.isin(["E0a_full_sent", "E6_kd_student"]) &
            (overlap_df.model_b == "E6_student_nokd")
        )
    ]
    display(subset.round(4))

print("\nSaved error-overlap results under:", EXP5)


In [ ]:

# EXPERIMENT 3 — Learned vs uniform transition/start priors
# Requires EXP1_CACHE and helpers from the previous experiment cells.
# Does not change checkpoints or submission files.

from pathlib import Path
import numpy as np
import pandas as pd

EXP3 = Path("/kaggle/working/phase3_rr/experiments/exp3_transition_variants")
EXP3.mkdir(parents=True, exist_ok=True)

K = len(LABEL2ID)
LABEL_NAMES = list(LABEL2ID.keys())

# Freeze the strongest current pairwise candidates from Experiment 1.
TRANS_CFG = {
    "CL": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {"ArgumentRespondent": 1.0, "ArgumentPetitioner": 0.75},
    },
    "IT": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {"RulingByLowerCourt": 1.0, "RulingByPresentCourt": 1.0},
    },
}

transition_rows = []
transition_doc_rows = []

for dom in ["CL", "IT"]:
    cfg = TRANS_CFG[dom]
    cache = EXP1_CACHE[dom]
    examples = cache["examples"]
    y = cache["y"]

    # Reconstruct this domain's fixed ensemble.
    logits_a = load_arr(cfg["model_a"], dom, "dev_logits")
    logits_b = load_arr(cfg["model_b"], dom, "dev_logits")
    probs = weighted_probs(
        logits_a, logits_b,
        weight_a=cfg["weight_a"],
        method=cfg["method"],
    )

    # Learned transition/start priors: TRAIN ONLY.
    logT_learned, logS_learned = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, K, alpha=cfg["alpha"]
    )

    # Uniform log-probability priors.
    logT_uniform = np.full((K, K), -np.log(K), dtype=np.float64)
    logS_uniform = np.full(K, -np.log(K), dtype=np.float64)

    variants = {
        "learned_T_learned_start": (logT_learned, logS_learned),
        "uniform_T_learned_start": (logT_uniform, logS_learned),
        "learned_T_uniform_start": (logT_learned, logS_uniform),
        "uniform_T_uniform_start": (logT_uniform, logS_uniform),
    }

    for variant_name, (logT, logS) in variants.items():
        pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                cfg["smooth"], cfg["bias"]
            ),
            dtype=int,
        )
        score = exp1_score(y, pred)
        transition_rows.append({
            "domain": dom,
            "variant": variant_name,
            "smooth": cfg["smooth"],
            "alpha": cfg["alpha"],
            "ensemble_method": cfg["method"],
            "weight_a": cfg["weight_a"],
            **score,
        })

        for doc_id in dict.fromkeys(e["doc"] for e in examples):
            ix = np.asarray(
                [i for i, e in enumerate(examples) if e["doc"] == doc_id],
                dtype=int,
            )
            doc_score = exp1_score(y[ix], pred[ix])
            transition_doc_rows.append({
                "domain": dom,
                "variant": variant_name,
                "doc_id": str(doc_id),
                "n_sentences": len(ix),
                **doc_score,
            })

transition_df = pd.DataFrame(transition_rows)
transition_doc_df = pd.DataFrame(transition_doc_rows)

transition_df.to_csv(EXP3 / "transition_variant_scores.csv", index=False)
transition_doc_df.to_csv(EXP3 / "per_document_scores.csv", index=False)

print("=== Overall transition comparison ===")
display(
    transition_df.sort_values(
        ["domain", "macro_f1_all13"],
        ascending=[True, False],
    ).round(5)
)

print("=== Per-document robustness ===")
display(
    transition_doc_df.sort_values(["domain", "doc_id"]).round(5)
)

print("\nSaved to:", EXP3)


In [ ]:

# ============================================================
# EXPERIMENT 4 — TEMPERATURE SCALING
# Full fixed cell
#
# - No retraining
# - Transition statistics estimated from TRAIN documents only
# - Uses the current best ensemble configuration for each domain
# - Uses actual prf_report() dictionary keys
# - Does not overwrite the existing candidate submission
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score

EXP4_DIR = os.path.join(
    WORK, "experiments", "exp4_temperature"
)
os.makedirs(EXP4_DIR, exist_ok=True)

TEMP_CFG = {
    "CL": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

TEMPERATURES = [0.70, 0.85, 1.00, 1.15, 1.30, 1.50]


def temperature_scale_probs(probs, temperature):
    """Temperature-scale ensemble probabilities."""
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, 1e-12, 1.0)

    # T < 1 sharpens predictions; T > 1 softens them.
    log_probs = np.log(probs)
    scaled = softmax(log_probs / float(temperature), axis=1)

    return scaled


def calculate_metrics(gold, pred):
    """Return metrics using the same definitions as prf_report()."""
    gold = np.asarray(gold, dtype=int)
    pred = np.asarray(pred, dtype=int)

    report = prf_report(gold, pred, K)

    return {
        "macro_f1": report["macro_f1"],
        "macro_f1_all13": report["macro_f1_all"],
        "macro_precision": report["macro_p"],
        "macro_recall": report["macro_r"],
        "accuracy": report["acc"],
    }


overall_rows = []
doc_rows = []

for dom in ["CL", "IT"]:
    print(f"\nRunning temperature experiment for {dom}...")

    cfg = TEMP_CFG[dom]

    # Cached dev examples and flattened gold labels.
    examples = EXP1_CACHE[dom]["examples"]
    y = np.asarray(EXP1_CACHE[dom]["y"], dtype=int)

    # Load existing dev logits and reconstruct the selected ensemble.
    logits_a = load_arr(
        cfg["model_a"], dom, "dev_logits"
    )
    logits_b = load_arr(
        cfg["model_b"], dom, "dev_logits"
    )

    base_probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_a"],
        method=cfg["method"],
    )

    # Safety checks: ensure probability rows align with labels.
    assert base_probs.ndim == 2, (
        f"{dom}: expected 2D ensemble probabilities"
    )
    assert base_probs.shape[0] == len(y), (
        f"{dom}: {base_probs.shape[0]} probability rows "
        f"but {len(y)} gold labels"
    )
    assert base_probs.shape[1] == K, (
        f"{dom}: expected {K} label probabilities, "
        f"got {base_probs.shape[1]}"
    )

    # IMPORTANT: transition estimates come from TRAIN only.
    logT, logS = transition_logprobs(
        DOCS[dom]["train"],
        LABEL2ID,
        K,
        alpha=cfg["alpha"],
    )

    # Gold labels for each dev document, in document order.
    doc_gold = [
        np.asarray(
            [LABEL2ID[label] for label in doc["labels"]],
            dtype=int,
        )
        for doc in DOCS[dom]["dev"]
    ]

    # Confirm flattened document labels align with cached labels.
    flattened_doc_gold = (
        np.concatenate(doc_gold)
        if doc_gold
        else np.asarray([], dtype=int)
    )
    assert np.array_equal(flattened_doc_gold, y), (
        f"{dom}: DOCS dev labels do not match EXP1_CACHE order. "
        "Check document ordering before continuing."
    )

    for temp in TEMPERATURES:
        scaled_probs = temperature_scale_probs(
            base_probs, temp
        )

        pred = final_predict(
            scaled_probs,
            examples,
            logT,
            logS,
            cfg["smooth"],
            cfg["bias"],
        )
        pred = np.asarray(pred, dtype=int)

        assert len(pred) == len(y), (
            f"{dom}, T={temp}: prediction count mismatch"
        )

        # Overall dev metrics.
        metrics = calculate_metrics(y, pred)

        overall_rows.append({
            "domain": dom,
            "temperature": temp,
            **metrics,
            "ensemble_method": cfg["method"],
            "weight_a": cfg["weight_a"],
            "smooth": cfg["smooth"],
            "alpha": cfg["alpha"],
        })

        # Per-document robustness.
        offset = 0

        for doc_idx, gold in enumerate(doc_gold):
            n = len(gold)
            doc_pred = pred[offset:offset + n]

            doc_metrics = calculate_metrics(
                gold, doc_pred
            )

            doc_rows.append({
                "domain": dom,
                "temperature": temp,
                "doc_id": doc_idx,
                "n_sentences": n,
                **doc_metrics,
            })

            offset += n

        assert offset == len(pred), (
            f"{dom}: document lengths do not match predictions"
        )


# Build and save result tables.
overall_df = pd.DataFrame(overall_rows)
doc_df = pd.DataFrame(doc_rows)

# Sort each domain by all-13 macro-F1, then ordinary macro-F1.
overall_df = overall_df.sort_values(
    ["domain", "macro_f1_all13", "macro_f1"],
    ascending=[True, False, False],
).reset_index(drop=True)

doc_df = doc_df.sort_values(
    ["domain", "temperature", "doc_id"]
).reset_index(drop=True)

overall_path = os.path.join(
    EXP4_DIR, "temperature_scores.csv"
)
doc_path = os.path.join(
    EXP4_DIR, "temperature_per_document.csv"
)

overall_df.to_csv(overall_path, index=False)
doc_df.to_csv(doc_path, index=False)

print("\n" + "=" * 75)
print("OVERALL TEMPERATURE COMPARISON")
print("=" * 75)

display(
    overall_df.round(5)
)

print("\n" + "=" * 75)
print("PER-DOCUMENT ROBUSTNESS")
print("=" * 75)

display(
    doc_df.round(5)
)

# Show the T=1 control against the known Experiment 3 values.
print("\n" + "=" * 75)
print("T=1.0 CONTROL CHECK")
print("=" * 75)

control = overall_df[
    np.isclose(overall_df["temperature"], 1.0)
][[
    "domain",
    "macro_f1",
    "macro_f1_all13",
    "macro_precision",
    "macro_recall",
    "accuracy",
]]

display(control.round(5))

print("\nSaved results:")
print("Overall:", overall_path)
print("Per-document:", doc_path)
print("\nExisting candidate submission was not modified.")


In [ ]:

# ============================================================
# ROBUSTNESS CHECK 1: LEAVE-ONE-DOCUMENT-OUT (LODO)
#
# Evaluation-only exclusion:
# - Predictions are unchanged when a document is excluded.
# - No dev labels are used to estimate transitions.
# - No model retraining or parameter tuning.
# - Existing candidate submission is untouched.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

ROBUST_DIR = os.path.join(
    WORK, "experiments", "robustness_lodo"
)
os.makedirs(ROBUST_DIR, exist_ok=True)

ROBUST_CFG = {
    "CL": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}


def robustness_metrics(gold, pred):
    gold = np.asarray(gold, dtype=int)
    pred = np.asarray(pred, dtype=int)

    report = prf_report(gold, pred, K)

    return {
        "macro_f1": report["macro_f1"],
        "macro_f1_all13": report["macro_f1_all"],
        "macro_precision": report["macro_p"],
        "macro_recall": report["macro_r"],
        "accuracy": report["acc"],
    }


lodo_rows = []
document_rows = []
ROBUST_PREDICTIONS = {}

for dom in ["CL", "IT"]:
    cfg = ROBUST_CFG[dom]
    dev_docs = DOCS[dom]["dev"]

    # Development examples and labels in their original order.
    examples = EXP1_CACHE[dom]["examples"]
    y = np.asarray(EXP1_CACHE[dom]["y"], dtype=int)

    # Reconstruct the chosen ensemble.
    logits_a = load_arr(
        cfg["model_a"], dom, "dev_logits"
    )
    logits_b = load_arr(
        cfg["model_b"], dom, "dev_logits"
    )

    probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_a"],
        method=cfg["method"],
    )

    # Temperature is 1.0: no temperature transformation required.
    # Transitions estimated only from training documents.
    logT, logS = transition_logprobs(
        DOCS[dom]["train"],
        LABEL2ID,
        K,
        alpha=cfg["alpha"],
    )

    pred = np.asarray(
        final_predict(
            probs,
            examples,
            logT,
            logS,
            cfg["smooth"],
            cfg["bias"],
        ),
        dtype=int,
    )

    assert len(pred) == len(y), (
        f"{dom}: prediction/label count mismatch"
    )

    # Verify the development-document boundaries.
    gold_docs = [
        np.asarray(
            [LABEL2ID[label] for label in doc["labels"]],
            dtype=int,
        )
        for doc in dev_docs
    ]

    flattened_gold = np.concatenate(gold_docs)
    assert np.array_equal(flattened_gold, y), (
        f"{dom}: cached label order differs from DOCS dev order"
    )

    doc_lengths = [len(g) for g in gold_docs]
    assert sum(doc_lengths) == len(pred)

    # Create document slices into the unchanged flattened predictions.
    offsets = np.cumsum([0] + doc_lengths)
    pred_docs = [
        pred[offsets[i]:offsets[i + 1]]
        for i in range(len(doc_lengths))
    ]

    ROBUST_PREDICTIONS[dom] = {
        "gold_docs": gold_docs,
        "pred_docs": pred_docs,
        "lengths": doc_lengths,
    }

    # Full development-set score.
    full_metrics = robustness_metrics(y, pred)

    lodo_rows.append({
        "domain": dom,
        "evaluation": "FULL_DEV",
        "excluded_doc": "none",
        "excluded_sentences": 0,
        **full_metrics,
    })

    # Per-document scores.
    for i, (gold_doc, pred_doc) in enumerate(
        zip(gold_docs, pred_docs)
    ):
        metrics = robustness_metrics(gold_doc, pred_doc)

        document_rows.append({
            "domain": dom,
            "doc_id": i,
            "n_sentences": len(gold_doc),
            **metrics,
        })

    # Leave one document out of evaluation at a time.
    for excluded_idx in range(len(dev_docs)):
        kept_gold = [
            gold_docs[i]
            for i in range(len(dev_docs))
            if i != excluded_idx
        ]
        kept_pred = [
            pred_docs[i]
            for i in range(len(dev_docs))
            if i != excluded_idx
        ]

        y_kept = np.concatenate(kept_gold)
        pred_kept = np.concatenate(kept_pred)

        metrics = robustness_metrics(y_kept, pred_kept)

        lodo_rows.append({
            "domain": dom,
            "evaluation": "LODO",
            "excluded_doc": excluded_idx,
            "excluded_sentences": doc_lengths[excluded_idx],
            **metrics,
        })


lodo_df = pd.DataFrame(lodo_rows)
document_df = pd.DataFrame(document_rows)

lodo_path = os.path.join(
    ROBUST_DIR, "lodo_scores.csv"
)
document_path = os.path.join(
    ROBUST_DIR, "per_document_scores.csv"
)

lodo_df.to_csv(lodo_path, index=False)
document_df.to_csv(document_path, index=False)

print("=" * 75)
print("LEAVE-ONE-DOCUMENT-OUT RESULTS")
print("=" * 75)
display(
    lodo_df.round(5)
)

print("\n" + "=" * 75)
print("PER-DOCUMENT SCORES")
print("=" * 75)
display(
    document_df.round(5)
)

# Summarize score variation when each document is excluded.
summary_rows = []

for dom in ["CL", "IT"]:
    subset = lodo_df[
        (lodo_df["domain"] == dom)
        & (lodo_df["evaluation"] == "LODO")
    ]

    for metric in [
        "macro_f1",
        "macro_f1_all13",
        "accuracy",
    ]:
        vals = subset[metric].to_numpy()

        summary_rows.append({
            "domain": dom,
            "metric": metric,
            "LODO_min": vals.min(),
            "LODO_max": vals.max(),
            "LODO_mean": vals.mean(),
            "LODO_std": vals.std(ddof=0),
            "range": vals.max() - vals.min(),
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(
    ROBUST_DIR, "lodo_variability_summary.csv"
)
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 75)
print("LODO SCORE VARIABILITY")
print("=" * 75)
display(summary_df.round(5))

print("\nSaved results to:", ROBUST_DIR)


In [ ]:

# ============================================================
# ROBUSTNESS CHECK 2: PER-CLASS AND PER-DOCUMENT ERROR PROFILE
#
# Requires Cell 1 to have run successfully.
# Uses the exact same predictions; no tuning or retraining.
# ============================================================

from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
)

class_rows = []
pooled_class_rows = []
confusion_rows = []

for dom in ["CL", "IT"]:
    gold_docs = ROBUST_PREDICTIONS[dom]["gold_docs"]
    pred_docs = ROBUST_PREDICTIONS[dom]["pred_docs"]

    all_gold = np.concatenate(gold_docs)
    all_pred = np.concatenate(pred_docs)

    # Pooled per-class performance for the full domain dev set.
    p, r, f1, support = precision_recall_fscore_support(
        all_gold,
        all_pred,
        labels=list(range(K)),
        zero_division=0,
    )

    for label_id in range(K):
        pooled_class_rows.append({
            "domain": dom,
            "label_id": label_id,
            "label": LABELS[label_id],
            "support": int(support[label_id]),
            "precision": p[label_id],
            "recall": r[label_id],
            "f1": f1[label_id],
        })

    # Per-document, per-class metrics.
    for doc_idx, (gold, pred) in enumerate(
        zip(gold_docs, pred_docs)
    ):
        p_doc, r_doc, f1_doc, support_doc = (
            precision_recall_fscore_support(
                gold,
                pred,
                labels=list(range(K)),
                zero_division=0,
            )
        )

        for label_id in range(K):
            class_rows.append({
                "domain": dom,
                "doc_id": doc_idx,
                "n_sentences": len(gold),
                "label_id": label_id,
                "label": LABELS[label_id],
                "support": int(support_doc[label_id]),
                "precision": p_doc[label_id],
                "recall": r_doc[label_id],
                "f1": f1_doc[label_id],
            })

        cm = confusion_matrix(
            gold,
            pred,
            labels=list(range(K)),
        )

        for true_id in range(K):
            for pred_id in range(K):
                count = int(cm[true_id, pred_id])

                if count > 0:
                    confusion_rows.append({
                        "domain": dom,
                        "doc_id": doc_idx,
                        "true_label": LABELS[true_id],
                        "predicted_label": LABELS[pred_id],
                        "count": count,
                    })


class_df = pd.DataFrame(class_rows)
pooled_class_df = pd.DataFrame(pooled_class_rows)
confusion_df = pd.DataFrame(confusion_rows)

class_path = os.path.join(
    ROBUST_DIR, "per_document_per_class.csv"
)
pooled_path = os.path.join(
    ROBUST_DIR, "pooled_per_class.csv"
)
confusion_path = os.path.join(
    ROBUST_DIR, "nonzero_confusions.csv"
)

class_df.to_csv(class_path, index=False)
pooled_class_df.to_csv(pooled_path, index=False)
confusion_df.to_csv(confusion_path, index=False)

print("=" * 75)
print("POOLED PER-CLASS PERFORMANCE")
print("=" * 75)

display(
    pooled_class_df.sort_values(
        ["domain", "f1"],
        ascending=[True, True],
    ).round(4)
)

print("\n" + "=" * 75)
print("LOWEST-F1 CLASSES WITH NONZERO GOLD SUPPORT")
print("=" * 75)

weak_classes = pooled_class_df[
    pooled_class_df["support"] > 0
].sort_values(
    ["domain", "f1"],
    ascending=[True, True],
).groupby("domain", group_keys=False).head(6)

display(weak_classes.round(4))

print("\n" + "=" * 75)
print("MOST COMMON CONFUSIONS")
print("=" * 75)

if len(confusion_df):
    top_confusions = (
        confusion_df
        .groupby(
            ["domain", "true_label", "predicted_label"],
            as_index=False,
        )["count"]
        .sum()
        .sort_values(
            ["domain", "count"],
            ascending=[True, False],
        )
        .groupby("domain", group_keys=False)
        .head(15)
    )
    display(top_confusions)
else:
    print("No off-diagonal confusions found.")

print("\nSaved files:")
print(class_path)
print(pooled_path)
print(confusion_path)


In [ ]:

# ============================================================
# NEXT EXPERIMENT: TARGETED ERROR INSPECTION
#
# Requires Cell 1 (ROBUST_PREDICTIONS) and the existing helpers.
# Does not retrain, tune, or modify submissions.
# ============================================================

import os
import pandas as pd
import numpy as np

ERROR_DIR = os.path.join(
    WORK, "experiments", "targeted_error_inspection"
)
os.makedirs(ERROR_DIR, exist_ok=True)

TARGET_PAIRS = {
    "CL": [
        ("ArgumentRespondent", "RatioOfTheDecision"),
        ("ArgumentPetitioner", "ArgumentRespondent"),
        ("RulingByLowerCourt", "PrecedentReliedUpon"),
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
    ],
    "IT": [
        ("ArgumentRespondent", "ArgumentPetitioner"),
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
        ("PrecedentNotReliedUpon", "PrecedentReliedUpon"),
        ("RulingByLowerCourt", "PrecedentReliedUpon"),
        ("None", "Fact"),
    ],
}

# Locate sentence text in the dev documents without assuming a single
# schema for every dataset representation.
def get_sentence_text(doc, i):
    for key in [
        "sentences", "text", "texts",
        "sents", "sentence_texts"
    ]:
        value = doc.get(key) if isinstance(doc, dict) else None

        if isinstance(value, (list, tuple)) and i < len(value):
            item = value[i]

            if isinstance(item, str):
                return item

            if isinstance(item, dict):
                for text_key in ["text", "sentence", "sent"]:
                    if text_key in item:
                        return str(item[text_key])

        if isinstance(value, str):
            return value

    return "[Sentence text field not found; inspect DOCS schema]"


error_rows = []

for dom in ["CL", "IT"]:
    gold_docs = ROBUST_PREDICTIONS[dom]["gold_docs"]
    pred_docs = ROBUST_PREDICTIONS[dom]["pred_docs"]
    dev_docs = DOCS[dom]["dev"]

    wanted = {
        (LABEL2ID[a], LABEL2ID[b])
        for a, b in TARGET_PAIRS[dom]
    }

    for doc_idx, (gold, pred) in enumerate(
        zip(gold_docs, pred_docs)
    ):
        for sent_idx, (g, p) in enumerate(zip(gold, pred)):
            if g == p or (int(g), int(p)) not in wanted:
                continue

            error_rows.append({
                "domain": dom,
                "doc_id": doc_idx,
                "sentence_idx": sent_idx,
                "true_label": LABELS[int(g)],
                "predicted_label": LABELS[int(p)],
                "sentence": get_sentence_text(
                    dev_docs[doc_idx], sent_idx
                ),
            })

errors_df = pd.DataFrame(error_rows)

if len(errors_df):
    errors_df = errors_df.sort_values(
        ["domain", "true_label", "predicted_label", "doc_id"]
    )

errors_path = os.path.join(
    ERROR_DIR, "targeted_errors.csv"
)
errors_df.to_csv(errors_path, index=False)

print("Targeted errors found:", len(errors_df))
print("Saved:", errors_path)

display(errors_df.head(100))


In [ ]:

# ============================================================
# TARGETED ERROR ANALYSIS — AGGREGATES AND CLUSTERS
# Requires: errors_df, ROBUST_PREDICTIONS, LABELS, LABEL2ID
# ============================================================

import numpy as np
import pandas as pd

# 1. Most frequent true-label -> predicted-label confusions
confusion_summary = (
    errors_df
    .groupby(
        ["domain", "true_label", "predicted_label"],
        as_index=False
    )
    .agg(
        errors=("sentence_idx", "size"),
        documents=("doc_id", "nunique"),
    )
    .sort_values(
        ["domain", "errors"],
        ascending=[True, False]
    )
)

print("=== TOP CONFUSIONS ===")
display(
    confusion_summary.groupby("domain", group_keys=False).head(12)
)

# 2. Which documents contain the targeted errors?
document_summary = (
    errors_df
    .groupby(["domain", "doc_id"], as_index=False)
    .agg(
        targeted_errors=("sentence_idx", "size"),
        distinct_confusions=("predicted_label", "nunique"),
    )
    .sort_values(
        ["domain", "targeted_errors"],
        ascending=[True, False]
    )
)

print("\n=== TARGETED ERRORS BY DOCUMENT ===")
display(document_summary)

# 3. Check consecutive error runs within each document.
# This can reveal long stretches of the same confusion.
run_rows = []

for (dom, doc_id), group in errors_df.groupby(["domain", "doc_id"]):
    group = group.sort_values("sentence_idx")
    records = group.to_dict("records")

    if not records:
        continue

    current = [records[0]]

    for rec in records[1:]:
        prev = current[-1]

        same_pair = (
            rec["true_label"] == prev["true_label"]
            and rec["predicted_label"] == prev["predicted_label"]
        )
        adjacent = rec["sentence_idx"] == prev["sentence_idx"] + 1

        if same_pair and adjacent:
            current.append(rec)
        else:
            if len(current) >= 3:
                run_rows.append({
                    "domain": dom,
                    "doc_id": doc_id,
                    "start_sentence": current[0]["sentence_idx"],
                    "end_sentence": current[-1]["sentence_idx"],
                    "run_length": len(current),
                    "true_label": current[0]["true_label"],
                    "predicted_label": current[0]["predicted_label"],
                    "first_sentence": current[0]["sentence"],
                })
            current = [rec]

    if len(current) >= 3:
        run_rows.append({
            "domain": dom,
            "doc_id": doc_id,
            "start_sentence": current[0]["sentence_idx"],
            "end_sentence": current[-1]["sentence_idx"],
            "run_length": len(current),
            "true_label": current[0]["true_label"],
            "predicted_label": current[0]["predicted_label"],
            "first_sentence": current[0]["sentence"],
        })

runs_df = pd.DataFrame(run_rows)

print("\n=== CONSECUTIVE ERROR RUNS (LENGTH >= 3) ===")
if len(runs_df):
    runs_df = runs_df.sort_values(
        ["run_length"], ascending=False
    )
    display(runs_df.head(30))
else:
    print("No consecutive runs of length >= 3 found.")

# 4. Save aggregates for later reference.
if "ERROR_DIR" in globals():
    confusion_summary.to_csv(
        os.path.join(ERROR_DIR, "confusion_summary.csv"),
        index=False
    )
    document_summary.to_csv(
        os.path.join(ERROR_DIR, "document_error_summary.csv"),
        index=False
    )
    runs_df.to_csv(
        os.path.join(ERROR_DIR, "consecutive_error_runs.csv"),
        index=False
    )

print("\nAnalysis complete.")


In [ ]:

# ============================================================
# REPRESENTATIVE ERROR EXAMPLES
# Requires: errors_df, confusion_summary
# ============================================================

representatives = []

for dom in ["CL", "IT"]:
    top_pairs = (
        confusion_summary[confusion_summary["domain"] == dom]
        .head(5)
    )

    for _, pair in top_pairs.iterrows():
        subset = errors_df[
            (errors_df["domain"] == dom)
            & (errors_df["true_label"] == pair["true_label"])
            & (errors_df["predicted_label"] == pair["predicted_label"])
        ].sort_values(["doc_id", "sentence_idx"])

        # Prefer examples from different documents.
        selected = []
        seen_docs = set()

        for _, row in subset.iterrows():
            if row["doc_id"] not in seen_docs:
                selected.append(row)
                seen_docs.add(row["doc_id"])
            if len(selected) == 2:
                break

        # If only one document has examples, add another from it.
        if len(selected) < 2:
            for _, row in subset.iterrows():
                if not any(
                    row["doc_id"] == s["doc_id"]
                    and row["sentence_idx"] == s["sentence_idx"]
                    for s in selected
                ):
                    selected.append(row)
                if len(selected) == 2:
                    break

        for row in selected:
            representatives.append({
                "domain": row["domain"],
                "true_label": row["true_label"],
                "predicted_label": row["predicted_label"],
                "doc_id": row["doc_id"],
                "sentence_idx": row["sentence_idx"],
                "sentence": row["sentence"],
            })

representatives_df = pd.DataFrame(representatives)

print("=== REPRESENTATIVE EXAMPLES ===")
display(representatives_df)

representatives_path = os.path.join(
    ERROR_DIR, "representative_errors.csv"
)
representatives_df.to_csv(representatives_path, index=False)
print("Saved:", representatives_path)


In [ ]:

# ============================================================
# EXPERIMENT 6: RAW vs BIAS-ONLY vs FULL SEQUENCE DECODER
#
# Requires:
#   WORK, DOCS, LABEL2ID, LABELS, K
#   EXP1_CACHE, load_arr, weighted_probs
#   transition_logprobs, final_predict, prf_report
#
# No training, no test labels, no submission modification.
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy.special import softmax

DECODER_DIR = os.path.join(
    WORK, "experiments", "decoder_contribution"
)
os.makedirs(DECODER_DIR, exist_ok=True)

DECODER_CFG = {
    "CL": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

# Confusions to investigate, grounded in the previous error analysis.
FOCUS_PAIRS = {
    "CL": [
        ("ArgumentRespondent", "RatioOfTheDecision"),
        ("ArgumentPetitioner", "ArgumentRespondent"),
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
    ],
    "IT": [
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
        ("PrecedentNotReliedUpon", "PrecedentReliedUpon"),
        ("ArgumentRespondent", "ArgumentPetitioner"),
        ("RulingByLowerCourt", "PrecedentReliedUpon"),
    ],
}


def add_class_bias_to_probs(probs, bias):
    """Apply the same additive class bias in log-probability space."""
    logp = np.log(np.clip(probs, 1e-12, 1.0)).copy()

    for label, amount in bias.items():
        class_id = LABEL2ID[label]
        logp[:, class_id] += float(amount)

    return softmax(logp, axis=1)


def metric_row(dom, stage, gold, pred):
    report = prf_report(
        np.asarray(gold, dtype=int),
        np.asarray(pred, dtype=int),
        K,
    )

    return {
        "domain": dom,
        "stage": stage,
        "macro_f1": report["macro_f1"],
        "macro_f1_all13": report["macro_f1_all"],
        "macro_precision": report["macro_p"],
        "macro_recall": report["macro_r"],
        "accuracy": report["acc"],
    }


stage_rows = []
pair_rows = []
document_stage_rows = []
decoder_predictions = {}

for dom in ["CL", "IT"]:
    cfg = DECODER_CFG[dom]
    examples = EXP1_CACHE[dom]["examples"]
    gold = np.asarray(EXP1_CACHE[dom]["y"], dtype=int)

    logits_a = load_arr(
        cfg["model_a"], dom, "dev_logits"
    )
    logits_b = load_arr(
        cfg["model_b"], dom, "dev_logits"
    )

    probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_a"],
        method=cfg["method"],
    )

    # Stage 1: raw ensemble predictions.
    raw_pred = np.argmax(probs, axis=1)

    # Stage 2: class bias without sequence decoding.
    biased_probs = add_class_bias_to_probs(
        probs, cfg["bias"]
    )
    bias_pred = np.argmax(biased_probs, axis=1)

    # Stage 3: current complete decoder.
    logT, logS = transition_logprobs(
        DOCS[dom]["train"],
        LABEL2ID,
        K,
        alpha=cfg["alpha"],
    )

    full_pred = np.asarray(
        final_predict(
            probs,
            examples,
            logT,
            logS,
            cfg["smooth"],
            cfg["bias"],
        ),
        dtype=int,
    )

    stages = {
        "raw_ensemble": raw_pred,
        "bias_only": bias_pred,
        "full_decoder": full_pred,
    }

    decoder_predictions[dom] = stages

    for stage_name, pred in stages.items():
        stage_rows.append(
            metric_row(dom, stage_name, gold, pred)
        )

    # Document-level scores at each stage.
    gold_docs = ROBUST_PREDICTIONS[dom]["gold_docs"]
    lengths = [len(x) for x in gold_docs]
    offsets = np.cumsum([0] + lengths)

    for stage_name, pred in stages.items():
        for doc_idx, gold_doc in enumerate(gold_docs):
            p_doc = pred[offsets[doc_idx]:offsets[doc_idx + 1]]
            row = metric_row(dom, stage_name, gold_doc, p_doc)

            document_stage_rows.append({
                "domain": dom,
                "doc_id": doc_idx,
                "n_sentences": len(gold_doc),
                **{
                    key: value
                    for key, value in row.items()
                    if key not in ["domain", "stage"]
                },
                "stage": stage_name,
            })

    # For each target confusion, count how each stage handles
    # the same gold examples.
    for true_label, wrong_label in FOCUS_PAIRS[dom]:
        true_id = LABEL2ID[true_label]
        wrong_id = LABEL2ID[wrong_label]
        mask = gold == true_id

        for stage_name, pred in stages.items():
            pair_rows.append({
                "domain": dom,
                "true_label": true_label,
                "confused_label": wrong_label,
                "stage": stage_name,
                "gold_support": int(mask.sum()),
                "predicted_as_confused_label": int(
                    np.sum(mask & (pred == wrong_id))
                ),
                "correct_for_true_label": int(
                    np.sum(mask & (pred == true_id))
                ),
                "predicted_as_any_other_label": int(
                    np.sum(mask & (pred != true_id) & (pred != wrong_id))
                ),
            })


stage_df = pd.DataFrame(stage_rows)
pair_df = pd.DataFrame(pair_rows)
doc_stage_df = pd.DataFrame(document_stage_rows)

stage_df.to_csv(
    os.path.join(DECODER_DIR, "overall_stage_scores.csv"),
    index=False,
)
pair_df.to_csv(
    os.path.join(DECODER_DIR, "target_confusions_by_stage.csv"),
    index=False,
)
doc_stage_df.to_csv(
    os.path.join(DECODER_DIR, "document_stage_scores.csv"),
    index=False,
)

print("=" * 75)
print("OVERALL METRICS BY DECODING STAGE")
print("=" * 75)
display(stage_df.round(5))

print("\n" + "=" * 75)
print("TARGET CONFUSIONS BY DECODING STAGE")
print("=" * 75)
display(
    pair_df.sort_values(
        ["domain", "true_label", "stage"]
    )
)

print("\n" + "=" * 75)
print("PER-DOCUMENT METRICS BY DECODING STAGE")
print("=" * 75)
display(
    doc_stage_df.sort_values(
        ["domain", "doc_id", "stage"]
    ).round(5)
)

print("\nSaved results to:", DECODER_DIR)


In [ ]:

# ============================================================
# EXPERIMENT 7: LOCAL SMOOTHING ABLATION
# Fixed ensemble, fixed bias, fixed transition alpha.
# Compare selected smoothing vs nearby weaker/stronger values.
# ============================================================

import os
import numpy as np
import pandas as pd

ABL_DIR = os.path.join(
    WORK, "experiments", "local_smoothing_ablation"
)
os.makedirs(ABL_DIR, exist_ok=True)

ABL_CFG = {
    "CL": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.55,
        "method": "arithmetic",
        "current_smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
        "smooth_values": [0.35, 0.50, 0.65],
    },
    "IT": {
        "model_a": "E0a_full_sent",
        "model_b": "E6_kd_student",
        "weight_a": 0.50,
        "method": "geometric",
        "current_smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
        "smooth_values": [0.10, 0.25, 0.40],
    },
}

ABL_FOCUS = {
    "CL": [
        ("ArgumentRespondent", "RatioOfTheDecision"),
        ("ArgumentPetitioner", "ArgumentRespondent"),
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
    ],
    "IT": [
        ("PrecedentReliedUpon", "RatioOfTheDecision"),
        ("PrecedentNotReliedUpon", "PrecedentReliedUpon"),
        ("ArgumentRespondent", "ArgumentPetitioner"),
        ("RulingByLowerCourt", "PrecedentReliedUpon"),
    ],
}

abl_overall = []
abl_confusions = []
abl_docs = []

for dom in ["CL", "IT"]:
    cfg = ABL_CFG[dom]
    examples = EXP1_CACHE[dom]["examples"]
    gold = np.asarray(EXP1_CACHE[dom]["y"], dtype=int)

    logits_a = load_arr(
        cfg["model_a"], dom, "dev_logits"
    )
    logits_b = load_arr(
        cfg["model_b"], dom, "dev_logits"
    )

    probs = weighted_probs(
        logits_a,
        logits_b,
        weight_a=cfg["weight_a"],
        method=cfg["method"],
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"],
        LABEL2ID,
        K,
        alpha=cfg["alpha"],
    )

    gold_docs = ROBUST_PREDICTIONS[dom]["gold_docs"]
    lengths = [len(x) for x in gold_docs]
    offsets = np.cumsum([0] + lengths)

    for smooth in cfg["smooth_values"]:
        pred = np.asarray(
            final_predict(
                probs,
                examples,
                logT,
                logS,
                smooth,
                cfg["bias"],
            ),
            dtype=int,
        )

        report = prf_report(gold, pred, K)

        abl_overall.append({
            "domain": dom,
            "smooth": smooth,
            "is_current": np.isclose(
                smooth, cfg["current_smooth"]
            ),
            "macro_f1": report["macro_f1"],
            "macro_f1_all13": report["macro_f1_all"],
            "macro_precision": report["macro_p"],
            "macro_recall": report["macro_r"],
            "accuracy": report["acc"],
        })

        # Targeted confusion counts.
        for true_label, confused_label in ABL_FOCUS[dom]:
            true_id = LABEL2ID[true_label]
            confused_id = LABEL2ID[confused_label]
            mask = gold == true_id

            abl_confusions.append({
                "domain": dom,
                "smooth": smooth,
                "true_label": true_label,
                "confused_label": confused_label,
                "gold_support": int(mask.sum()),
                "predicted_as_confused": int(
                    np.sum(mask & (pred == confused_id))
                ),
                "correct": int(
                    np.sum(mask & (pred == true_id))
                ),
            })

        # Document-level stability.
        for doc_idx, gold_doc in enumerate(gold_docs):
            pred_doc = pred[offsets[doc_idx]:offsets[doc_idx + 1]]
            doc_report = prf_report(gold_doc, pred_doc, K)

            abl_docs.append({
                "domain": dom,
                "smooth": smooth,
                "doc_id": doc_idx,
                "n_sentences": len(gold_doc),
                "macro_f1": doc_report["macro_f1"],
                "macro_f1_all13": doc_report["macro_f1_all"],
                "accuracy": doc_report["acc"],
            })


abl_overall_df = pd.DataFrame(abl_overall)
abl_confusions_df = pd.DataFrame(abl_confusions)
abl_docs_df = pd.DataFrame(abl_docs)

abl_overall_df.to_csv(
    os.path.join(ABL_DIR, "overall.csv"), index=False
)
abl_confusions_df.to_csv(
    os.path.join(ABL_DIR, "targeted_confusions.csv"), index=False
)
abl_docs_df.to_csv(
    os.path.join(ABL_DIR, "per_document.csv"), index=False
)

print("=" * 75)
print("OVERALL SMOOTHING ABLATION")
print("=" * 75)
display(
    abl_overall_df.sort_values(
        ["domain", "macro_f1_all13"],
        ascending=[True, False]
    ).round(5)
)

print("\n" + "=" * 75)
print("TARGETED CONFUSIONS")
print("=" * 75)
display(
    abl_confusions_df.sort_values(
        ["domain", "true_label", "smooth"]
    )
)

print("\n" + "=" * 75)
print("PER-DOCUMENT SCORES")
print("=" * 75)
display(
    abl_docs_df.sort_values(
        ["domain", "smooth", "doc_id"]
    ).round(5)
)

print("\nSaved to:", ABL_DIR)


In [ ]:
from pathlib import Path
import os, glob, zipfile, hashlib, json

WORK = Path("/kaggle/working/phase3_rr")
CANDIDATE_DIR = WORK / "final_candidate_20261014"
CANDIDATE_ZIP = CANDIDATE_DIR / "submission_candidate.zip"

print("WORK exists:", WORK.exists())
print("Candidate directory exists:", CANDIDATE_DIR.exists())
print("Candidate ZIP exists:", CANDIDATE_ZIP.exists())

if not CANDIDATE_ZIP.exists():
    matches = list(WORK.rglob("submission_candidate.zip"))
    if len(matches) == 1:
        CANDIDATE_ZIP = matches[0]
    else:
        print("Candidate ZIP matches:", matches)
        raise FileNotFoundError(
            "Could not uniquely locate submission_candidate.zip. "
            "Set CANDIDATE_ZIP to the exact existing ZIP path."
        )

print("\nCandidate:", CANDIDATE_ZIP)
print("Size (bytes):", CANDIDATE_ZIP.stat().st_size)

# Display available previous submissions without modifying them.
print("\nPossible previous submissions:")
for p in sorted(Path("/mnt/data").glob("scoring_result*.zip")):
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

In [ ]:
assert CANDIDATE_ZIP.exists(), f"Missing candidate: {CANDIDATE_ZIP}"
assert CANDIDATE_ZIP.stat().st_size > 0, "Candidate ZIP is empty."

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    bad_member = zf.testzip()
    members = zf.namelist()

    print("ZIP integrity:", "PASS" if bad_member is None else f"FAIL: {bad_member}")
    print("Files inside ZIP:")
    for info in zf.infolist():
        if not info.is_dir():
            print(f"  {info.filename} — {info.file_size:,} bytes")

    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")

assert members, "The ZIP contains no files."
print("\nArchive validation passed.")

In [ ]:
from collections import Counter

def describe_json(obj, depth=0, max_depth=3):
    indent = "  " * depth

    if isinstance(obj, dict):
        print(f"{indent}dict with {len(obj)} keys")
        for key, value in list(obj.items())[:12]:
            print(f"{indent}  key={key!r}, type={type(value).__name__}")
            if depth < max_depth and isinstance(value, (dict, list)):
                describe_json(value, depth + 1, max_depth)
            elif not isinstance(value, (dict, list)):
                print(f"{indent}    sample={repr(value)[:180]}")

    elif isinstance(obj, list):
        print(f"{indent}list with {len(obj)} items")
        if obj:
            print(f"{indent}  first item:")
            describe_json(obj[0], depth + 1, max_depth)
            if len(obj) > 1:
                print(f"{indent}  second item type={type(obj[1]).__name__}")
    else:
        print(f"{indent}{type(obj).__name__}: {repr(obj)[:180]}")

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    json_members = [
        n for n in zf.namelist()
        if n.lower().endswith(".json") and not n.endswith("/")
    ]

    print("JSON files found:", json_members)

    if not json_members:
        raise RuntimeError("No .json files found inside candidate ZIP.")

    for name in json_members:
        print("\n" + "=" * 70)
        print("FILE:", name)
        raw = zf.read(name)
        try:
            parsed = json.loads(raw.decode("utf-8-sig"))
        except Exception as e:
            print("Could not parse as UTF-8 JSON:", repr(e))
            continue

        describe_json(parsed)

In [ ]:
# Use the notebook's existing label list if present.
EXPECTED_LABELS = globals().get("LABELS", [
    "Fact",
    "Issue",
    "ArgumentPetitioner",
    "ArgumentRespondent",
    "Statute",
    "Dissent",
    "PrecedentReliedUpon",
    "PrecedentNotReliedUpon",
    "PrecedentOverruled",
    "RulingByLowerCourt",
    "RatioOfTheDecision",
    "RulingByPresentCourt",
    "None",
])
EXPECTED_LABEL_SET = set(EXPECTED_LABELS)

PREDICTION_KEYS = {
    "label", "labels", "prediction", "predictions",
    "predicted_label", "predicted_labels", "pred",
}

def find_prediction_fields(obj, path="$"):
    found = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            child_path = f"{path}.{key}"
            if key.lower() in PREDICTION_KEYS:
                found.append((child_path, value))
            found.extend(find_prediction_fields(value, child_path))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            found.extend(find_prediction_fields(value, f"{path}[{i}]"))

    return found

def flatten_values(value):
    if isinstance(value, list):
        out = []
        for item in value:
            out.extend(flatten_values(item))
        return out
    if isinstance(value, dict):
        return []
    return [value]

all_checks_found = 0

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    for name in json_members:
        if name not in zf.namelist():
            continue

        try:
            obj = json.loads(zf.read(name).decode("utf-8-sig"))
        except Exception:
            continue

        fields = find_prediction_fields(obj)
        print(f"\n{name}: {len(fields)} likely prediction field(s)")

        for field_path, value in fields:
            # Ignore object-valued fields; they may be metadata.
            values = flatten_values(value)
            string_values = [v for v in values if isinstance(v, str)]

            if not string_values:
                print(f"  {field_path}: no string labels to check")
                continue

            all_checks_found += 1
            counts = Counter(string_values)
            unknown = sorted(set(string_values) - EXPECTED_LABEL_SET)

            print(f"  Field: {field_path}")
            print(f"  String values: {len(string_values):,}")
            print(f"  Distinct values: {len(counts)}")
            print(f"  Top values: {counts.most_common(10)}")
            print(f"  Unknown labels: {unknown if unknown else 'None'}")

            if unknown:
                raise ValueError(
                    f"Unexpected label value(s) in {name} at {field_path}: {unknown}"
                )

if all_checks_found == 0:
    print(
        "\nNo standard string-valued prediction fields were detected. "
        "Use Cell 3's schema preview to identify the actual prediction field "
        "before treating label validation as complete."
    )
else:
    print("\nAll detected string-valued prediction fields use expected label names.")

In [ ]:
import shutil
import hashlib
from datetime import datetime

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_ZIP = CANDIDATE_DIR / f"submission_candidate_backup_{timestamp}.zip"

# Avoid overwriting even in the unlikely event of a filename collision.
suffix = 1
while BACKUP_ZIP.exists():
    BACKUP_ZIP = CANDIDATE_DIR / f"submission_candidate_backup_{timestamp}_{suffix}.zip"
    suffix += 1

original_hash = sha256_file(CANDIDATE_ZIP)
shutil.copy2(CANDIDATE_ZIP, BACKUP_ZIP)
backup_hash = sha256_file(BACKUP_ZIP)

print("Original:", CANDIDATE_ZIP)
print("Backup:  ", BACKUP_ZIP)
print("Original SHA-256:", original_hash)
print("Backup SHA-256:  ", backup_hash)
print("Byte-identical:", original_hash == backup_hash)

assert original_hash == backup_hash, "Backup checksum mismatch!"
print("\nBackup verified successfully.")

In [ ]:
import zipfile

assert CANDIDATE_ZIP.exists()
assert CANDIDATE_ZIP.stat().st_size > 0

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    assert zf.testzip() is None, "ZIP integrity check failed."
    files = [n for n in zf.namelist() if not n.endswith("/")]
    assert files, "ZIP contains no files."

print("FINAL ARTIFACT CHECK")
print("--------------------")
print("Candidate:", CANDIDATE_ZIP)
print("Size:", f"{CANDIDATE_ZIP.stat().st_size:,} bytes")
print("SHA-256:", sha256_file(CANDIDATE_ZIP))
print("ZIP integrity: PASS")
print("Contains files: PASS")
print("Original candidate modified: NO")
print("Test labels used: NO")

In [ ]:
import json
import zipfile
from collections import Counter

EXPECTED_LABELS = globals().get("LABELS", [
    "Fact",
    "Issue",
    "ArgumentPetitioner",
    "ArgumentRespondent",
    "Statute",
    "Dissent",
    "PrecedentReliedUpon",
    "PrecedentNotReliedUpon",
    "PrecedentOverruled",
    "RulingByLowerCourt",
    "RatioOfTheDecision",
    "RulingByPresentCourt",
    "None",
])
VALID_LABELS = set(EXPECTED_LABELS)

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    prediction = json.loads(
        zf.read("prediction.json").decode("utf-8-sig")
    )

assert isinstance(prediction, dict), "Expected a dictionary keyed by document ID."
assert len(prediction) > 0, "No documents found in prediction.json."

errors = []
total_predictions = 0
label_counts = Counter()

for doc_id, labels in prediction.items():
    if not isinstance(doc_id, str):
        errors.append(f"Non-string document ID: {doc_id!r}")

    if not isinstance(labels, list):
        errors.append(f"{doc_id}: expected a list of labels.")
        continue

    if not labels:
        errors.append(f"{doc_id}: empty prediction list.")
        continue

    for i, label in enumerate(labels):
        total_predictions += 1

        if not isinstance(label, str):
            errors.append(
                f"{doc_id}, sentence {i}: non-string label {label!r}"
            )
        elif label not in VALID_LABELS:
            errors.append(
                f"{doc_id}, sentence {i}: unknown label {label!r}"
            )
        else:
            label_counts[label] += 1

print("Documents:", len(prediction))
print("Total sentence predictions:", total_predictions)
print("Expected labels:", len(VALID_LABELS))
print("Observed labels:", len(label_counts))
print("Label distribution:", dict(label_counts))

if errors:
    print(f"\nFAILED: {len(errors)} issue(s). First 30:")
    for err in errors[:30]:
        print(" -", err)
    raise ValueError("Submission structure/label validation failed.")

print("\nPASS: Every document has a non-empty list of valid labels.")

In [ ]:
# Inspect the existing test-data structure without modifying it.
for domain in ("CL", "IT"):
    docs = DOCS[domain]["test"]
    print(f"\n{domain}: {len(docs)} test documents")

    if docs:
        first = docs[0]
        print("First document type:", type(first).__name__)

        if isinstance(first, dict):
            print("First document keys:", list(first.keys()))
            print("Document ID candidates:",
                  {k: first[k] for k in first if "id" in k.lower()})
            for key, value in first.items():
                if isinstance(value, list):
                    print(f"List field {key!r}: length={len(value)}")

In [ ]:
import json
import zipfile

# Load the existing candidate; do not modify it.
with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    prediction = json.loads(
        zf.read("prediction.json").decode("utf-8-sig")
    )

# Build the expected document ID -> sentence count mapping from test inputs.
expected = {}

for domain in ("CL", "IT"):
    for doc in DOCS[domain]["test"]:
        doc_id = doc["id"]

        if doc_id in expected:
            raise ValueError(f"Duplicate test document ID: {doc_id}")

        expected[doc_id] = len(doc["sents"])

actual_ids = set(prediction)
expected_ids = set(expected)

missing_ids = sorted(expected_ids - actual_ids)
extra_ids = sorted(actual_ids - expected_ids)

count_mismatches = []
for doc_id in sorted(expected_ids & actual_ids):
    predicted = prediction[doc_id]

    if not isinstance(predicted, list):
        count_mismatches.append(
            (doc_id, expected[doc_id], "not a list")
        )
    elif len(predicted) != expected[doc_id]:
        count_mismatches.append(
            (doc_id, expected[doc_id], len(predicted))
        )

print("Expected documents:", len(expected_ids))
print("Submitted documents:", len(actual_ids))
print("Missing document IDs:", missing_ids)
print("Unexpected document IDs:", extra_ids)
print("Sentence-count mismatches:", count_mismatches)

if missing_ids or extra_ids or count_mismatches:
    raise ValueError(
        "FAIL: Candidate document IDs or sentence counts do not match test inputs."
    )

print("\nPASS: All document IDs and sentence counts match exactly.")
print("Total aligned sentence predictions:", sum(expected.values()))

In [ ]:
# CELL 1 — CONSOLIDATE SAVED EXPERIMENT RESULTS
# Read-only for source artifacts; writes only to a new timestamped report folder.

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

WORK = Path(WORK)
EXPERIMENT_ROOT = WORK / "experiments"
REPORT_DIR = EXPERIMENT_ROOT / f"final_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
REPORT_DIR.mkdir(parents=True, exist_ok=False)

# All experiment areas we expect from this notebook.
EXPERIMENT_DIRS = {
    "exp1_weighted_and_model_comparison": EXPERIMENT_ROOT / "exp1_model_comparison",
    "weighted_ensemble_v1": EXPERIMENT_ROOT / "weighted_ensemble_v1",
    "class_bias_sweep": EXPERIMENT_ROOT / "joint_class_bias_v1",
    "joint_bias_refinement": EXPERIMENT_ROOT / "joint_bias_refinement_v1",
    "transition_variants": EXPERIMENT_ROOT / "exp3_transition_variants",
    "temperature": EXPERIMENT_ROOT / "exp4_temperature",
    "error_overlap": EXPERIMENT_ROOT / "exp1_model_comparison" / "error_overlap",
    "robustness_lodo": EXPERIMENT_ROOT / "robustness_lodo",
    "decoder_contribution": EXPERIMENT_ROOT / "decoder_contribution",
    "local_smoothing_ablation": EXPERIMENT_ROOT / "local_smoothing_ablation",
}

inventory_rows = []
metric_frames = []
config_rows = []

for experiment_name, folder in EXPERIMENT_DIRS.items():
    if not folder.exists():
        inventory_rows.append({
            "experiment": experiment_name,
            "file": "",
            "status": "DIRECTORY_MISSING",
            "rows": None,
            "columns": "",
        })
        continue

    files = sorted(folder.rglob("*"))
    for path in files:
        if not path.is_file():
            continue

        if path.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(path)
                inventory_rows.append({
                    "experiment": experiment_name,
                    "file": str(path),
                    "status": "READ_OK",
                    "rows": len(df),
                    "columns": ", ".join(map(str, df.columns)),
                })

                # Include metric-bearing tables, preserving all original columns.
                metric_cols = [
                    c for c in df.columns
                    if any(term in str(c).lower() for term in (
                        "macro_f1", "macro_precision", "macro_recall",
                        "accuracy", "temperature", "weight", "smooth",
                        "variant", "candidate", "trial", "stage",
                        "method", "domain",
                    ))
                ]

                if any("macro_f1" in str(c).lower() for c in df.columns):
                    part = df.copy()
                    part.insert(0, "source_file", str(path))
                    part.insert(1, "experiment", experiment_name)
                    metric_frames.append(part)

            except Exception as e:
                inventory_rows.append({
                    "experiment": experiment_name,
                    "file": str(path),
                    "status": f"CSV_READ_ERROR: {e}",
                    "rows": None,
                    "columns": "",
                })

        elif path.suffix.lower() == ".json":
            try:
                with open(path, "r", encoding="utf-8") as f:
                    obj = json.load(f)

                inventory_rows.append({
                    "experiment": experiment_name,
                    "file": str(path),
                    "status": "READ_OK",
                    "rows": len(obj) if isinstance(obj, (dict, list)) else None,
                    "columns": "JSON config/data",
                })

                # Preserve configuration values as serialized JSON.
                config_rows.append({
                    "experiment": experiment_name,
                    "file": str(path),
                    "json": json.dumps(obj, ensure_ascii=False, sort_keys=True),
                })

            except Exception as e:
                inventory_rows.append({
                    "experiment": experiment_name,
                    "file": str(path),
                    "status": f"JSON_READ_ERROR: {e}",
                    "rows": None,
                    "columns": "",
                })

inventory_df = pd.DataFrame(inventory_rows)
configs_df = pd.DataFrame(config_rows)

inventory_path = REPORT_DIR / "experiment_file_inventory.csv"
configs_path = REPORT_DIR / "experiment_configs_index.csv"

inventory_df.to_csv(inventory_path, index=False)
configs_df.to_csv(configs_path, index=False)

if metric_frames:
    metrics_df = pd.concat(metric_frames, ignore_index=True, sort=False)

    # prf_report's macro_f1 averages classes present in gold or predictions.
    # macro_f1_all13 averages all 13 classes, including absent classes.
    metrics_df["scoring_convention_note"] = np.where(
        "macro_f1_all13" in metrics_df.columns,
        "See macro_f1 and macro_f1_all13 columns; preserve both",
        "Inspect source cell/helper; convention not inferable from column name alone",
    )

    metrics_path = REPORT_DIR / "all_saved_metric_rows.csv"
    metrics_df.to_csv(metrics_path, index=False)
    print("Consolidated metric rows:", len(metrics_df))
    print("Metrics saved:", metrics_path)
else:
    metrics_df = pd.DataFrame()
    print("No CSV with a macro_f1 column was found.")

print("\nInventory:", inventory_path)
print("Config index:", configs_path)
print("\nFile inventory by experiment:")
display(
    inventory_df.groupby(["experiment", "status"], dropna=False)
    .size().rename("files").reset_index()
)

print("\nAvailable metric columns:")
if not metrics_df.empty:
    print(list(metrics_df.columns))
    for col in ("domain", "stage", "macro_f1", "macro_f1_all13",
                "accuracy", "method", "weight_E0a", "smooth", "alpha"):
        if col in metrics_df.columns:
            print(f"  {col}: present")

In [ ]:
# CELL 2 — CONSISTENT WEIGHTED ENSEMBLE RE-SCORE
# Validation labels only. No training and no test prediction access.

from pathlib import Path
import json
import numpy as np
import pandas as pd

WEIGHT_AUDIT_DIR = REPORT_DIR / "weighted_rescore"
WEIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=False)

MODEL_A = "E0a_full_sent"
MODEL_B = "E6_kd_student"

# Frozen baseline settings from the selected candidate configuration.
# Only ensemble weights change in this experiment.
BASELINE = {
    "CL": {
        "weight_E0a": 0.55,
        "method": "arithmetic",
        "smooth": 0.50,
        "alpha": 0.25,
        "bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
    },
    "IT": {
        "weight_E0a": 0.50,
        "method": "geometric",
        "smooth": 0.25,
        "alpha": 1.0,
        "bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
    },
}

WEIGHTS_TO_TEST = sorted(set([0.00, 0.25, 0.40, 0.50, 0.55, 0.60, 0.75, 1.00]))

overall_rows = []
doc_rows = []
class_rows = []
prediction_cache = {}

for dom in ("CL", "IT"):
    cfg = BASELINE[dom]

    # Match Cell 77's example-building settings exactly.
    examples = build_examples(
        DOCS[dom]["dev"],
        ctx=True,
        k_left=3,
        k_right=1,
        t_budget=96,
        l_budget=160,
        r_budget=64,
        max_sent=96,
    )
    y_true = np.asarray([int(e["y"]) for e in examples], dtype=int)

    logits_a = np.asarray(load_arr(MODEL_A, dom, "dev_logits"))
    logits_b = np.asarray(load_arr(MODEL_B, dom, "dev_logits"))

    assert logits_a.shape == logits_b.shape
    assert len(y_true) == len(logits_a), (
        dom, len(y_true), len(logits_a)
    )
    assert logits_a.shape[1] == len(LABELS)
    assert np.isfinite(logits_a).all() and np.isfinite(logits_b).all()

    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS), alpha=cfg["alpha"]
    )

    # Weight 1.0 and 0.0 are the raw standalone model probabilities
    # passed through the same domain postprocessing as the ensembles.
    for method in sorted(set([cfg["method"], "arithmetic", "geometric"])):
        for weight_a in WEIGHTS_TO_TEST:
            # The primary sweep changes weights only under the existing method.
            # Other methods are tested only at the baseline weight for reference.
            if method != cfg["method"] and weight_a != cfg["weight_E0a"]:
                continue

            probs = weighted_probs(
                logits_a, logits_b, weight_a=weight_a, method=method
            )
            pred = np.asarray(
                final_predict(
                    probs, examples, logT, logS,
                    smooth=cfg["smooth"],
                    class_bias=cfg["bias"],
                ),
                dtype=int,
            )

            report = prf_report(y_true, pred, len(LABELS))
            name = (
                "baseline"
                if method == cfg["method"]
                and np.isclose(weight_a, cfg["weight_E0a"])
                else "candidate"
            )

            key = (dom, method, float(weight_a))
            prediction_cache[key] = pred

            overall_rows.append({
                "domain": dom,
                "candidate_type": name,
                "model_a": MODEL_A,
                "model_b": MODEL_B,
                "method": method,
                "weight_E0a": float(weight_a),
                "weight_KD": float(1 - weight_a),
                "smooth": cfg["smooth"],
                "transition_alpha_train_only": cfg["alpha"],
                "class_bias": json.dumps(cfg["bias"], sort_keys=True),
                "macro_f1_present_classes": report["macro_f1"],
                "macro_f1_all13": report["macro_f1_all"],
                "macro_precision_present_classes": report["macro_p"],
                "macro_recall_present_classes": report["macro_r"],
                "accuracy": report["acc"],
                "n_present": report["n_present"],
            })

            for j, label in enumerate(LABELS):
                class_rows.append({
                    "domain": dom,
                    "method": method,
                    "weight_E0a": float(weight_a),
                    "label": label,
                    "precision": float(report["prec"][j]),
                    "recall": float(report["rec"][j]),
                    "f1": float(report["f1"][j]),
                    "support": int(report["support"][j]),
                })

            doc_ids = list(dict.fromkeys(str(e["doc"]) for e in examples))
            for doc_id in doc_ids:
                ix = np.asarray(
                    [i for i, e in enumerate(examples)
                     if str(e["doc"]) == doc_id],
                    dtype=int,
                )
                dr = prf_report(y_true[ix], pred[ix], len(LABELS))
                doc_rows.append({
                    "domain": dom,
                    "method": method,
                    "weight_E0a": float(weight_a),
                    "doc_id": doc_id,
                    "n_sentences": len(ix),
                    "macro_f1_present_classes": dr["macro_f1"],
                    "macro_f1_all13": dr["macro_f1_all"],
                    "accuracy": dr["acc"],
                })

overall_df = pd.DataFrame(overall_rows)
class_df = pd.DataFrame(class_rows)
doc_df = pd.DataFrame(doc_rows)

overall_path = WEIGHT_AUDIT_DIR / "consistent_weight_scores.csv"
class_path = WEIGHT_AUDIT_DIR / "consistent_weight_per_class.csv"
doc_path = WEIGHT_AUDIT_DIR / "consistent_weight_per_document.csv"

overall_df.to_csv(overall_path, index=False)
class_df.to_csv(class_path, index=False)
doc_df.to_csv(doc_path, index=False)

for dom in ("CL", "IT"):
    print(f"\n{'=' * 25} {dom} {'=' * 25}")
    subset = overall_df[overall_df["domain"] == dom].copy()
    subset = subset.sort_values(
        ["macro_f1_present_classes", "macro_f1_all13"],
        ascending=False,
    )
    display(subset.round(5))

print("\nSaved:")
print(overall_path)
print(class_path)
print(doc_path)

In [ ]:
# CELL 3 — JOINT-BIAS REVIEW WITH DOCUMENT-LEVEL ROBUSTNESS
# Validation-only. Keeps the selected baseline weights/methods/smoothing fixed.

import json
import numpy as np
import pandas as pd

BIAS_AUDIT_DIR = REPORT_DIR / "bias_review"
BIAS_AUDIT_DIR.mkdir(parents=True, exist_ok=False)

BIAS_REVIEW_CFG = {
    "CL": {
        **BASELINE["CL"],
        "original_bias": {"ArgumentRespondent": 1.0},
        "refined_bias": {
            "ArgumentRespondent": 1.0,
            "ArgumentPetitioner": 0.75,
        },
        "tuned_class": "ArgumentPetitioner",
        "trial_values": [0.25, 0.50, 0.75],
    },
    "IT": {
        **BASELINE["IT"],
        "original_bias": {"RulingByLowerCourt": 1.0},
        "refined_bias": {
            "RulingByLowerCourt": 1.0,
            "RulingByPresentCourt": 1.0,
        },
        "tuned_class": "RulingByPresentCourt",
        "trial_values": [0.50, 0.75, 1.00, 1.25],
    },
}

bias_overall_rows = []
bias_doc_rows = []
bias_class_rows = []

for dom, cfg in BIAS_REVIEW_CFG.items():
    examples = build_examples(
        DOCS[dom]["dev"],
        ctx=True,
        k_left=3,
        k_right=1,
        t_budget=96,
        l_budget=160,
        r_budget=64,
        max_sent=96,
    )
    y_true = np.asarray([int(e["y"]) for e in examples], dtype=int)

    logits_a = np.asarray(load_arr(MODEL_A, dom, "dev_logits"))
    logits_b = np.asarray(load_arr(MODEL_B, dom, "dev_logits"))
    assert len(examples) == len(logits_a) == len(logits_b)

    probs = weighted_probs(
        logits_a, logits_b,
        weight_a=cfg["weight_E0a"],
        method=cfg["method"],
    )
    logT, logS = transition_logprobs(
        DOCS[dom]["train"], LABEL2ID, len(LABELS),
        alpha=cfg["alpha"],
    )

    trials = [
        ("original_one_class_bias", dict(cfg["original_bias"])),
        ("refined_joint_bias", dict(cfg["refined_bias"])),
    ]

    # Controlled local search: preserve the original target bias.
    for value in cfg["trial_values"]:
        trial_bias = dict(cfg["original_bias"])
        trial_bias[cfg["tuned_class"]] = float(value)
        trials.append((
            f'{cfg["tuned_class"]}={value:+.2f}',
            trial_bias,
        ))

    # Deduplicate configurations with identical bias dictionaries.
    seen = set()
    for trial_name, bias in trials:
        signature = tuple(sorted(bias.items()))
        if signature in seen:
            continue
        seen.add(signature)

        pred = np.asarray(
            final_predict(
                probs, examples, logT, logS,
                smooth=cfg["smooth"],
                class_bias=bias,
            ),
            dtype=int,
        )
        report = prf_report(y_true, pred, len(LABELS))

        bias_overall_rows.append({
            "domain": dom,
            "trial": trial_name,
            "class_bias": json.dumps(bias, sort_keys=True),
            "method": cfg["method"],
            "weight_E0a": cfg["weight_E0a"],
            "weight_KD": 1.0 - cfg["weight_E0a"],
            "smooth": cfg["smooth"],
            "alpha": cfg["alpha"],
            "macro_f1_present_classes": report["macro_f1"],
            "macro_f1_all13": report["macro_f1_all"],
            "macro_precision_present_classes": report["macro_p"],
            "macro_recall_present_classes": report["macro_r"],
            "accuracy": report["acc"],
        })

        for j, label in enumerate(LABELS):
            bias_class_rows.append({
                "domain": dom,
                "trial": trial_name,
                "label": label,
                "precision": float(report["prec"][j]),
                "recall": float(report["rec"][j]),
                "f1": float(report["f1"][j]),
                "support": int(report["support"][j]),
            })

        for doc_id in dict.fromkeys(str(e["doc"]) for e in examples):
            ix = np.asarray([
                i for i, e in enumerate(examples)
                if str(e["doc"]) == doc_id
            ], dtype=int)

            dr = prf_report(y_true[ix], pred[ix], len(LABELS))
            bias_doc_rows.append({
                "domain": dom,
                "trial": trial_name,
                "doc_id": doc_id,
                "n_sentences": len(ix),
                "macro_f1_present_classes": dr["macro_f1"],
                "macro_f1_all13": dr["macro_f1_all"],
                "accuracy": dr["acc"],
            })

bias_overall_df = pd.DataFrame(bias_overall_rows)
bias_class_df = pd.DataFrame(bias_class_rows)
bias_doc_df = pd.DataFrame(bias_doc_rows)

bias_overall_df.to_csv(
    BIAS_AUDIT_DIR / "bias_overall.csv", index=False
)
bias_class_df.to_csv(
    BIAS_AUDIT_DIR / "bias_per_class.csv", index=False
)
bias_doc_df.to_csv(
    BIAS_AUDIT_DIR / "bias_per_document.csv", index=False
)

print("=== Overall bias comparison ===")
display(
    bias_overall_df.sort_values(
        ["domain", "macro_f1_present_classes"],
        ascending=[True, False],
    ).round(5)
)

print("\n=== Per-document results ===")
display(
    bias_doc_df.sort_values(["domain", "doc_id", "trial"]).round(5)
)

# Summarize the refined-minus-original change across documents.
spread_rows = []
for dom in ("CL", "IT"):
    sub = bias_doc_df[bias_doc_df["domain"] == dom]
    old = sub[sub["trial"] == "original_one_class_bias"].set_index("doc_id")
    new = sub[sub["trial"] == "refined_joint_bias"].set_index("doc_id")

    common = sorted(set(old.index) & set(new.index))
    diffs = np.asarray([
        new.loc[d, "macro_f1_present_classes"]
        - old.loc[d, "macro_f1_present_classes"]
        for d in common
    ], dtype=float)

    spread_rows.append({
        "domain": dom,
        "documents_compared": len(diffs),
        "documents_improved": int((diffs > 0).sum()),
        "documents_tied": int(np.isclose(diffs, 0).sum()),
        "documents_worse": int((diffs < 0).sum()),
        "mean_document_delta": float(diffs.mean()) if len(diffs) else np.nan,
        "median_document_delta": float(np.median(diffs)) if len(diffs) else np.nan,
        "min_document_delta": float(diffs.min()) if len(diffs) else np.nan,
        "max_document_delta": float(diffs.max()) if len(diffs) else np.nan,
    })

spread_df = pd.DataFrame(spread_rows)
spread_df.to_csv(
    BIAS_AUDIT_DIR / "refined_vs_original_document_spread.csv",
    index=False,
)

print("\n=== Does the refinement help across documents? ===")
display(spread_df.round(5))
print("\nSaved bias reports to:", BIAS_AUDIT_DIR)

In [ ]:
# CELL 4 — FREEZE BASELINE CONFIG + VERIFY CANDIDATE/BACKUP
# Does not modify the candidate ZIP or backup ZIP.

from pathlib import Path
from datetime import datetime
import hashlib
import json
import zipfile

CANDIDATE_DIR = Path(WORK) / "final_candidate_20261014"
CANDIDATE_ZIP = CANDIDATE_DIR / "submission_candidate.zip"

assert CANDIDATE_ZIP.exists(), f"Candidate missing: {CANDIDATE_ZIP}"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Identify an existing backup, but never create or overwrite one here.
backup_candidates = sorted(
    CANDIDATE_DIR.glob("submission_candidate_backup_*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
BACKUP_ZIP = backup_candidates[0] if backup_candidates else None

with zipfile.ZipFile(CANDIDATE_ZIP, "r") as zf:
    bad = zf.testzip()
    assert bad is None, f"Candidate ZIP integrity failure: {bad}"
    assert "prediction.json" in zf.namelist(), (
        "Expected prediction.json at the ZIP root."
    )

candidate_hash_before = sha256_file(CANDIDATE_ZIP)

backup_hash = None
backup_matches = None
if BACKUP_ZIP is not None:
    with zipfile.ZipFile(BACKUP_ZIP, "r") as zf:
        bad = zf.testzip()
        assert bad is None, f"Backup ZIP integrity failure: {bad}"

    backup_hash = sha256_file(BACKUP_ZIP)
    backup_matches = backup_hash == candidate_hash_before

# This freezes the baseline configuration for the experiment record.
# It does NOT assert that this configuration produced the existing ZIP.
FROZEN_CONFIG = {
    "purpose": "validation_experiment_baseline",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "domains": BASELINE,
    "models": [MODEL_A, MODEL_B],
    "scoring": {
        "macro_f1_present_classes": (
            "prf_report macro_f1: labels present in gold or predictions"
        ),
        "macro_f1_all13": (
            "prf_report macro_f1_all: arithmetic mean over all 13 labels"
        ),
    },
    "transition_statistics": (
        "Estimated from DOCS[domain]['train'] only"
    ),
    "test_labels_used_for_tuning": False,
    "candidate_zip_modified": False,
}

frozen_path = REPORT_DIR / "frozen_baseline_config.json"
manifest_path = REPORT_DIR / "artifact_checksum_manifest.json"

# Report paths are unique because REPORT_DIR has a unique timestamp.
with open(frozen_path, "x", encoding="utf-8") as f:
    json.dump(FROZEN_CONFIG, f, indent=2, ensure_ascii=False)

manifest = {
    "candidate_zip": str(CANDIDATE_ZIP),
    "candidate_sha256_before": candidate_hash_before,
    "candidate_size_bytes": CANDIDATE_ZIP.stat().st_size,
    "candidate_zip_integrity": "PASS",
    "backup_zip": str(BACKUP_ZIP) if BACKUP_ZIP else None,
    "backup_sha256": backup_hash,
    "backup_matches_candidate_byte_for_byte": backup_matches,
    "candidate_zip_modified_by_this_cell": False,
    "frozen_config_file": str(frozen_path),
}

with open(manifest_path, "x", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# Confirm candidate remained unchanged while writing the audit files.
candidate_hash_after = sha256_file(CANDIDATE_ZIP)
assert candidate_hash_before == candidate_hash_after, (
    "Candidate ZIP hash changed during this cell!"
)

print("Frozen config:", frozen_path)
print("Checksum manifest:", manifest_path)
print("Candidate ZIP:", CANDIDATE_ZIP)
print("Candidate SHA-256:", candidate_hash_after)
print("Candidate unchanged: PASS")
print("Existing backup:", BACKUP_ZIP)
print("Backup hash matches candidate:", backup_matches)
print("\nFrozen baseline configuration:")
print(json.dumps(FROZEN_CONFIG, indent=2))

In [ ]:
# CELL 5 — FINAL DECISION SUMMARY
# Reads only the audit results produced by Cells 2 and 3.

decision_rows = []

for dom in ("CL", "IT"):
    overall = overall_df[overall_df["domain"] == dom].copy()

    baseline_cfg = BASELINE[dom]
    baseline_row = overall[
        (overall["method"] == baseline_cfg["method"])
        & np.isclose(
            overall["weight_E0a"].astype(float),
            baseline_cfg["weight_E0a"],
        )
    ]

    if len(baseline_row) != 1:
        raise ValueError(
            f"{dom}: expected one baseline row, found {len(baseline_row)}"
        )

    baseline_row = baseline_row.iloc[0]
    same_method = overall[overall["method"] == baseline_cfg["method"]]
    best_row = same_method.sort_values(
        ["macro_f1_present_classes", "macro_f1_all13"],
        ascending=False,
    ).iloc[0]

    best_weight = float(best_row["weight_E0a"])
    baseline_weight = float(baseline_cfg["weight_E0a"])

    best_doc = doc_df[
        (doc_df["domain"] == dom)
        & (doc_df["method"] == baseline_cfg["method"])
        & np.isclose(doc_df["weight_E0a"].astype(float), best_weight)
    ].set_index("doc_id")

    baseline_doc = doc_df[
        (doc_df["domain"] == dom)
        & (doc_df["method"] == baseline_cfg["method"])
        & np.isclose(
            doc_df["weight_E0a"].astype(float), baseline_weight
        )
    ].set_index("doc_id")

    common_docs = sorted(set(best_doc.index) & set(baseline_doc.index))
    doc_deltas = np.asarray([
        best_doc.loc[d, "macro_f1_present_classes"]
        - baseline_doc.loc[d, "macro_f1_present_classes"]
        for d in common_docs
    ], dtype=float)

    decision_rows.append({
        "domain": dom,
        "baseline_weight_E0a": baseline_weight,
        "best_sweep_weight_E0a": best_weight,
        "baseline_macro_f1": float(
            baseline_row["macro_f1_present_classes"]
        ),
        "best_sweep_macro_f1": float(
            best_row["macro_f1_present_classes"]
        ),
        "delta_macro_f1": float(
            best_row["macro_f1_present_classes"]
            - baseline_row["macro_f1_present_classes"]
        ),
        "baseline_macro_f1_all13": float(baseline_row["macro_f1_all13"]),
        "best_sweep_macro_f1_all13": float(best_row["macro_f1_all13"]),
        "delta_macro_f1_all13": float(
            best_row["macro_f1_all13"] - baseline_row["macro_f1_all13"]
        ),
        "documents_compared": len(doc_deltas),
        "documents_improved": int((doc_deltas > 0).sum()),
        "documents_tied": int(np.isclose(doc_deltas, 0).sum()),
        "documents_worse": int((doc_deltas < 0).sum()),
        "mean_document_delta": (
            float(doc_deltas.mean()) if len(doc_deltas) else np.nan
        ),
        "interpretation": (
            "Review per-document spread; do not auto-promote"
        ),
    })

decision_df = pd.DataFrame(decision_rows)
decision_path = REPORT_DIR / "final_weight_decision_summary.csv"
decision_df.to_csv(decision_path, index=False)

print("=== Weight sweep decision summary ===")
display(decision_df.round(6))

print("\n=== Refined bias decision summary ===")
bias_compare = bias_overall_df[
    bias_overall_df["trial"].isin([
        "original_one_class_bias",
        "refined_joint_bias",
    ])
].pivot(
    index="domain",
    columns="trial",
    values=["macro_f1_present_classes", "macro_f1_all13", "accuracy"],
)
display(bias_compare.round(6))

print("\nDecision file:", decision_path)
print("\nImportant:")
print("- Do not promote a setting on a tiny macro-F1 gain alone.")
print("- Check all-13 macro-F1, per-class metrics, and per-document changes.")
print("- Keep the frozen baseline unless the evidence supports a change.")
print("- These results select among development-set candidates, not test labels.")